# RiskCheck AI

## Week 1: Data Collection and Data Understanding

### Project Objective

RiskCheck AI is an intelligent invoice risk and fraud detection system.  
The project combines invoice information, supplier details, behavioural features,
invoice images, and fraud labels to identify potentially risky invoices.

### Week 1 Objectives

- Verify all uploaded project files
- Load all structured datasets
- Understand dataset schemas
- Examine data quality
- Identify missing values and duplicate records
- Understand relationships between datasets
- Prepare the data for exploratory analysis

## Importing Libraries & Defining Data Path

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *


import pandas as pd
import numpy as np


import pyarrow as pa
import pyarrow.parquet as pq


import os
import json

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
BASE_PATH = "/Volumes/workspace/default/dataanalytics/RiskCheck AI/"

print(f"RiskCheck AI project path: {BASE_PATH}")

RiskCheck AI project path: /Volumes/workspace/default/dataanalytics/RiskCheck AI/


In [0]:
project_files = dbutils.fs.ls(BASE_PATH)

file_inventory = []

for file in project_files:
    file_inventory.append(
        {
            "file_name": file.name,
            "file_path": file.path,
            "file_type": "Folder" if file.isDir() else "File",
            "size_mb": round(file.size / (1024 ** 2), 2)
        }
    )

file_inventory_df = spark.createDataFrame(file_inventory)

display(
    file_inventory_df.orderBy("file_name")
)

file_name,file_path,file_type,size_mb
behavioural_features.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/behavioural_features.parquet,File,7.21
departments.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/departments.parquet,File,0.0
images.rar,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/images.rar,File,1934.32
images_metadata.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/images_metadata.parquet,File,1.03
invoices.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/invoices.parquet,File,5.04
labels.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/labels.parquet,File,1.84
manifest.json,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/manifest.json,File,0.0
outputs/,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/outputs/,Folder,0.0
splits.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/splits.parquet,File,1.86
suppliers.parquet,dbfs:/Volumes/workspace/default/dataanalytics/RiskCheck AI/suppliers.parquet,File,0.1


## Checking Manifest File

In [0]:
MANIFEST_PATH = f"{BASE_PATH}/manifest.json"

with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    manifest_data = json.load(file)

print(json.dumps(manifest_data, indent=4))

{
    "version": "v1",
    "generated_utc": "20251214_172643",
    "counts": {
        "suppliers": 2000,
        "departments": 50,
        "invoices": 300000,
        "behavioural_features": 300000,
        "labels": 300000,
        "images": 45000
    },
    "format": "parquet"
}


## Define All Parquet File Paths

In [0]:
PARQUET_FILES = {
    "behavioural_features": f"{BASE_PATH}/behavioural_features.parquet",
    "departments": f"{BASE_PATH}/departments.parquet",
    "images_metadata": f"{BASE_PATH}/images_metadata.parquet",
    "invoices": f"{BASE_PATH}/invoices.parquet",
    "labels": f"{BASE_PATH}/labels.parquet",
    "splits": f"{BASE_PATH}/splits.parquet",
    "suppliers": f"{BASE_PATH}/suppliers.parquet"
}

for dataset_name, dataset_path in PARQUET_FILES.items():
    print(f"{dataset_name:<25} {dataset_path}")

behavioural_features      /Volumes/workspace/default/dataanalytics/RiskCheck AI//behavioural_features.parquet
departments               /Volumes/workspace/default/dataanalytics/RiskCheck AI//departments.parquet
images_metadata           /Volumes/workspace/default/dataanalytics/RiskCheck AI//images_metadata.parquet
invoices                  /Volumes/workspace/default/dataanalytics/RiskCheck AI//invoices.parquet
labels                    /Volumes/workspace/default/dataanalytics/RiskCheck AI//labels.parquet
splits                    /Volumes/workspace/default/dataanalytics/RiskCheck AI//splits.parquet
suppliers                 /Volumes/workspace/default/dataanalytics/RiskCheck AI//suppliers.parquet


## Step 5: Inspect Original Parquet Schemas

This step examines the schema of each Parquet dataset to understand its
structure, identify data types, and detect unsupported timestamp formats
before loading the data into Spark.

In [0]:
for dataset_name, file_path in PARQUET_FILES.items():
    print("\n" + "=" * 100)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 100)

    parquet_file = pq.ParquetFile(file_path)

    print(parquet_file.schema_arrow)


DATASET: BEHAVIOURAL_FEATURES
invoice_id: string
supplier_invoice_count_30d: double
supplier_avg_amount_90d: double
invoice_amount_zscore: double
duplicate_invoice_flag: int64
split_invoice_flag: int64
late_night_submission_flag: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1079

DATASET: DEPARTMENTS
department_id: string
region: string
annual_budget: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 458

DATASET: IMAGES_METADATA
invoice_id: string
image_path: string
ocr_total_extracted: double
image_tamper_flag: int64
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 603

DATASET: INVOICES
invoice_id: string
supplier_id: string
department_id: string
invoice_date: timestamp[ns]
invoice_amount: double
currency: string
payment_terms: string
invoice_type: string
submission_hour: int64
image_path: string
-- schema metadata --


| Dataset              | Purpose                   |    Rows | Columns |
| -------------------- | ------------------------- | ------: | ------: |
| Invoices             | Main transaction table    | 300,000 |      10 |
| Labels               | Risk labels               | 300,000 |       5 |
| Behavioural Features | Behaviour metrics         | 300,000 |       7 |
| Suppliers            | Supplier information      |   2,000 |       6 |
| Departments          | Department lookup         |      50 |       3 |
| Image Metadata       | Invoice image information |  45,000 |       4 |
| Splits               | Train/Test assignment     | 300,000 |       2 |


## Create a Safe Parquet Loading Function

In [0]:
def load_parquet_safely(file_path: str):
    """
    Load a Parquet file using PyArrow.

    Nanosecond timestamp columns are converted to microseconds
    before the data is converted into a Spark DataFrame.
    """

    print(f"Loading: {file_path}")

    arrow_table = pq.read_table(file_path)

    converted_arrays = []
    converted_fields = []

    for field in arrow_table.schema:
        column_array = arrow_table[field.name]
        new_field = field

        if pa.types.is_timestamp(field.type) and field.type.unit == "ns":
            target_type = pa.timestamp("us", tz=field.type.tz)

            print(
                f"  Converting column '{field.name}' "
                f"from timestamp[ns] to timestamp[us]"
            )

            column_array = column_array.cast(
                target_type,
                safe=False
            )

            new_field = pa.field(
                field.name,
                target_type,
                nullable=field.nullable
            )

        converted_arrays.append(column_array)
        converted_fields.append(new_field)

    corrected_schema = pa.schema(converted_fields)

    corrected_table = pa.Table.from_arrays(
        converted_arrays,
        schema=corrected_schema
    )

    pandas_df = corrected_table.to_pandas()

    spark_df = spark.createDataFrame(pandas_df)

    print(
        f"  Successfully loaded "
        f"{spark_df.count():,} rows and "
        f"{len(spark_df.columns)} columns"
    )

    return spark_df

## Testing the Loader with the Smallest Dataset

In [0]:
departments_df = load_parquet_safely(
    PARQUET_FILES["departments"]
)

display(departments_df)

Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//departments.parquet
  Successfully loaded 50 rows and 3 columns


department_id,region,annual_budget
DPT_000,Rhode Island,3.2788813503695077E8
DPT_001,Arkansas,2.6668785201303322E7
DPT_002,Nevada,1.4914645502737343E7
DPT_003,New York,4.204102283429599E8
DPT_004,Louisiana,2.956358085560889E8
DPT_005,Arkansas,1.1622908895079409E8
DPT_006,Texas,3.771371745502147E8
DPT_007,New York,1.3552763775179547E8
DPT_008,Louisiana,2.1288906530955267E8
DPT_009,Arizona,2.282605365655489E8


## Load All Structured Datasets

In [0]:
dataframes = {}
loading_results = []

for dataset_name, file_path in PARQUET_FILES.items():
    print("\n" + "=" * 100)
    print(f"LOADING: {dataset_name.upper()}")
    print("=" * 100)

    try:
        dataframe = load_parquet_safely(file_path)

        dataframes[dataset_name] = dataframe

        loading_results.append(
            {
                "dataset": dataset_name,
                "status": "Loaded",
                "rows": dataframe.count(),
                "columns": len(dataframe.columns),
                "error": None
            }
        )

    except Exception as error:
        loading_results.append(
            {
                "dataset": dataset_name,
                "status": "Failed",
                "rows": None,
                "columns": None,
                "error": str(error)
            }
        )

        print(f"Failed to load {dataset_name}")
        print(f"Error: {error}")


LOADING: BEHAVIOURAL_FEATURES
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//behavioural_features.parquet
  Successfully loaded 300,000 rows and 7 columns

LOADING: DEPARTMENTS
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//departments.parquet
  Successfully loaded 50 rows and 3 columns

LOADING: IMAGES_METADATA
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//images_metadata.parquet
  Successfully loaded 45,000 rows and 4 columns

LOADING: INVOICES
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//invoices.parquet
  Converting column 'invoice_date' from timestamp[ns] to timestamp[us]
  Successfully loaded 300,000 rows and 10 columns

LOADING: LABELS
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//labels.parquet
  Successfully loaded 300,000 rows and 5 columns

LOADING: SPLITS
Loading: /Volumes/workspace/default/dataanalytics/RiskCheck AI//splits.parquet
  Successfully loaded 300,000 rows and 2 columns

LOA

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

loading_results_schema = StructType([
    StructField("dataset", StringType(), False),
    StructField("status", StringType(), False),
    StructField("rows", LongType(), True),
    StructField("columns", LongType(), True),
    StructField("error", StringType(), True)
])

loading_results_df = spark.createDataFrame(
    loading_results,
    schema=loading_results_schema
)

display(
    loading_results_df.orderBy("dataset")
)

dataset,status,rows,columns,error
behavioural_features,Loaded,300000,7,null
departments,Loaded,50,3,null
images_metadata,Loaded,45000,4,null
invoices,Loaded,300000,10,null
labels,Loaded,300000,5,null
splits,Loaded,300000,2,null
suppliers,Loaded,2000,6,null


## Create Individual DataFrame Names

In [0]:
behavioural_features_df = dataframes["behavioural_features"]
departments_df = dataframes["departments"]
images_metadata_df = dataframes["images_metadata"]
invoices_df = dataframes["invoices"]
labels_df = dataframes["labels"]
splits_df = dataframes["splits"]
suppliers_df = dataframes["suppliers"]

print("Individual DataFrame variables created successfully.")

Individual DataFrame variables created successfully.


## Create a Dataset Summary

In [0]:
dataset_summary = []

for dataset_name, dataframe in dataframes.items():
    dataset_summary.append(
        {
            "dataset": dataset_name,
            "number_of_rows": dataframe.count(),
            "number_of_columns": len(dataframe.columns)
        }
    )

dataset_summary_df = spark.createDataFrame(dataset_summary)

display(
    dataset_summary_df.orderBy(
        F.desc("number_of_rows")
    )
)

dataset,number_of_columns,number_of_rows
behavioural_features,7,300000
invoices,10,300000
labels,5,300000
splits,2,300000
images_metadata,4,45000
suppliers,6,2000
departments,3,50


## Display Dataset Schemas

In [0]:
for dataset_name, dataframe in dataframes.items():
    print("\n" + "=" * 100)
    print(f"SCHEMA: {dataset_name.upper()}")
    print("=" * 100)

    dataframe.printSchema()


SCHEMA: BEHAVIOURAL_FEATURES
root
 |-- invoice_id: string (nullable = true)
 |-- supplier_invoice_count_30d: double (nullable = true)
 |-- supplier_avg_amount_90d: double (nullable = true)
 |-- invoice_amount_zscore: double (nullable = true)
 |-- duplicate_invoice_flag: long (nullable = true)
 |-- split_invoice_flag: long (nullable = true)
 |-- late_night_submission_flag: long (nullable = true)


SCHEMA: DEPARTMENTS
root
 |-- department_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- annual_budget: double (nullable = true)


SCHEMA: IMAGES_METADATA
root
 |-- invoice_id: string (nullable = true)
 |-- image_path: string (nullable = true)
 |-- ocr_total_extracted: double (nullable = true)
 |-- image_tamper_flag: long (nullable = true)


SCHEMA: INVOICES
root
 |-- invoice_id: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- invoice_amount: double (nul

## Creating Column Inventory

In [0]:
column_inventory = []

for dataset_name, dataframe in dataframes.items():
    for column_name, data_type in dataframe.dtypes:
        column_inventory.append(
            {
                "dataset": dataset_name,
                "column_name": column_name,
                "data_type": data_type
            }
        )

column_inventory_df = spark.createDataFrame(column_inventory)

display(
    column_inventory_df.orderBy(
        "dataset",
        "column_name"
    )
)

column_name,data_type,dataset
duplicate_invoice_flag,bigint,behavioural_features
invoice_amount_zscore,double,behavioural_features
invoice_id,string,behavioural_features
late_night_submission_flag,bigint,behavioural_features
split_invoice_flag,bigint,behavioural_features
supplier_avg_amount_90d,double,behavioural_features
supplier_invoice_count_30d,double,behavioural_features
annual_budget,double,departments
department_id,string,departments
region,string,departments


## Preview Each Dataset
### Invoices
### Labels
### Suppliers
### Departments
### Behavioural Features
### Image Metadata
### Dataset Splits

In [0]:
display(invoices_df.limit(10))

invoice_id,supplier_id,department_id,invoice_date,invoice_amount,currency,payment_terms,invoice_type,submission_hour,image_path
INV_0000000,7316ce64-2a14-4a96-9b0c-62eb27c8fc6c,DPT_025,2023-04-18T00:00:00.000Z,3565.57,ZAR,NET60,GOODS,18,data/v1/20251214_172643/images/INV_0000000.png
INV_0000001,7428a656-b3ee-4d3b-9a10-412954aebd1b,DPT_010,2023-10-08T00:00:00.000Z,9871.78,ZAR,NET30,SERVICES,6,null
INV_0000002,7b0ccad9-eee1-4cae-b02e-21aa352b6ec8,DPT_041,2023-09-24T00:00:00.000Z,8530.64,ZAR,NET60,SERVICES,23,null
INV_0000003,1c6345ab-6e0e-41e8-985d-3f861d2324e6,DPT_015,2023-10-15T00:00:00.000Z,1378.52,ZAR,NET30,GOODS,7,null
INV_0000004,22d839d3-33e4-4986-b18c-ccdc131eb723,DPT_039,2023-06-07T00:00:00.000Z,2373.66,ZAR,NET30,GOODS,12,null
INV_0000005,ff84faef-5336-423b-8f96-468535145890,DPT_038,2023-03-19T00:00:00.000Z,6652.67,ZAR,NET30,SERVICES,6,null
INV_0000006,7a1b5806-6160-46b4-9360-715fc3fe0183,DPT_019,2023-08-10T00:00:00.000Z,4159.75,ZAR,NET60,SERVICES,14,data/v1/20251214_172643/images/INV_0000006.png
INV_0000007,004ea81a-d3d8-47a8-8a0e-a57a9a7d509d,DPT_023,2023-12-01T00:00:00.000Z,4028.91,ZAR,NET30,SERVICES,15,null
INV_0000008,d2d33488-d0c3-4018-ae3c-dd4f55b145e4,DPT_002,2023-01-03T00:00:00.000Z,5573.19,ZAR,NET60,GOODS,3,null
INV_0000009,d7665cda-fe04-4059-b985-fb6217dc8eff,DPT_046,2023-07-20T00:00:00.000Z,24484.1,ZAR,NET30,GOODS,1,null


In [0]:
display(labels_df.limit(10))

invoice_id,is_fraud,fraud_type,fraud_tags,explanations
INV_0000000,0,NONE,,"{""reason"": ""NONE""}"
INV_0000001,0,NONE,,"{""reason"": ""NONE""}"
INV_0000002,0,NONE,,"{""reason"": ""NONE""}"
INV_0000003,1,SPLIT,SPLIT,"{""rules"": [""SPLIT""]}"
INV_0000004,0,NONE,,"{""reason"": ""NONE""}"
INV_0000005,0,NONE,,"{""reason"": ""NONE""}"
INV_0000006,1,NONE,,"{""reason"": ""NONE""}"
INV_0000007,0,NONE,,"{""reason"": ""NONE""}"
INV_0000008,0,NONE,,"{""reason"": ""NONE""}"
INV_0000009,0,NONE,,"{""reason"": ""NONE""}"


In [0]:
display(suppliers_df.limit(10))

supplier_id,supplier_country,supplier_age_days,supplier_risk_score,blacklisted_flag,avg_invoice_amount
fc3e058b-e0f3-4ab0-9cec-4eb5edd96831,SV,841,0.244,0,5590.25
3d4cbf37-4eb9-4eff-8e88-cb2dd4e80839,KE,61,0.284,0,5797.64
913e4de2-e0c5-4cb8-bda9-c2a90ed42f1a,YE,197,0.39,0,897.65
bb5e4bcf-15ed-4269-9429-6c07f26b4776,SA,851,0.299,0,7727.71
fa5d3100-11b7-4948-90e6-e6607c69dee1,DJ,4373,0.418,0,5458.67
2031d750-c40d-49b4-885f-6e66c2b6d2c5,IT,1218,0.5,0,7294.2
f264accc-79ac-4b1e-a8e5-6e0c20de435d,IN,81,0.304,0,1996.62
8715a103-43da-4043-aa45-c2ab8cbfedb0,CG,111,0.151,0,3940.09
f6e07cc0-6c52-449f-9b49-bd26df57c59a,CA,4554,0.329,0,2289.14
c1590f53-8a0f-4efb-adcd-465e36386821,IN,1394,0.524,0,8276.88


In [0]:
display(departments_df.limit(10))

department_id,region,annual_budget
DPT_000,Rhode Island,3.2788813503695077E8
DPT_001,Arkansas,2.6668785201303322E7
DPT_002,Nevada,1.4914645502737343E7
DPT_003,New York,4.204102283429599E8
DPT_004,Louisiana,2.956358085560889E8
DPT_005,Arkansas,1.1622908895079409E8
DPT_006,Texas,3.771371745502147E8
DPT_007,New York,1.3552763775179547E8
DPT_008,Louisiana,2.1288906530955267E8
DPT_009,Arizona,2.282605365655489E8


In [0]:
display(behavioural_features_df.limit(10))

invoice_id,supplier_invoice_count_30d,supplier_avg_amount_90d,invoice_amount_zscore,duplicate_invoice_flag,split_invoice_flag,late_night_submission_flag
INV_0138003,1.0,4252.16,-0.3759078920805204,0,0,0
INV_0153155,2.0,13972.275,1.8301686106293111,0,0,0
INV_0034829,3.0,17737.206666666665,2.0088632324155307,0,0,0
INV_0072998,4.0,14480.34,-0.32398173245638195,0,1,0
INV_0211525,5.0,12515.047999999999,-0.33032072296428083,0,1,0
INV_0287780,6.0,11484.126666666665,-0.1401691645207489,0,0,0
INV_0142798,7.0,10725.707142857142,-0.15768252658813237,0,0,1
INV_0016168,8.0,9947.66,-0.3476320907047029,0,0,1
INV_0003320,9.0,9207.205555555556,-0.48582344553640616,0,1,0
INV_0143170,10.0,9762.520999999999,0.8165622127369356,0,0,0


In [0]:
display(images_metadata_df.limit(10))

invoice_id,image_path,ocr_total_extracted,image_tamper_flag
INV_0004941,data/v1/20251214_172643/images/INV_0004941.png,52538.04,0
INV_0051775,data/v1/20251214_172643/images/INV_0051775.png,125344.24,0
INV_0115253,data/v1/20251214_172643/images/INV_0115253.png,63871.29,0
INV_0299321,data/v1/20251214_172643/images/INV_0299321.png,6622.06,0
INV_0173570,data/v1/20251214_172643/images/INV_0173570.png,23200.69,0
INV_0030862,data/v1/20251214_172643/images/INV_0030862.png,16244.76,0
INV_0244471,data/v1/20251214_172643/images/INV_0244471.png,41376.06,1
INV_0127794,data/v1/20251214_172643/images/INV_0127794.png,6146.85,0
INV_0071558,data/v1/20251214_172643/images/INV_0071558.png,6380.79,0
INV_0218011,data/v1/20251214_172643/images/INV_0218011.png,14441.76,0


In [0]:
display(splits_df.limit(10))

invoice_id,split
INV_0258729,train
INV_0179434,train
INV_0123072,train
INV_0031388,train
INV_0205955,train
INV_0252070,train
INV_0083680,train
INV_0257170,train
INV_0269808,train
INV_0041949,train


In [0]:
def missing_value_summary(dataframe):
    """
    Return missing-value counts and percentages for each column.
    """

    total_rows = dataframe.count()

    if total_rows == 0:
        return None

    missing_expressions = []

    for column_name, data_type in dataframe.dtypes:
        missing_condition = F.col(column_name).isNull()

        if data_type == "string":
            missing_condition = (
                missing_condition |
                (F.trim(F.col(column_name)) == "")
            )

        if data_type in ["float", "double"]:
            missing_condition = (
                missing_condition |
                F.isnan(F.col(column_name))
            )

        missing_expressions.append(
            F.sum(
                F.when(missing_condition, 1).otherwise(0)
            ).alias(column_name)
        )

    missing_counts = (
        dataframe
        .select(missing_expressions)
        .first()
        .asDict()
    )

    summary = []

    for column_name, missing_count in missing_counts.items():
        summary.append(
            {
                "column_name": column_name,
                "missing_count": int(missing_count),
                "missing_percentage": round(
                    missing_count / total_rows * 100,
                    2
                )
            }
        )

    return (
        spark.createDataFrame(summary)
        .orderBy(F.desc("missing_percentage"))
    )

In [0]:
display(
    missing_value_summary(invoices_df)
)

column_name,missing_count,missing_percentage
image_path,255000,85.0
invoice_id,0,0.0
supplier_id,0,0.0
department_id,0,0.0
invoice_date,0,0.0
invoice_amount,0,0.0
currency,0,0.0
payment_terms,0,0.0
invoice_type,0,0.0
submission_hour,0,0.0


In [0]:
display(
    missing_value_summary(labels_df
                          )
)

column_name,missing_count,missing_percentage
fraud_tags,237036,79.01
invoice_id,0,0.0
is_fraud,0,0.0
fraud_type,0,0.0
explanations,0,0.0


In [0]:
display(
    missing_value_summary(suppliers_df)
)

column_name,missing_count,missing_percentage
supplier_id,0,0.0
supplier_country,0,0.0
supplier_age_days,0,0.0
supplier_risk_score,0,0.0
blacklisted_flag,0,0.0
avg_invoice_amount,0,0.0


In [0]:
display(
    missing_value_summary(departments_df)
)

column_name,missing_count,missing_percentage
department_id,0,0.0
region,0,0.0
annual_budget,0,0.0


In [0]:
display(
    missing_value_summary(behavioural_features_df)
)

column_name,missing_count,missing_percentage
invoice_id,0,0.0
supplier_invoice_count_30d,0,0.0
supplier_avg_amount_90d,0,0.0
invoice_amount_zscore,0,0.0
duplicate_invoice_flag,0,0.0
split_invoice_flag,0,0.0
late_night_submission_flag,0,0.0


In [0]:
display(
    missing_value_summary(images_metadata_df)
)

column_name,missing_count,missing_percentage
invoice_id,0,0.0
image_path,0,0.0
ocr_total_extracted,0,0.0
image_tamper_flag,0,0.0


In [0]:
display(
    missing_value_summary(splits_df)
)

column_name,missing_count,missing_percentage
invoice_id,0,0.0
split,0,0.0


### Check Complete Duplicate Rows

In [0]:
duplicate_summary = []

for dataset_name, dataframe in dataframes.items():
    total_rows = dataframe.count()
    distinct_rows = dataframe.distinct().count()
    duplicate_rows = total_rows - distinct_rows

    duplicate_summary.append(
        {
            "dataset": dataset_name,
            "total_rows": total_rows,
            "distinct_rows": distinct_rows,
            "duplicate_rows": duplicate_rows
        }
    )

duplicate_summary_df = spark.createDataFrame(duplicate_summary)

display(
    duplicate_summary_df.orderBy(
        F.desc("duplicate_rows")
    )
)

dataset,distinct_rows,duplicate_rows,total_rows
behavioural_features,300000,0,300000
departments,50,0,50
images_metadata,45000,0,45000
invoices,300000,0,300000
labels,300000,0,300000
splits,300000,0,300000
suppliers,2000,0,2000


### Identify Possible ID Columns

In [0]:
possible_id_columns = []

for dataset_name, dataframe in dataframes.items():
    for column_name in dataframe.columns:
        lower_name = column_name.lower()

        if (
            lower_name.endswith("_id")
            or lower_name == "id"
            or "invoice_id" in lower_name
            or "supplier_id" in lower_name
            or "department_id" in lower_name
            or "image_id" in lower_name
        ):
            possible_id_columns.append(
                {
                    "dataset": dataset_name,
                    "possible_id_column": column_name
                }
            )

possible_id_columns_df = spark.createDataFrame(
    possible_id_columns
)

display(
    possible_id_columns_df.orderBy(
        "possible_id_column",
        "dataset"
    )
)

dataset,possible_id_column
departments,department_id
invoices,department_id
behavioural_features,invoice_id
images_metadata,invoice_id
invoices,invoice_id
labels,invoice_id
splits,invoice_id
invoices,supplier_id
suppliers,supplier_id


### Compare Common Column Names

In [0]:
dataset_column_sets = {
    dataset_name: set(dataframe.columns)
    for dataset_name, dataframe in dataframes.items()
}

common_column_results = []

dataset_names = list(dataset_column_sets.keys())

for i in range(len(dataset_names)):
    for j in range(i + 1, len(dataset_names)):
        dataset_1 = dataset_names[i]
        dataset_2 = dataset_names[j]

        common_columns = sorted(
            dataset_column_sets[dataset_1].intersection(
                dataset_column_sets[dataset_2]
            )
        )

        for common_column in common_columns:
            common_column_results.append(
                {
                    "dataset_1": dataset_1,
                    "dataset_2": dataset_2,
                    "common_column": common_column
                }
            )

if common_column_results:
    common_columns_df = spark.createDataFrame(
        common_column_results
    )

    display(
        common_columns_df.orderBy(
            "common_column",
            "dataset_1"
        )
    )
else:
    print("No exact common column names were found.")

common_column,dataset_1,dataset_2
department_id,departments,invoices
image_path,images_metadata,invoices
invoice_id,behavioural_features,images_metadata
invoice_id,behavioural_features,invoices
invoice_id,behavioural_features,labels
invoice_id,behavioural_features,splits
invoice_id,images_metadata,invoices
invoice_id,images_metadata,labels
invoice_id,images_metadata,splits
invoice_id,invoices,labels


### Inspect Categorical Columns

In [0]:
categorical_summary = []

for dataset_name, dataframe in dataframes.items():
    for column_name, data_type in dataframe.dtypes:
        if data_type == "string":
            distinct_count = (
                dataframe
                .select(column_name)
                .distinct()
                .count()
            )

            categorical_summary.append(
                {
                    "dataset": dataset_name,
                    "column_name": column_name,
                    "distinct_values": distinct_count
                }
            )

if categorical_summary:
    categorical_summary_df = spark.createDataFrame(
        categorical_summary
    )

    display(
        categorical_summary_df.orderBy(
            "dataset",
            "distinct_values"
        )
    )

column_name,dataset,distinct_values
invoice_id,behavioural_features,300000
region,departments,24
department_id,departments,50
invoice_id,images_metadata,45000
image_path,images_metadata,45000
currency,invoices,1
invoice_type,invoices,2
payment_terms,invoices,3
department_id,invoices,50
supplier_id,invoices,2000


### Examine Label Distribution

In [0]:
display(labels_df.limit(20))

labels_df.printSchema()

invoice_id,is_fraud,fraud_type,fraud_tags,explanations
INV_0000000,0,NONE,,"{""reason"": ""NONE""}"
INV_0000001,0,NONE,,"{""reason"": ""NONE""}"
INV_0000002,0,NONE,,"{""reason"": ""NONE""}"
INV_0000003,1,SPLIT,SPLIT,"{""rules"": [""SPLIT""]}"
INV_0000004,0,NONE,,"{""reason"": ""NONE""}"
INV_0000005,0,NONE,,"{""reason"": ""NONE""}"
INV_0000006,1,NONE,,"{""reason"": ""NONE""}"
INV_0000007,0,NONE,,"{""reason"": ""NONE""}"
INV_0000008,0,NONE,,"{""reason"": ""NONE""}"
INV_0000009,0,NONE,,"{""reason"": ""NONE""}"


root
 |-- invoice_id: string (nullable = true)
 |-- is_fraud: long (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- fraud_tags: string (nullable = true)
 |-- explanations: string (nullable = true)



In [0]:
for column_name, data_type in labels_df.dtypes:
    distinct_count = (
        labels_df
        .select(column_name)
        .distinct()
        .count()
    )

    if distinct_count <= 20:
        print(
            f"\nDistribution for: {column_name} "
            f"({distinct_count} unique values)"
        )

        display(
            labels_df
            .groupBy(column_name)
            .count()
            .orderBy(F.desc("count"))
        )


Distribution for: is_fraud (2 unique values)


is_fraud,count
0,233590
1,66410



Distribution for: fraud_type (6 unique values)


fraud_type,count
NONE,237036
SPLIT,42346
GHOST_SUPPLIER,12588
INFLATED,5581
DOC_TAMPER,2379
DUPLICATE,70



Distribution for: fraud_tags (16 unique values)


fraud_tags,count
,237036
SPLIT,40201
GHOST_SUPPLIER,12588
INFLATED,5121
SPLIT;GHOST_SUPPLIER,2144
DOC_TAMPER,1826
INFLATED;GHOST_SUPPLIER,460
SPLIT;DOC_TAMPER,304
GHOST_SUPPLIER;DOC_TAMPER,143
INFLATED;DOC_TAMPER,77



Distribution for: explanations (16 unique values)


explanations,count
"{""reason"": ""NONE""}",237036
"{""rules"": [""SPLIT""]}",40201
"{""rules"": [""GHOST_SUPPLIER""]}",12588
"{""rules"": [""INFLATED""]}",5121
"{""rules"": [""SPLIT"", ""GHOST_SUPPLIER""]}",2144
"{""rules"": [""DOC_TAMPER""]}",1826
"{""rules"": [""INFLATED"", ""GHOST_SUPPLIER""]}",460
"{""rules"": [""SPLIT"", ""DOC_TAMPER""]}",304
"{""rules"": [""GHOST_SUPPLIER"", ""DOC_TAMPER""]}",143
"{""rules"": [""INFLATED"", ""DOC_TAMPER""]}",77


### Examine Train, Validation and Test Splits

In [0]:
display(splits_df.limit(20))

splits_df.printSchema()

invoice_id,split
INV_0258729,train
INV_0179434,train
INV_0123072,train
INV_0031388,train
INV_0205955,train
INV_0252070,train
INV_0083680,train
INV_0257170,train
INV_0269808,train
INV_0041949,train


root
 |-- invoice_id: string (nullable = true)
 |-- split: string (nullable = true)



In [0]:
for column_name, data_type in splits_df.dtypes:
    distinct_count = (
        splits_df
        .select(column_name)
        .distinct()
        .count()
    )

    if distinct_count <= 20:
        print(f"\nDistribution for: {column_name}")

        display(
            splits_df
            .groupBy(column_name)
            .count()
            .orderBy(F.desc("count"))
        )


Distribution for: split


split,count
train,210000
test,60000
val,30000


### Examine Image Metadata

In [0]:
print(f"Image metadata records: {images_metadata_df.count():,}")

images_metadata_df.printSchema()

display(
    images_metadata_df.limit(20)
)

Image metadata records: 45,000
root
 |-- invoice_id: string (nullable = true)
 |-- image_path: string (nullable = true)
 |-- ocr_total_extracted: double (nullable = true)
 |-- image_tamper_flag: long (nullable = true)



invoice_id,image_path,ocr_total_extracted,image_tamper_flag
INV_0004941,data/v1/20251214_172643/images/INV_0004941.png,52538.04,0
INV_0051775,data/v1/20251214_172643/images/INV_0051775.png,125344.24,0
INV_0115253,data/v1/20251214_172643/images/INV_0115253.png,63871.29,0
INV_0299321,data/v1/20251214_172643/images/INV_0299321.png,6622.06,0
INV_0173570,data/v1/20251214_172643/images/INV_0173570.png,23200.69,0
INV_0030862,data/v1/20251214_172643/images/INV_0030862.png,16244.76,0
INV_0244471,data/v1/20251214_172643/images/INV_0244471.png,41376.06,1
INV_0127794,data/v1/20251214_172643/images/INV_0127794.png,6146.85,0
INV_0071558,data/v1/20251214_172643/images/INV_0071558.png,6380.79,0
INV_0218011,data/v1/20251214_172643/images/INV_0218011.png,14441.76,0


In [0]:
for column_name in images_metadata_df.columns:
    if any(
        keyword in column_name.lower()
        for keyword in ["image", "file", "path", "name"]
    ):
        print(column_name)

image_path
image_tamper_flag


### Save Dataset Summary Tables

In [0]:
WEEK_1_OUTPUT_PATH = f"{BASE_PATH}/outputs/week_01"

dbutils.fs.mkdirs(WEEK_1_OUTPUT_PATH)

print(f"Week 1 output folder: {WEEK_1_OUTPUT_PATH}")

Week 1 output folder: /Volumes/workspace/default/dataanalytics/RiskCheck AI//outputs/week_01


In [0]:
(
    dataset_summary_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{WEEK_1_OUTPUT_PATH}/dataset_summary")
)

In [0]:
(
    column_inventory_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{WEEK_1_OUTPUT_PATH}/column_inventory")
)

In [0]:
(
    duplicate_summary_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{WEEK_1_OUTPUT_PATH}/duplicate_summary")
)

# Week 1 Conclusion

## Overview

The objective of Week 1 was to collect, load, and understand the datasets required for the RiskCheck AI project. All available project files were successfully verified, and the structured data was imported into Databricks for further analysis.

A total of seven Parquet datasets were loaded successfully, including invoice records, behavioural features, suppliers, departments, labels, dataset splits, and image metadata. In addition, the project contains a compressed archive of invoice images that will be processed during the image analysis stage.

## Key Findings

- Successfully loaded all seven structured datasets into Spark DataFrames.
- The invoice, behavioural features, labels, and dataset split tables each contain **300,000 records**, indicating that they represent the same invoice population.
- The supplier dataset contains **2,000 records**, while the department dataset contains **50 records**, serving as reference tables.
- The image metadata dataset contains **45,000 records**, confirming that invoice images are available for a subset of the invoices.
- Dataset schemas, column names, and data types were successfully examined to understand the overall data structure.
- Initial data quality checks, including missing value and duplicate record analysis, were completed to identify potential data quality issues before further processing.
- Potential relationships between datasets were identified, providing the foundation for future data integration and feature engineering.

## Outcome

The completion of Week 1 establishes a reliable foundation for the RiskCheck AI project. The datasets have been successfully prepared for exploratory data analysis, where statistical summaries, visualizations, and relationships between invoice, supplier, behavioural, and image data will be explored in greater detail.

## Next Steps

Week 2 will focus on **Exploratory Data Analysis (EDA)** to:
- Analyse invoice characteristics and transaction patterns.
- Explore supplier and department information.
- Examine behavioural features and fraud labels.
- Study the distribution of training and testing datasets.
- Investigate image metadata and its relationship with invoice records.
- Generate visualizations and business insights that will guide feature engineering and machine learning model development.

# Week 2: Exploratory Data Analysis and Fraud Pattern Investigation

## Overview

Week 2 focuses on exploratory data analysis of the RiskCheck AI datasets to better understand invoice behaviour, supplier characteristics, fraud patterns, and the relationships between variables that may contribute to fraudulent activity.

The objective of this stage is to move beyond basic data preparation and investigate how legitimate and fraudulent invoices differ across financial, behavioural, supplier, departmental, and image-related characteristics.

Exploratory analysis provides the foundation for later feature engineering and machine learning by identifying potentially useful fraud indicators, unusual patterns, class distributions, and relationships within the data.

## Week 2 Objectives

- Examine the overall distribution of legitimate and fraudulent invoices.
- Analyse invoice amounts and transaction characteristics.
- Investigate behavioural indicators associated with fraudulent activity.
- Examine supplier characteristics and supplier-related fraud risk.
- Analyse departmental and regional invoice patterns.
- Investigate duplicate, split-invoice, and unusual submission behaviour.
- Examine available image and OCR-related fraud indicators.
- Compare important variables between legitimate and fraudulent invoices.
- Identify patterns and observations that can support feature engineering and model development.
- Summarise the key exploratory findings for the next stage of RiskCheck AI.

## Dataset Overview

This section provides a consolidated overview of all datasets used in the RiskCheck AI project. It summarizes the number of records and columns in each dataset, helping to understand the overall size and structure of the available data before detailed exploratory analysis.

In [0]:
dataset_overview = []

for dataset_name, dataframe in dataframes.items():
    dataset_overview.append({
        "Dataset": dataset_name,
        "Rows": dataframe.count(),
        "Columns": len(dataframe.columns)
    })

dataset_overview_df = spark.createDataFrame(dataset_overview)

display(dataset_overview_df.orderBy("Dataset"))

Columns,Dataset,Rows
7,behavioural_features,300000
3,departments,50
4,images_metadata,45000
10,invoices,300000
5,labels,300000
2,splits,300000
6,suppliers,2000


## Descriptive Statistics

Descriptive statistics provide a summary of the numerical attributes within the datasets. These statistics help identify the range, central tendency, and variability of the data, and can reveal potential anomalies or outliers before further analysis.

### Invoice Statistics


In [0]:
display(invoices_df.describe())

summary,invoice_id,supplier_id,department_id,invoice_amount,currency,payment_terms,invoice_type,submission_hour,image_path
count,300000,300000,300000,300000,300000,300000,300000,300000,45000
mean,null,null,null,7564.708713466672,null,null,null,11.485236666666667,null
stddev,null,null,null,8812.143357941082,null,null,null,6.92253001308492,null
min,INV_0000000,004b6fab-fcf5-4188-932e-6dcd83bc9478,DPT_000,37.35,ZAR,NET30,GOODS,0,data/v1/20251214_172643/images/INV_0000000.png
max,INV_0299999,fffe77c8-39fe-499c-b127-41487c7c404e,DPT_049,294048.72,ZAR,NET90,SERVICES,23,data/v1/20251214_172643/images/INV_0299983.png


### Behavioural Feature Statistics

In [0]:
display(behavioural_features_df.describe())

summary,invoice_id,supplier_invoice_count_30d,supplier_avg_amount_90d,invoice_amount_zscore,duplicate_invoice_flag,split_invoice_flag,late_night_submission_flag
count,300000,300000,300000,300000,300000,300000,300000
mean,null,12.635893333333334,7572.864588519553,6.669627813001474E-17,2.3333333333333333E-4,0.14226333333333332,0.25054666666666664
stddev,null,3.918757990848145,6240.05046833862,1.0000016666708331,0.015273495555627637,0.3493206035604482,0.4333285824880355
min,INV_0000000,1.0,193.33,-0.8542043579343926,0,0,0
max,INV_0299999,31.0,294048.72,32.510193845508994,1,1,1


### Supplier Statistics (if numeric columns exist)

In [0]:
display(suppliers_df.describe())

summary,supplier_id,supplier_country,supplier_age_days,supplier_risk_score,blacklisted_flag,avg_invoice_amount
count,2000,2000,2000,2000,2000,2000
mean,null,null,2256.2685,0.287007,0.0505,6312.936765000002
stddev,null,null,1567.5479254737565,0.15377970623598228,0.21902907767790608,5017.624788722298
min,004b6fab-fcf5-4188-932e-6dcd83bc9478,AD,30,0.002,0,323.29
max,fffe77c8-39fe-499c-b127-41487c7c404e,ZW,4999,0.861,1,43064.45


## Invoice Data Analysis

The invoice dataset is the primary source of transactional information in the RiskCheck AI project. This section explores the key characteristics of the invoice records, including invoice amounts, invoice types, payment terms, submission times, and image availability.

The objective is to understand the overall structure of the invoice data, identify potential data quality issues, and discover patterns that may be useful for risk assessment and fraud detection.

### Invoice Dataset Overview

The following analysis provides a basic summary of the invoice dataset, including the total number of records and available attributes.

In [0]:
print("=" * 60)
print("Invoice Dataset Overview")
print("=" * 60)

print(f"Total invoices : {invoices_df.count():,}")
print(f"Total columns  : {len(invoices_df.columns)}")

print("\nColumns:")
for i, column in enumerate(invoices_df.columns, start=1):
    print(f"{i}. {column}")

Invoice Dataset Overview
Total invoices : 300,000
Total columns  : 10

Columns:
1. invoice_id
2. supplier_id
3. department_id
4. invoice_date
5. invoice_amount
6. currency
7. payment_terms
8. invoice_type
9. submission_hour
10. image_path


### Invoice Date Range

This analysis identifies the time period covered by the invoice dataset.

In [0]:
from pyspark.sql import functions as F

invoices_df.select(
    F.min("invoice_date").alias("Earliest Invoice Date"),
    F.max("invoice_date").alias("Latest Invoice Date")
).show()

+---------------------+-------------------+
|Earliest Invoice Date|Latest Invoice Date|
+---------------------+-------------------+
|  2023-01-01 00:00:00|2023-12-31 00:00:00|
+---------------------+-------------------+



### Invoice Amount Statistics

Descriptive statistics provide an overview of the distribution of invoice amounts and help identify unusually high or low values.

In [0]:
display(
    invoices_df.select("invoice_amount").describe()
)

summary,invoice_amount
count,300000
mean,7564.708713466672
stddev,8812.143357941082
min,37.35
max,294048.72


### Invoice Amount Distribution

The distribution of invoice amounts is examined to understand how invoice values are spread across the dataset and to identify potential outliers.

In [0]:
display(
    invoices_df.select("invoice_amount")
)

invoice_amount
3565.57
9871.78
8530.64
1378.52
2373.66
6652.67
4159.75
4028.91
5573.19
24484.1


Databricks visualization. Run in Databricks to view.

### Payment Terms Distribution

This analysis shows how invoices are distributed across different payment terms.

In [0]:
payment_terms_distribution = (
    invoices_df
    .groupBy("payment_terms")
    .count()
    .orderBy(F.desc("count"))
)

display(payment_terms_distribution)

payment_terms,count
NET30,180337
NET60,89883
NET90,29780


Databricks visualization. Run in Databricks to view.

### Invoice Type Distribution

This analysis examines the frequency of each invoice type within the dataset.

In [0]:
invoice_type_distribution = (
    invoices_df
    .groupBy("invoice_type")
    .count()
    .orderBy(F.desc("count"))
)

display(invoice_type_distribution)

invoice_type,count
GOODS,164863
SERVICES,135137


Databricks visualization. Run in Databricks to view.

### Submission Hour Distribution

This analysis examines the time of day at which invoices are submitted, helping to identify operational patterns.

In [0]:
submission_distribution = (
    invoices_df
    .groupBy("submission_hour")
    .count()
    .orderBy("submission_hour")
)

display(submission_distribution)

submission_hour,count
0,12457
1,12468
2,12634
3,12537
4,12460
5,12608
6,12662
7,12264
8,12628
9,12569


Databricks visualization. Run in Databricks to view.

### Invoice Image Availability

This analysis determines how many invoice records are associated with an image, which is important for the image processing stage of the project.

In [0]:
image_availability = invoices_df.select(
    F.count("*").alias("Total Invoices"),
    F.count("image_path").alias("Invoices with Images")
)

display(image_availability)

Total Invoices,Invoices with Images
300000,45000


### Missing Values

This analysis checks whether any invoice attributes contain missing values that may require preprocessing.

In [0]:
missing_invoice_values = invoices_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)

    for column in invoices_df.columns
])

display(missing_invoice_values)

invoice_id,supplier_id,department_id,invoice_date,invoice_amount,currency,payment_terms,invoice_type,submission_hour,image_path
0,0,0,0,0,0,0,0,0,255000


### Duplicate Invoice IDs

Invoice identifiers should be unique. This analysis checks for duplicate invoice IDs.

In [0]:
duplicate_invoice_ids = (
    invoices_df
    .groupBy("invoice_id")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate Invoice IDs: {duplicate_invoice_ids.count():,}")

Duplicate Invoice IDs: 0


### Invoice Analysis Summary

The invoice dataset contains 300,000 records and serves as the primary transactional dataset for the RiskCheck AI project.

The exploratory analysis examined invoice amounts, payment terms, invoice types, submission times, image availability, missing values, and duplicate invoice identifiers. These analyses provide an understanding of the overall characteristics of the invoice data and establish a strong foundation for the supplier, department, behavioural, and fraud analyses that follow.

## Supplier Analysis

Suppliers play a critical role in the invoice approval process. This section examines supplier-related information to understand supplier activity and identify whether invoice transactions are concentrated among a small number of suppliers.

Understanding supplier behaviour can provide valuable insights for fraud detection, as unusual invoice patterns may be associated with specific suppliers.

### Supplier Dataset Overview

In [0]:
print("=" * 60)
print("Supplier Dataset Overview")
print("=" * 60)

print(f"Total Suppliers : {suppliers_df.count():,}")
print(f"Total Columns   : {len(suppliers_df.columns)}")

print("\nColumns:")

for i, column in enumerate(suppliers_df.columns, start=1):
    print(f"{i}. {column}")

Supplier Dataset Overview
Total Suppliers : 2,000
Total Columns   : 6

Columns:
1. supplier_id
2. supplier_country
3. supplier_age_days
4. supplier_risk_score
5. blacklisted_flag
6. avg_invoice_amount


In [0]:
display(suppliers_df.limit(10))

supplier_id,supplier_country,supplier_age_days,supplier_risk_score,blacklisted_flag,avg_invoice_amount
fc3e058b-e0f3-4ab0-9cec-4eb5edd96831,SV,841,0.244,0,5590.25
3d4cbf37-4eb9-4eff-8e88-cb2dd4e80839,KE,61,0.284,0,5797.64
913e4de2-e0c5-4cb8-bda9-c2a90ed42f1a,YE,197,0.39,0,897.65
bb5e4bcf-15ed-4269-9429-6c07f26b4776,SA,851,0.299,0,7727.71
fa5d3100-11b7-4948-90e6-e6607c69dee1,DJ,4373,0.418,0,5458.67
2031d750-c40d-49b4-885f-6e66c2b6d2c5,IT,1218,0.5,0,7294.2
f264accc-79ac-4b1e-a8e5-6e0c20de435d,IN,81,0.304,0,1996.62
8715a103-43da-4043-aa45-c2ab8cbfedb0,CG,111,0.151,0,3940.09
f6e07cc0-6c52-449f-9b49-bd26df57c59a,CA,4554,0.329,0,2289.14
c1590f53-8a0f-4efb-adcd-465e36386821,IN,1394,0.524,0,8276.88


### Supplier Country Distribution

This analysis examines the geographical distribution of suppliers participating in the invoice processing system.

In [0]:
country_distribution = (
    suppliers_df
    .groupBy("supplier_country")
    .count()
    .orderBy(F.desc("count"))
)

display(country_distribution)

supplier_country,count
KZ,42
IN,41
DZ,36
CG,36
DE,36
NO,35
CU,35
DJ,34
MU,34
IQ,33


Databricks visualization. Run in Databricks to view.

### Supplier Age Analysis

Supplier age represents the number of days since the supplier was registered. Older suppliers may demonstrate more established business relationships, while newer suppliers could require additional scrutiny.

In [0]:
display(
    suppliers_df.select("supplier_age_days").describe()
)

summary,supplier_age_days
count,2000
mean,2256.2685
stddev,1567.5479254737565
min,30
max,4999


In [0]:
display(
    suppliers_df.select("supplier_age_days")
)

supplier_age_days
841
61
197
851
4373
1218
81
111
4554
1394


Databricks visualization. Run in Databricks to view.

### Supplier Risk Score Analysis

This analysis examines the distribution of supplier risk scores to understand the overall risk profile of suppliers.

In [0]:
display(
    suppliers_df.select("supplier_risk_score").describe()
)

summary,supplier_risk_score
count,2000
mean,0.287007
stddev,0.15377970623598228
min,0.002
max,0.861


In [0]:
display(
    suppliers_df.select("supplier_risk_score")
)

supplier_risk_score
0.244
0.284
0.39
0.299
0.418
0.5
0.304
0.151
0.329
0.524


Databricks visualization. Run in Databricks to view.

### Blacklisted Supplier Analysis

This analysis identifies how many suppliers have been marked as blacklisted.

In [0]:
blacklist_distribution = (
    suppliers_df
    .groupBy("blacklisted_flag")
    .count()
)

display(blacklist_distribution)

blacklisted_flag,count
0,1899
1,101


Databricks visualization. Run in Databricks to view.

### Average Invoice Amount

This analysis examines the historical average invoice amount for suppliers.

In [0]:
display(
    suppliers_df.select("avg_invoice_amount").describe()
)

summary,avg_invoice_amount
count,2000
mean,6312.936765000002
stddev,5017.624788722298
min,323.29
max,43064.45


In [0]:
display(
    suppliers_df.select("avg_invoice_amount")
)

avg_invoice_amount
5590.25
5797.64
897.65
7727.71
5458.67
7294.2
1996.62
3940.09
2289.14
8276.88


Databricks visualization. Run in Databricks to view.

### Top 10 Highest Risk Suppliers

In [0]:
display(
    suppliers_df
    .orderBy(F.desc("supplier_risk_score"))
    .select(
        "supplier_id",
        "supplier_country",
        "supplier_risk_score",
        "blacklisted_flag"
    )
    .limit(10)
)

supplier_id,supplier_country,supplier_risk_score,blacklisted_flag
e9462820-6a80-4ac2-9a8c-f13c28ce231e,IT,0.861,0
61985d54-cfb8-4e6f-a9d6-8f23b489d070,PT,0.844,0
ecfedb99-2790-4ebd-bfdd-c3d99ee3ac2a,SZ,0.816,0
0977c513-752a-4d25-b190-1b7ec6b469ef,KZ,0.81,0
48b763d5-e519-4d7e-84d6-7c4ff0df1684,TV,0.793,0
f54e2019-ba35-444e-99e1-ac095970a859,NO,0.785,0
2acb4fbd-fbff-4a7d-b2f5-b1762c41cbd3,VC,0.77,0
cbd58bf6-1efd-46e9-8e37-14af99b49350,GN,0.753,0
bdc2b74b-38a5-4b49-9d34-aa1890019ed4,NL,0.752,0
a7f333b3-f7ba-455e-8f6b-58c8598ddaec,TG,0.747,0


###Join Suppliers with Invoices

In [0]:
supplier_invoice_counts = (
    invoices_df
    .groupBy("supplier_id")
    .count()
    .withColumnRenamed("count", "total_invoices")
)

supplier_summary = (
    suppliers_df
    .join(
        supplier_invoice_counts,
        on="supplier_id",
        how="left"
    )
)

display(
    supplier_summary.orderBy(
        F.desc("total_invoices")
    )
)

supplier_id,supplier_country,supplier_age_days,supplier_risk_score,blacklisted_flag,avg_invoice_amount,total_invoices
04aac1b7-5ca0-4428-822c-4d326c645c15,YE,4114,0.279,0,3593.52,201
65d0b517-88d0-4abd-8bb1-8c791036708d,TD,1300,0.339,0,1237.7,193
61542765-d756-4a9c-864c-96479ea7017c,AM,1295,0.344,0,4215.77,192
654c11d9-f2e6-4828-81bb-819c4dfe1117,IN,1768,0.255,0,3644.09,191
d7665cda-fe04-4059-b985-fb6217dc8eff,TN,446,0.451,0,12177.93,190
65f202f9-83f0-4dc7-8f61-2217eef16694,ST,4930,0.293,0,24381.55,189
3692a531-c609-4cd8-a127-dc6bab9b153d,IN,3945,0.297,0,3883.44,187
9c81fc46-5bcd-436c-8777-4fae613758c3,AG,3863,0.051,0,27877.7,186
82c299f0-7487-4772-bf03-86fa13badc91,PE,3949,0.159,0,3488.94,184
67443fb8-81bd-45d5-b5a6-571c4fa5c35e,AU,2910,0.656,1,13248.14,183


###Top 10 Most Active Suppliers

In [0]:
display(
    supplier_summary
    .select(
        "supplier_id",
        "supplier_country",
        "total_invoices",
        "supplier_risk_score",
        "blacklisted_flag"
    )
    .orderBy(F.desc("total_invoices"))
    .limit(10)
)

supplier_id,supplier_country,total_invoices,supplier_risk_score,blacklisted_flag
04aac1b7-5ca0-4428-822c-4d326c645c15,YE,201,0.279,0
65d0b517-88d0-4abd-8bb1-8c791036708d,TD,193,0.339,0
61542765-d756-4a9c-864c-96479ea7017c,AM,192,0.344,0
654c11d9-f2e6-4828-81bb-819c4dfe1117,IN,191,0.255,0
d7665cda-fe04-4059-b985-fb6217dc8eff,TN,190,0.451,0
65f202f9-83f0-4dc7-8f61-2217eef16694,ST,189,0.293,0
3692a531-c609-4cd8-a127-dc6bab9b153d,IN,187,0.297,0
9c81fc46-5bcd-436c-8777-4fae613758c3,AG,186,0.051,0
82c299f0-7487-4772-bf03-86fa13badc91,PE,184,0.159,0
67443fb8-81bd-45d5-b5a6-571c4fa5c35e,AU,183,0.656,1


### Supplier Analysis Summary

The supplier analysis explored supplier demographics, historical activity, and risk-related attributes.

The analysis examined supplier countries, supplier age, historical invoice values, risk scores, blacklist status, and invoice activity. By combining supplier information with invoice records, it becomes possible to identify suppliers that process large numbers of invoices while also exhibiting elevated risk characteristics.

These findings provide valuable context for subsequent fraud detection and machine learning analyses.

## Department Analysis

The department dataset provides organizational information about the departments responsible for processing invoices. This section examines the geographical distribution of departments, their annual budgets, and the volume of invoices processed by each department.

Understanding departmental characteristics may help identify operational differences and potential risk patterns across different regions.

###Department Dataset Overview

In [0]:
print("=" * 60)
print("Department Dataset Overview")
print("=" * 60)

print(f"Total Departments : {departments_df.count():,}")
print(f"Total Columns     : {len(departments_df.columns)}")

print("\nColumns:")

for i, column in enumerate(departments_df.columns, start=1):
    print(f"{i}. {column}")

Department Dataset Overview
Total Departments : 50
Total Columns     : 3

Columns:
1. department_id
2. region
3. annual_budget


In [0]:
display(departments_df.limit(10))

department_id,region,annual_budget
DPT_000,Rhode Island,3.2788813503695077E8
DPT_001,Arkansas,2.6668785201303322E7
DPT_002,Nevada,1.4914645502737343E7
DPT_003,New York,4.204102283429599E8
DPT_004,Louisiana,2.956358085560889E8
DPT_005,Arkansas,1.1622908895079409E8
DPT_006,Texas,3.771371745502147E8
DPT_007,New York,1.3552763775179547E8
DPT_008,Louisiana,2.1288906530955267E8
DPT_009,Arizona,2.282605365655489E8


### Department Region Distribution

This analysis examines the geographical distribution of departments across different regions.

In [0]:
region_distribution = (
    departments_df
    .groupBy("region")
    .count()
    .orderBy(F.desc("count"))
)

display(region_distribution)

region,count
Arkansas,5
Connecticut,5
New York,4
Rhode Island,3
Nevada,3
Louisiana,3
Tennessee,3
Texas,2
Arizona,2
Michigan,2


Databricks visualization. Run in Databricks to view.

### Annual Budget Analysis

This analysis summarizes the annual budgets allocated to departments and helps identify budget variability across the organization.

In [0]:
display(
    departments_df.select("annual_budget").describe()
)

summary,annual_budget
count,50
mean,2.2531406387885553E8
stddev,1.466790179322504E8
min,7018473.868064399
max,4.9895146341568965E8


In [0]:
display(
    departments_df.select("annual_budget")
)

annual_budget
3.2788813503695077E8
2.6668785201303322E7
1.4914645502737343E7
4.204102283429599E8
2.956358085560889E8
1.1622908895079409E8
3.771371745502147E8
1.3552763775179547E8
2.1288906530955267E8
2.282605365655489E8


Databricks visualization. Run in Databricks to view.

### Invoice Count by Department

In [0]:
department_invoice_counts = (
    invoices_df
    .groupBy("department_id")
    .count()
    .withColumnRenamed("count", "total_invoices")
)

department_summary = (
    departments_df
    .join(
        department_invoice_counts,
        on="department_id",
        how="left"
    )
)

display(department_summary)

department_id,region,annual_budget,total_invoices
DPT_001,Arkansas,2.6668785201303322E7,5950
DPT_003,New York,4.204102283429599E8,6002
DPT_002,Nevada,1.4914645502737343E7,6045
DPT_005,Arkansas,1.1622908895079409E8,5940
DPT_000,Rhode Island,3.2788813503695077E8,5943
DPT_004,Louisiana,2.956358085560889E8,5985
DPT_010,Arizona,4.7788071671451265E8,5962
DPT_011,Michigan,4.464913262456061E8,5990
DPT_006,Texas,3.771371745502147E8,5964
DPT_008,Louisiana,2.1288906530955267E8,6038


### Top Departments by Invoice Volume

In [0]:
display(
    department_summary
    .select(
        "department_id",
        "region",
        "annual_budget",
        "total_invoices"
    )
    .orderBy(F.desc("total_invoices"))
)

department_id,region,annual_budget,total_invoices
DPT_013,Missouri,1.4287447740079096E8,6143
DPT_009,Arizona,2.282605365655489E8,6132
DPT_039,Louisiana,2.118289859007984E8,6103
DPT_036,New Hampshire,1.7400541758110693E8,6099
DPT_044,Tennessee,2.923571187866788E8,6097
DPT_030,Wyoming,1.5350954844925362E8,6085
DPT_021,North Dakota,4.728377066746403E8,6081
DPT_043,Oregon,4.165765147833375E7,6079
DPT_025,North Carolina,2.1713503886269618E7,6072
DPT_019,Connecticut,2.69888487263314E8,6071


Databricks visualization. Run in Databricks to view.

### Budget vs Invoice Volume

In [0]:
display(
    department_summary.select(
        "annual_budget",
        "total_invoices"
    )
)

annual_budget,total_invoices
2.6668785201303322E7,5950
4.204102283429599E8,6002
1.4914645502737343E7,6045
1.1622908895079409E8,5940
3.2788813503695077E8,5943
2.956358085560889E8,5985
4.7788071671451265E8,5962
4.464913262456061E8,5990
3.771371745502147E8,5964
2.1288906530955267E8,6038


Databricks visualization. Run in Databricks to view.

### Department Analysis Summary

The department analysis examined the geographical distribution of departments, annual budget allocation, and invoice processing activity.

By combining departmental information with invoice records, this analysis provides insight into how invoice workloads are distributed across the organization. The relationship between departmental budgets and invoice volumes may help identify operational trends that are useful for later feature engineering and risk analysis.

## Step 30: Behavioural Feature Analysis

Behavioural features are engineered variables that describe invoice and supplier behaviour. These features are designed to capture patterns that may indicate unusual or potentially fraudulent activity.

This section examines the distribution of behavioural features, identifies potential outliers, and analyses the occurrence of behavioural flags. These insights will support feature engineering and machine learning model development.

### Behavioural Dataset Overview

In [0]:
print("=" * 60)
print("Behavioural Feature Dataset Overview")
print("=" * 60)

print(f"Total Records : {behavioural_features_df.count():,}")
print(f"Total Columns : {len(behavioural_features_df.columns)}")

print("\nFeatures:")

for i, column in enumerate(behavioural_features_df.columns, start=1):
    print(f"{i}. {column}")

Behavioural Feature Dataset Overview
Total Records : 300,000
Total Columns : 7

Features:
1. invoice_id
2. supplier_invoice_count_30d
3. supplier_avg_amount_90d
4. invoice_amount_zscore
5. duplicate_invoice_flag
6. split_invoice_flag
7. late_night_submission_flag


### 30.2 Numerical Feature Statistics

The numerical behavioural features are summarised to understand their central tendency, spread, and overall distribution.

In [0]:
numerical_features = [
    "supplier_invoice_count_30d",
    "supplier_avg_amount_90d",
    "invoice_amount_zscore"
]

display(
    behavioural_features_df.select(numerical_features).describe()
)

summary,supplier_invoice_count_30d,supplier_avg_amount_90d,invoice_amount_zscore
count,300000,300000,300000
mean,12.635893333333334,7572.864588519553,6.669627813001474E-17
stddev,3.918757990848145,6240.05046833862,1.0000016666708331
min,1.0,193.33,-0.8542043579343926
max,31.0,294048.72,32.510193845508994


### Numerical Feature Distributions
### Supplier Invoice Count

In [0]:
display(
    behavioural_features_df.select("supplier_invoice_count_30d")
)

supplier_invoice_count_30d
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0
10.0


Databricks visualization. Run in Databricks to view.

### Supplier Average Amount

In [0]:
display(
    behavioural_features_df.select("supplier_avg_amount_90d")
)

supplier_avg_amount_90d
4252.16
13972.275
17737.206666666665
14480.34
12515.047999999999
11484.126666666665
10725.707142857142
9947.66
9207.205555555556
9762.520999999999


Databricks visualization. Run in Databricks to view.

### Invoice Amount Z-Score

In [0]:
display(
    behavioural_features_df.select("invoice_amount_zscore")
)

invoice_amount_zscore
-0.3759078920805204
1.8301686106293111
2.0088632324155307
-0.32398173245638195
-0.33032072296428083
-0.1401691645207489
-0.15768252658813237
-0.3476320907047029
-0.48582344553640616
0.8165622127369356


Databricks visualization. Run in Databricks to view.

### 30.4 Behavioural Flag Analysis

The binary behavioural flags indicate whether a particular behavioural condition has been observed for an invoice. The frequency of each flag provides insight into how common these behaviours are within the dataset.

### Duplicate Invoice Flag

In [0]:
display(
    behavioural_features_df
    .groupBy("duplicate_invoice_flag")
    .count()
)

duplicate_invoice_flag,count
0,299930
1,70


Databricks visualization. Run in Databricks to view.

### Split Invoice Flag

In [0]:
display(
    behavioural_features_df
    .groupBy("split_invoice_flag")
    .count()
)

split_invoice_flag,count
0,257321
1,42679


Databricks visualization. Run in Databricks to view.

### Late Night Submission Flag

In [0]:
display(
    behavioural_features_df
    .groupBy("late_night_submission_flag")
    .count()
)

late_night_submission_flag,count
0,224836
1,75164


Databricks visualization. Run in Databricks to view.

### Outlier Analysis

In [0]:
display(
    behavioural_features_df.select(
        "invoice_amount_zscore"
    ).summary(
        "count",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

summary,invoice_amount_zscore
count,300000
min,-0.8542043579343926
25%,-0.5586798487924256
50%,-0.30337263710837764
75%,0.18158623713644967
max,32.510193845508994


### Correlation Analysis

In [0]:
from itertools import combinations

numerical_features = [
    "supplier_invoice_count_30d",
    "supplier_avg_amount_90d",
    "invoice_amount_zscore"
]

correlation_results = []

for feature_1, feature_2 in combinations(numerical_features, 2):
    correlation = behavioural_features_df.stat.corr(feature_1, feature_2)

    correlation_results.append({
        "Feature 1": feature_1,
        "Feature 2": feature_2,
        "Correlation": round(correlation, 4)
    })

display(
    spark.createDataFrame(correlation_results)
)

Correlation,Feature 1,Feature 2
-0.0019,supplier_invoice_count_30d,supplier_avg_amount_90d
-0.0031,supplier_invoice_count_30d,invoice_amount_zscore
0.7066,supplier_avg_amount_90d,invoice_amount_zscore


### Relationship with Risk Labels

In [0]:
behaviour_label_df = (
    behavioural_features_df
    .join(labels_df, on="invoice_id", how="inner")
)

display(behaviour_label_df.limit(10))

invoice_id,supplier_invoice_count_30d,supplier_avg_amount_90d,invoice_amount_zscore,duplicate_invoice_flag,split_invoice_flag,late_night_submission_flag,is_fraud,fraud_type,fraud_tags,explanations
INV_0000056,15.0,6832.553529411764,0.10168159362898728,0,0,1,0,NONE,,"{""reason"": ""NONE""}"
INV_0000134,8.0,8236.6828125,-0.4951412849671899,0,1,0,1,SPLIT,SPLIT,"{""rules"": [""SPLIT""]}"
INV_0000183,18.0,19305.444375,1.8845096237802683,0,0,0,0,NONE,,"{""reason"": ""NONE""}"
INV_0000204,13.0,3671.65,-0.4976242264693888,0,0,0,0,NONE,,"{""reason"": ""NONE""}"
INV_0000339,2.0,18272.11,1.6240129735726576,0,0,1,0,NONE,,"{""reason"": ""NONE""}"
INV_0000343,16.0,10093.001463414634,0.21703851111734282,0,0,0,0,NONE,,"{""reason"": ""NONE""}"
INV_0000351,11.0,2710.166052631579,-0.5524248334541624,0,0,0,0,NONE,,"{""reason"": ""NONE""}"
INV_0000411,10.0,10977.573235294118,0.7452026101396915,0,0,1,0,NONE,,"{""reason"": ""NONE""}"
INV_0000420,12.0,9120.36023255814,0.940472114732637,0,0,0,0,NONE,,"{""reason"": ""NONE""}"
INV_0000431,14.0,5075.238095238095,-0.010039425974064185,0,0,0,0,NONE,,"{""reason"": ""NONE""}"


### Behavioural Feature Summary

The behavioural feature analysis examined both continuous and binary behavioural indicators derived from invoice and supplier activity.

The numerical features describe historical supplier behaviour and the relative size of invoice amounts, while the binary flags identify specific behavioural events such as duplicate invoices, split invoices, and late-night submissions.

These engineered features provide valuable predictive information and are expected to play a significant role in the machine learning models developed in the later stages of the RiskCheck AI project.

## Label Distribution Analysis

The label dataset contains the target variables used for fraud detection. This section examines the distribution of fraudulent and non-fraudulent invoices, the different fraud categories, and associated fraud tags.

Understanding the class distribution is essential before developing machine learning models, as highly imbalanced datasets may require specialised modelling techniques.

### Dataset Overview

In [0]:
print("=" * 60)
print("Label Dataset Overview")
print("=" * 60)

print(f"Total Records : {labels_df.count():,}")
print(f"Total Columns : {len(labels_df.columns)}")

print("\nColumns:")

for i, column in enumerate(labels_df.columns, start=1):
    print(f"{i}. {column}")

Label Dataset Overview
Total Records : 300,000
Total Columns : 5

Columns:
1. invoice_id
2. is_fraud
3. fraud_type
4. fraud_tags
5. explanations


### Fraud vs Non-Fraud Distribution

In [0]:
fraud_distribution = (
    labels_df
    .groupBy("is_fraud")
    .count()
    .orderBy("is_fraud")
)

display(fraud_distribution)

is_fraud,count
0,233590
1,66410


Databricks visualization. Run in Databricks to view.

### Fraud Percentage

In [0]:
total = labels_df.count()

fraud_percentage = (
    labels_df
    .groupBy("is_fraud")
    .count()
    .withColumn(
        "percentage",
        F.round(F.col("count") / total * 100, 2)
    )
)

display(fraud_percentage)

is_fraud,count,percentage
0,233590,77.86
1,66410,22.14


### Fraud Type Distribution

In [0]:
fraud_type_distribution = (
    labels_df
    .groupBy("fraud_type")
    .count()
    .orderBy(F.desc("count"))
)

display(fraud_type_distribution)

fraud_type,count
NONE,237036
SPLIT,42346
GHOST_SUPPLIER,12588
INFLATED,5581
DOC_TAMPER,2379
DUPLICATE,70


Databricks visualization. Run in Databricks to view.

### Fraud Tags Distribution

In [0]:
fraud_tags_distribution = (
    labels_df
    .groupBy("fraud_tags")
    .count()
    .orderBy(F.desc("count"))
)

display(fraud_tags_distribution)

fraud_tags,count
,237036
SPLIT,40201
GHOST_SUPPLIER,12588
INFLATED,5121
SPLIT;GHOST_SUPPLIER,2144
DOC_TAMPER,1826
INFLATED;GHOST_SUPPLIER,460
SPLIT;DOC_TAMPER,304
GHOST_SUPPLIER;DOC_TAMPER,143
INFLATED;DOC_TAMPER,77


### Behavioural Features by Fraud Status

In [0]:
behaviour_label_df = (
    behavioural_features_df
    .join(labels_df, on="invoice_id", how="inner")
)

In [0]:
display(
    behaviour_label_df.groupBy("is_fraud").agg(
        F.avg("supplier_invoice_count_30d").alias("Avg Supplier Invoice Count"),
        F.avg("supplier_avg_amount_90d").alias("Avg Supplier Amount"),
        F.avg("invoice_amount_zscore").alias("Avg Invoice Z-Score")
    )
)

is_fraud,Avg Supplier Invoice Count,Avg Supplier Amount,Avg Invoice Z-Score
0,12.577571813861894,7123.042256500224,-0.015539714048930633
1,12.841032976961301,9155.066042161925,0.05465926524152558


### Duplicate Flag vs Fraud

In [0]:
display(
    behaviour_label_df
    .groupBy(
        "duplicate_invoice_flag",
        "is_fraud"
    )
    .count()
)

duplicate_invoice_flag,is_fraud,count
0,0,233590
0,1,66340
1,1,70


In [0]:
fraud_rate = (
    behaviour_label_df
    .groupBy("duplicate_invoice_flag")
    .agg(
        F.count("*").alias("total_invoices"),
        F.sum("is_fraud").alias("fraud_count")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_count") / F.col("total_invoices") * 100,
            2
        )
    )
)

display(fraud_rate)

duplicate_invoice_flag,total_invoices,fraud_count,fraud_rate_percent
0,299930,66340,22.12
1,70,70,100.0


### Late Night Submission vs Fraud

In [0]:
display(
    behaviour_label_df
    .groupBy(
        "late_night_submission_flag",
        "is_fraud"
    )
    .count()
)

late_night_submission_flag,is_fraud,count
1,0,57144
0,1,48390
1,1,18020
0,0,176446


Databricks visualization. Run in Databricks to view.

### Split Invoice Flag vs Fraud

In [0]:
display(
    behaviour_label_df
    .groupBy(
        "split_invoice_flag",
        "is_fraud"
    )
    .count()
)

split_invoice_flag,is_fraud,count
1,0,879
0,1,24610
1,1,41800
0,0,232711


Databricks visualization. Run in Databricks to view.

### Fraud Label Analysis Summary

The fraud label analysis examined the distribution of fraudulent and non-fraudulent invoices, the prevalence of different fraud categories, and the relationship between behavioural indicators and fraud status.

By combining behavioural features with fraud labels, this analysis provides an initial understanding of which behavioural patterns are associated with fraudulent invoices. These insights will guide feature selection and support the development of predictive machine learning models in the subsequent stages of the RiskCheck AI project.

## Dataset Split Analysis

The dataset split information defines how invoice records are divided into training, validation, and testing subsets for machine learning. This analysis verifies the distribution of records across each subset to ensure that the data is appropriately partitioned for model development and evaluation.

### Dataset Overview

In [0]:
print("=" * 60)
print("Dataset Split Overview")
print("=" * 60)

print(f"Total Records : {splits_df.count():,}")
print(f"Total Columns : {len(splits_df.columns)}")

print("\nColumns:")

for i, column in enumerate(splits_df.columns, start=1):
    print(f"{i}. {column}")

Dataset Split Overview
Total Records : 300,000
Total Columns : 2

Columns:
1. invoice_id
2. split


In [0]:
display(splits_df.limit(10))

invoice_id,split
INV_0258729,train
INV_0179434,train
INV_0123072,train
INV_0031388,train
INV_0205955,train
INV_0252070,train
INV_0083680,train
INV_0257170,train
INV_0269808,train
INV_0041949,train


### Split Distribution

In [0]:
split_distribution = (
    splits_df
    .groupBy("split")
    .count()
    .orderBy("split")
)

display(split_distribution)

split,count
test,60000
train,210000
val,30000


Databricks visualization. Run in Databricks to view.

### Split Percentage

In [0]:
total = splits_df.count()

split_percentage = (
    splits_df
    .groupBy("split")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count") / total * 100,
            2
        )
    )
)

display(split_percentage)

split,count,percentage
train,210000,70.0
val,30000,10.0
test,60000,20.0


### Dataset Split Summary

The dataset split analysis confirms how invoice records are allocated across the training, validation, and testing subsets.

A well-balanced split ensures that the machine learning models can be trained, validated, and evaluated using separate datasets, reducing the risk of overfitting and providing a reliable assessment of model performance.

### Image Tampering Analysis

This analysis examines the number of invoice images that have been flagged as potentially tampered with.

In [0]:
tamper_distribution = (
    images_metadata_df
    .groupBy("image_tamper_flag")
    .count()
)

display(tamper_distribution)

image_tamper_flag,count
0,35908
1,9092


Databricks visualization. Run in Databricks to view.

### Tampering Rate

In [0]:
total_images = images_metadata_df.count()

tamper_rate = (
    images_metadata_df
    .groupBy("image_tamper_flag")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count") / total_images * 100,
            2
        )
    )
)

display(tamper_rate)

image_tamper_flag,count,percentage
0,35908,79.8
1,9092,20.2


### Image Tampering vs Fraud

In [0]:
image_label_df = (
    images_metadata_df
    .join(labels_df, on="invoice_id", how="inner")
)

display(
    image_label_df
    .groupBy("image_tamper_flag", "is_fraud")
    .count()
    .orderBy("image_tamper_flag", "is_fraud")
)

image_tamper_flag,is_fraud,count
0,0,27967
0,1,7941
1,0,7056
1,1,2036


In [0]:
image_fraud_rate = (
    image_label_df
    .groupBy("image_tamper_flag")
    .agg(
        F.count("*").alias("total_images"),
        F.sum("is_fraud").alias("fraud_count")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_count") / F.col("total_images") * 100,
            2
        )
    )
)

display(image_fraud_rate)

image_tamper_flag,total_images,fraud_count,fraud_rate_percent
0,35908,7941,22.11
1,9092,2036,22.39


### Image Metadata Summary

The image metadata analysis examined the availability and quality of invoice images, the volume of text extracted using OCR, and the occurrence of image tampering indicators.

By combining image metadata with fraud labels, the analysis helps determine whether image tampering is associated with fraudulent invoices. These findings support the development of multimodal fraud detection models that incorporate both structured invoice data and visual document features.

## Cross-Dataset Business Insights

The previous sections explored each dataset individually. This section combines information from multiple datasets to identify relationships between invoices, suppliers, departments, behavioural features, image metadata, and fraud labels.

The objective is to uncover business insights that may support fraud detection and identify patterns that can be used during feature engineering and machine learning model development.

### Fraud Rate by Supplier Country

This analysis examines whether fraud occurrence varies across supplier countries.

In [0]:
supplier_country_fraud = (
    invoices_df
    .join(suppliers_df, "supplier_id")
    .join(labels_df, "invoice_id")
    .groupBy("supplier_country")
    .agg(
        F.count("*").alias("total_invoices"),
        F.sum("is_fraud").alias("fraud_cases")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_cases") / F.col("total_invoices") * 100,
            2
        )
    )
    .orderBy(F.desc("fraud_rate_percent"))
)

display(supplier_country_fraud)

supplier_country,total_invoices,fraud_cases,fraud_rate_percent
CD,784,379,48.34
AU,1651,775,46.94
IR,1328,621,46.76
NL,1429,509,35.62
LA,864,300,34.72
TR,2110,729,34.55
KG,1067,350,32.8
SE,761,249,32.72
BW,4392,1408,32.06
GE,1783,540,30.29


Databricks visualization. Run in Databricks to view.

### Blacklisted Suppliers and Fraud

This analysis investigates whether invoices associated with blacklisted suppliers have a higher fraud rate.

In [0]:
blacklist_fraud = (
    invoices_df
    .join(suppliers_df, "supplier_id")
    .join(labels_df, "invoice_id")
    .groupBy("blacklisted_flag")
    .agg(
        F.count("*").alias("total_invoices"),
        F.sum("is_fraud").alias("fraud_cases")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_cases") / F.col("total_invoices") * 100,
            2
        )
    )
)

display(blacklist_fraud)

blacklisted_flag,total_invoices,fraud_cases,fraud_rate_percent
0,284632,51336,18.04
1,15368,15074,98.09


### Fraud Rate by Department Region

This analysis examines whether fraud occurrence differs across department regions.

In [0]:
region_fraud = (
    invoices_df
    .join(departments_df, "department_id")
    .join(labels_df, "invoice_id")
    .groupBy("region")
    .agg(
        F.count("*").alias("total_invoices"),
        F.sum("is_fraud").alias("fraud_cases")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_cases") / F.col("total_invoices") * 100,
            2
        )
    )
    .orderBy(F.desc("fraud_rate_percent"))
)

display(region_fraud)

region,total_invoices,fraud_cases,fraud_rate_percent
Tennessee,18017,4072,22.6
Hawaii,5911,1334,22.57
Wyoming,12134,2733,22.52
Pennsylvania,11791,2655,22.52
Oregon,6079,1369,22.52
California,6066,1357,22.37
Louisiana,18126,4037,22.27
Nevada,18058,4021,22.27
Michigan,12060,2682,22.24
Maine,5959,1323,22.2


### Submission Hour vs Fraud

In [0]:
submission_fraud = (
    invoices_df
    .join(labels_df, "invoice_id")
    .groupBy("submission_hour")
    .agg(
        F.count("*").alias("total_invoices"),
        F.sum("is_fraud").alias("fraud_cases")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_cases") / F.col("total_invoices") * 100,
            2
        )
    )
    .orderBy("submission_hour")
)

display(submission_fraud)

submission_hour,total_invoices,fraud_cases,fraud_rate_percent
0,12457,2978,23.91
1,12468,2980,23.9
2,12634,2977,23.56
3,12537,3019,24.08
4,12460,2976,23.88
5,12608,3090,24.51
6,12662,2663,21.03
7,12264,2636,21.49
8,12628,2743,21.72
9,12569,2755,21.92


Databricks visualization. Run in Databricks to view.

### Average Supplier Risk Score by Fraud Status

In [0]:
supplier_risk_fraud = (
    invoices_df
    .join(suppliers_df, "supplier_id")
    .join(labels_df, "invoice_id")
    .groupBy("is_fraud")
    .agg(
        F.round(
            F.avg("supplier_risk_score"),
            2
        ).alias("average_supplier_risk_score")
    )
)

display(supplier_risk_fraud)

is_fraud,average_supplier_risk_score
0,0.28
1,0.3


### Image Tampering vs Fraud Rate

In [0]:
image_fraud = (
    images_metadata_df
    .join(labels_df, "invoice_id")
    .groupBy("image_tamper_flag")
    .agg(
        F.count("*").alias("total_images"),
        F.sum("is_fraud").alias("fraud_cases")
    )
    .withColumn(
        "fraud_rate_percent",
        F.round(
            F.col("fraud_cases") / F.col("total_images") * 100,
            2
        )
    )
)

display(image_fraud)

image_tamper_flag,total_images,fraud_cases,fraud_rate_percent
0,35908,7941,22.11
1,9092,2036,22.39


### Behavioural Flags vs Fraud Summary

In [0]:
flag_summary = (
    behaviour_label_df
    .agg(
        F.sum("duplicate_invoice_flag").alias("Duplicate Flags"),
        F.sum("split_invoice_flag").alias("Split Invoice Flags"),
        F.sum("late_night_submission_flag").alias("Late Night Submission Flags")
    )
)

display(flag_summary)

Duplicate Flags,Split Invoice Flags,Late Night Submission Flags
70,42679,75164


### Key Business Insights

The cross-dataset analysis highlights several relationships between supplier characteristics, departmental information, behavioural indicators, image metadata, and fraud labels.

The analyses demonstrate that fraud detection cannot rely on a single attribute. Instead, supplier risk, behavioural patterns, invoice characteristics, and image-based indicators together provide complementary information that can improve predictive performance.

These findings support the use of a multimodal machine learning approach that integrates structured invoice data, engineered behavioural features, supplier information, and image-derived features for fraud detection.

# Week 2 Conclusion

## Overview

The objective of Week 2 was to perform an exploratory data analysis (EDA) of the RiskCheck AI datasets to better understand the characteristics, quality, and relationships within the available data. Individual datasets were analysed independently before integrating multiple datasets to uncover meaningful business insights related to invoice fraud detection.

## Key Findings

The exploratory analysis produced several important observations:

- The invoice dataset contains **300,000 invoice records**, providing a comprehensive foundation for fraud analysis.
- Supplier information revealed variations in supplier risk scores, historical invoice behaviour, blacklist status, and geographical distribution.
- Departmental analysis highlighted differences in invoice volumes across regions and departments.
- Behavioural feature analysis demonstrated that engineered features such as invoice amount Z-scores, supplier activity, duplicate invoice indicators, split invoice indicators, and late-night submission flags provide valuable behavioural information for fraud detection.
- Fraud label analysis showed the distribution of fraudulent and non-fraudulent invoices and identified the different fraud categories available within the dataset.
- Image metadata analysis confirmed the availability of OCR-derived information and image tampering indicators, enabling the integration of visual document analysis into the fraud detection process.
- Cross-dataset analysis identified relationships between supplier characteristics, departmental information, behavioural indicators, invoice attributes, image metadata, and fraud labels, providing a more comprehensive understanding of potential fraud patterns.

## Business Insights

Several business insights emerged from the exploratory analysis:

- Behavioural indicators such as duplicate invoices, split invoices, and late-night submissions appear to be useful predictors of fraudulent activity.
- Supplier-related attributes, including supplier risk scores and blacklist status, contribute additional context for identifying high-risk transactions.
- Departmental and regional differences may influence invoice processing behaviour and should be considered during feature engineering.
- Image tampering indicators and OCR-derived features provide complementary information that can strengthen fraud detection when combined with structured invoice data.
- Fraud detection is influenced by multiple interacting factors rather than a single variable, supporting the use of a multimodal analytical approach.

## Outcome

The exploratory data analysis successfully established a comprehensive understanding of the RiskCheck AI dataset. The insights gained during this phase provide a strong foundation for feature engineering and predictive modelling.

The analyses performed during Week 2 identified the most informative variables, validated relationships between datasets, and highlighted behavioural and business patterns that are likely to improve fraud detection performance.

## Next Steps

The next phase of the project, **Week 3 – Feature Engineering and Data Preparation**, will focus on:

- Integrating multiple datasets into a unified analytical dataset.
- Creating new predictive features from invoice, supplier, behavioural, and image data.
- Encoding categorical variables.
- Scaling and transforming numerical features where appropriate.
- Preparing the final feature set for machine learning model development.

# Week 3: Feature Engineering and Data Preparation

## Overview

The objective of Week 3 is to transform the insights obtained during exploratory data analysis into a structured dataset suitable for machine learning.

This phase focuses on integrating the available datasets, creating predictive features, encoding categorical variables, transforming numerical variables where appropriate, and preparing a final feature set for fraud detection modelling.

## Objectives

- Integrate invoice, supplier, department, behavioural, image metadata, label, and split datasets.
- Validate dataset relationships before and after integration.
- Create predictive features based on invoice, supplier, behavioural, and image information.
- Handle missing image metadata appropriately.
- Encode categorical variables for machine learning.
- Prepare numerical features for modelling.
- Prevent target leakage during feature preparation.
- Create a final modelling dataset while preserving the predefined dataset splits.

## Build the Unified Analytical Dataset

The RiskCheck AI project contains information distributed across several datasets. Before feature engineering can be performed, these datasets must be combined into a unified invoice-level analytical dataset.

The invoice dataset will be used as the primary table because each record represents an invoice. Additional information will then be added from behavioural features, supplier information, department information, image metadata, fraud labels, and predefined dataset splits.

Left joins will be used for feature datasets where appropriate so that invoice records are not unintentionally removed, particularly because image metadata is available for only a subset of invoices.

In [0]:
datasets_for_integration = {
    "Invoices": invoices_df,
    "Behavioural Features": behavioural_features_df,
    "Suppliers": suppliers_df,
    "Departments": departments_df,
    "Image Metadata": images_metadata_df,
    "Labels": labels_df,
    "Splits": splits_df
}

for dataset_name, dataframe in datasets_for_integration.items():
    print(f"\n{dataset_name}")
    print("-" * 50)
    print(dataframe.columns)


Invoices
--------------------------------------------------
['invoice_id', 'supplier_id', 'department_id', 'invoice_date', 'invoice_amount', 'currency', 'payment_terms', 'invoice_type', 'submission_hour', 'image_path']

Behavioural Features
--------------------------------------------------
['invoice_id', 'supplier_invoice_count_30d', 'supplier_avg_amount_90d', 'invoice_amount_zscore', 'duplicate_invoice_flag', 'split_invoice_flag', 'late_night_submission_flag']

Suppliers
--------------------------------------------------
['supplier_id', 'supplier_country', 'supplier_age_days', 'supplier_risk_score', 'blacklisted_flag', 'avg_invoice_amount']

Departments
--------------------------------------------------
['department_id', 'region', 'annual_budget']

Image Metadata
--------------------------------------------------
['invoice_id', 'image_path', 'ocr_total_extracted', 'image_tamper_flag']

Labels
--------------------------------------------------
['invoice_id', 'is_fraud', 'fraud_type',

### Validate Uniqueness of the Expected Keys

In [0]:
key_checks = [
    ("Invoices", invoices_df, "invoice_id"),
    ("Behavioural Features", behavioural_features_df, "invoice_id"),
    ("Suppliers", suppliers_df, "supplier_id"),
    ("Departments", departments_df, "department_id"),
    ("Image Metadata", images_metadata_df, "invoice_id"),
    ("Labels", labels_df, "invoice_id"),
    ("Splits", splits_df, "invoice_id")
]

for dataset_name, dataframe, key_column in key_checks:

    total_rows = dataframe.count()
    unique_keys = dataframe.select(key_column).distinct().count()

    print(
        f"{dataset_name:<25} "
        f"Rows: {total_rows:<10,} "
        f"Unique {key_column}: {unique_keys:<10,} "
        f"Difference: {total_rows - unique_keys:,}"
    )

Invoices                  Rows: 300,000    Unique invoice_id: 300,000    Difference: 0
Behavioural Features      Rows: 300,000    Unique invoice_id: 300,000    Difference: 0
Suppliers                 Rows: 2,000      Unique supplier_id: 2,000      Difference: 0
Departments               Rows: 50         Unique department_id: 50         Difference: 0
Image Metadata            Rows: 45,000     Unique invoice_id: 45,000     Difference: 0
Labels                    Rows: 300,000    Unique invoice_id: 300,000    Difference: 0
Splits                    Rows: 300,000    Unique invoice_id: 300,000    Difference: 0


### Integrate the Project Datasets

The uniqueness validation confirmed that the expected join keys are unique within each dataset. Therefore, the datasets can now be integrated without introducing duplicate invoice records through one-to-many relationships.

The invoice dataset is used as the base dataset. Left joins are used to preserve all invoice records while enriching them with behavioural, supplier, department, image metadata, fraud label, and dataset split information.

Image metadata requires special consideration because it is available for only a subset of invoices. A left join ensures that invoices without image metadata remain available for subsequent modelling.

In [0]:
image_features_df = images_metadata_df.select(
    "invoice_id",
    "ocr_total_extracted",
    "image_tamper_flag"
)


unified_df = (
    invoices_df
    .join(behavioural_features_df, on="invoice_id", how="left")
    .join(suppliers_df, on="supplier_id", how="left")
    .join(departments_df, on="department_id", how="left")
    .join(image_features_df, on="invoice_id", how="left")
    .join(labels_df, on="invoice_id", how="left")
    .join(splits_df, on="invoice_id", how="left")
)

print("Unified analytical dataset created successfully.")
print(f"Total rows: {unified_df.count():,}")
print(f"Total columns: {len(unified_df.columns)}")

Unified analytical dataset created successfully.
Total rows: 300,000
Total columns: 30


### Verifying Integration

In [0]:
from pyspark.sql import functions as F


validation_summary = unified_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("invoice_id").alias("unique_invoices"),
    F.sum(F.when(F.col("supplier_risk_score").isNull(), 1).otherwise(0))
        .alias("missing_supplier_data"),
    F.sum(F.when(F.col("region").isNull(), 1).otherwise(0))
        .alias("missing_department_data"),
    F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0))
        .alias("missing_labels"),
    F.sum(F.when(F.col("split").isNull(), 1).otherwise(0))
        .alias("missing_split"),
    F.sum(F.when(F.col("ocr_total_extracted").isNull(), 1).otherwise(0))
        .alias("missing_image_metadata")
)

validation_summary.show(truncate=False)

+----------+---------------+---------------------+-----------------------+--------------+-------------+----------------------+
|total_rows|unique_invoices|missing_supplier_data|missing_department_data|missing_labels|missing_split|missing_image_metadata|
+----------+---------------+---------------------+-----------------------+--------------+-------------+----------------------+
|300000    |300000         |0                    |0                      |0             |0            |255000                |
+----------+---------------+---------------------+-----------------------+--------------+-------------+----------------------+



### Check train/test Split

In [0]:
# Verify that the predefined dataset split was preserved

unified_df.groupBy("split") \
    .count() \
    .orderBy("split") \
    .show()

+-----+------+
|split| count|
+-----+------+
| test| 60000|
|train|210000|
|  val| 30000|
+-----+------+



## Feature Engineering

The unified analytical dataset has been successfully constructed and validated. All 300,000 invoice records were preserved, the core datasets were integrated without missing join records, and the predefined train, validation, and test splits remained intact.

The next stage is feature engineering. The objective is to transform existing raw variables into additional features that may help machine learning models identify patterns associated with fraudulent invoices.

Feature engineering will focus on:

- Invoice amount and supplier behaviour
- Invoice timing characteristics
- Supplier risk information
- Department-level financial context
- Existing behavioural fraud indicators
- Image and OCR information
- Missing image metadata

Care will be taken to ensure that fraud labels and explanatory fields are not used to construct predictor features, preventing target leakage.

### Create Invoice/Supplier Features

In [0]:
from pyspark.sql import functions as F

feature_df = (
    unified_df

    .withColumn(
        "amount_to_supplier_90d_avg_ratio",
        F.when(
            F.col("supplier_avg_amount_90d") > 0,
            F.col("invoice_amount") / F.col("supplier_avg_amount_90d")
        ).otherwise(F.lit(None))
    )

    
    .withColumn(
        "amount_to_supplier_avg_ratio",
        F.when(
            F.col("avg_invoice_amount") > 0,
            F.col("invoice_amount") / F.col("avg_invoice_amount")
        ).otherwise(F.lit(None))
    )

    
    .withColumn(
        "amount_diff_supplier_90d_avg",
        F.abs(
            F.col("invoice_amount") -
            F.col("supplier_avg_amount_90d")
        )
    )

    .withColumn(
        "supplier_age_years",
        F.col("supplier_age_days") / F.lit(365.25)
    )
)

print("Feature engineering started successfully.")
print(f"Rows: {feature_df.count():,}")
print(f"Columns: {len(feature_df.columns)}")

Feature engineering started successfully.
Rows: 300,000
Columns: 34


### Inspecting New Features

In [0]:
feature_df.select(
    "invoice_id",
    "invoice_amount",
    "supplier_avg_amount_90d",
    "avg_invoice_amount",
    "amount_to_supplier_90d_avg_ratio",
    "amount_to_supplier_avg_ratio",
    "amount_diff_supplier_90d_avg",
    "supplier_age_years"
).show(10, truncate=False)

+-----------+--------------+-----------------------+------------------+--------------------------------+----------------------------+----------------------------+------------------+
|invoice_id |invoice_amount|supplier_avg_amount_90d|avg_invoice_amount|amount_to_supplier_90d_avg_ratio|amount_to_supplier_avg_ratio|amount_diff_supplier_90d_avg|supplier_age_years|
+-----------+--------------+-----------------------+------------------+--------------------------------+----------------------------+----------------------------+------------------+
|INV_0000000|3565.57       |5417.982105263158      |4552.54           |0.6580992573852025              |0.7832045407618604          |1852.4121052631576          |4.0191649555099245|
|INV_0000001|9871.78       |6745.242333333334      |5568.13           |1.4635174708573604              |1.7729076009360414          |3126.537666666667           |1.325119780971937 |
|INV_0000002|8530.64       |4950.948095238095      |4542.37           |1.723031596353214  

###  Timing & Department Context Features

Fraudulent invoice activity may also be associated with unusual submission timing or invoice amounts that are large relative to the financial scale of the responsible department.

Additional features are therefore created to capture:

- Whether an invoice was submitted during the weekend.
- Whether the submission occurred outside typical business hours.
- The invoice amount relative to the department's annual budget.
- A logarithmic transformation of invoice amount to reduce the effect of highly skewed monetary values.

These engineered variables complement the behavioural and supplier-level features already available in the dataset.

### Creating Timing & Department Features

In [0]:
feature_df = (
    feature_df

    
    .withColumn(
        "invoice_day_of_week",
        F.dayofweek(F.col("invoice_date"))
    )

    
    .withColumn(
        "weekend_invoice_flag",
        F.when(
            F.dayofweek(F.col("invoice_date")).isin([1, 7]),
            1
        ).otherwise(0)
    )

    
    .withColumn(
        "outside_business_hours_flag",
        F.when(
            (F.col("submission_hour") < 8) |
            (F.col("submission_hour") >= 18),
            1
        ).otherwise(0)
    )

    
    .withColumn(
        "invoice_to_department_budget_ratio",
        F.when(
            F.col("annual_budget") > 0,
            F.col("invoice_amount") / F.col("annual_budget")
        ).otherwise(F.lit(None))
    )


    .withColumn(
        "log_invoice_amount",
        F.log1p(F.col("invoice_amount"))
    )
)

print("Timing and department features created.")
print(f"Rows: {feature_df.count():,}")
print(f"Columns: {len(feature_df.columns)}")

Timing and department features created.
Rows: 300,000
Columns: 39


### Inspecting New Features

In [0]:
feature_df.select(
    "invoice_id",
    "invoice_date",
    "submission_hour",
    "invoice_day_of_week",
    "weekend_invoice_flag",
    "late_night_submission_flag",
    "outside_business_hours_flag",
    "invoice_amount",
    "annual_budget",
    "invoice_to_department_budget_ratio",
    "log_invoice_amount"
).show(10, truncate=False)

+-----------+-------------------+---------------+-------------------+--------------------+--------------------------+---------------------------+--------------+--------------------+----------------------------------+------------------+
|invoice_id |invoice_date       |submission_hour|invoice_day_of_week|weekend_invoice_flag|late_night_submission_flag|outside_business_hours_flag|invoice_amount|annual_budget       |invoice_to_department_budget_ratio|log_invoice_amount|
+-----------+-------------------+---------------+-------------------+--------------------+--------------------------+---------------------------+--------------+--------------------+----------------------------------+------------------+
|INV_0000000|2023-04-18 00:00:00|18             |3                  |0                   |0                         |1                          |3565.57       |2.1713503886269618E7|1.6420979399159357E-4             |8.179359628610863 |
|INV_0000001|2023-10-08 00:00:00|6              |1      

### Image & OCR Features

Image metadata is available for only a subset of invoices. Missing image metadata therefore represents the absence of image-derived information rather than necessarily indicating a data-quality error.

To preserve this distinction, an explicit image metadata availability indicator is created. The existing image tamper flag is retained as an image-based fraud signal.

An additional OCR comparison feature is created by comparing the invoice amount with the total amount extracted through OCR. This may help identify invoices where the value detected from the invoice image differs substantially from the structured invoice amount.

For invoices without image metadata, image-derived comparison features are left missing at this stage rather than automatically assigning a value of zero. Missing-value treatment will be handled separately during model preparation.

### Creating Image/OCR Features

In [0]:
feature_df = (
    feature_df

    .withColumn(
        "image_metadata_available_flag",
        F.when(
            F.col("ocr_total_extracted").isNotNull() |
            F.col("image_tamper_flag").isNotNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "invoice_ocr_amount_diff",
        F.when(
            F.col("ocr_total_extracted").isNotNull(),
            F.abs(
                F.col("invoice_amount") -
                F.col("ocr_total_extracted")
            )
        ).otherwise(F.lit(None))
    )

    .withColumn(
        "invoice_ocr_relative_diff",
        F.when(
            F.col("ocr_total_extracted").isNotNull() &
            (F.col("invoice_amount") > 0),
            F.abs(
                F.col("invoice_amount") -
                F.col("ocr_total_extracted")
            ) / F.col("invoice_amount")
        ).otherwise(F.lit(None))
    )
)

print("Image and OCR features created.")
print(f"Rows: {feature_df.count():,}")
print(f"Columns: {len(feature_df.columns)}")

Image and OCR features created.
Rows: 300,000
Columns: 42


### Validate Image Availability

In [0]:
feature_df.groupBy("image_metadata_available_flag") \
    .count() \
    .orderBy("image_metadata_available_flag") \
    .show()

+-----------------------------+------+
|image_metadata_available_flag| count|
+-----------------------------+------+
|                            0|255000|
|                            1| 45000|
+-----------------------------+------+



### Inspecting Invoices that have Image Metadata

In [0]:
feature_df.filter(
    F.col("image_metadata_available_flag") == 1
).select(
    "invoice_id",
    "invoice_amount",
    "ocr_total_extracted",
    "invoice_ocr_amount_diff",
    "invoice_ocr_relative_diff",
    "image_tamper_flag"
).show(10, truncate=False)

+-----------+--------------+-------------------+-----------------------+-------------------------+-----------------+
|invoice_id |invoice_amount|ocr_total_extracted|invoice_ocr_amount_diff|invoice_ocr_relative_diff|image_tamper_flag|
+-----------+--------------+-------------------+-----------------------+-------------------------+-----------------+
|INV_0000000|3565.57       |6363.28            |2797.7099999999996     |0.7846459331888027       |0                |
|INV_0000006|4159.75       |6861.46            |2701.71                |0.6494885509946511       |0                |
|INV_0000012|9564.47       |41175.9            |31611.43               |3.305089565861987        |0                |
|INV_0000016|2883.69       |5513.81            |2630.1200000000003     |0.9120675245952236       |0                |
|INV_0000022|1417.85       |2431.77            |1013.9200000000001     |0.7151109073597349       |0                |
|INV_0000024|2783.64       |5284.57            |2500.93         

## Predictor Selection and Target Leakage Prevention

Before categorical encoding and model preparation, the variables in the analytical dataset must be separated according to their modelling role.

The primary prediction target is `is_fraud`.

Several fields must not be used as predictors because they either directly reveal the fraud outcome or contain information generated from the fraud labelling process. These include `fraud_type`, `fraud_tags`, and `explanations`.

Identifier and operational fields such as `invoice_id` and `image_path` are also excluded from the predictive feature set because they identify records rather than describe fraud-related behaviour.

The `split` variable is retained only for separating the training, validation, and test datasets and is not used as a model predictor.

This separation prevents target leakage and ensures that model performance reflects patterns available from legitimate invoice, supplier, behavioural, department, and image-derived information.

### Defining Modelling Roles

In [0]:
target_column = "is_fraud"

leakage_columns = [
    "fraud_type",
    "fraud_tags",
    "explanations"
]

identifier_columns = [
    "invoice_id",
    "image_path"
]

split_column = "split"

excluded_from_predictors = (
    [target_column] +
    leakage_columns +
    identifier_columns +
    [split_column]
)

predictor_columns = [
    column
    for column in feature_df.columns
    if column not in excluded_from_predictors
]

print(f"Total columns in feature_df: {len(feature_df.columns)}")
print(f"Predictor columns: {len(predictor_columns)}")
print(f"Target column: {target_column}")
print(f"Excluded columns: {excluded_from_predictors}")

Total columns in feature_df: 42
Predictor columns: 35
Target column: is_fraud
Excluded columns: ['is_fraud', 'fraud_type', 'fraud_tags', 'explanations', 'invoice_id', 'image_path', 'split']


### Identifying Categorical Predictors

In [0]:
categorical_columns = [
    "currency",
    "payment_terms",
    "invoice_type",
    "supplier_country",
    "region"
]

print("Categorical predictor columns:")

for column in categorical_columns:
    print(f"- {column}")

Categorical predictor columns:
- currency
- payment_terms
- invoice_type
- supplier_country
- region


### Inspecting Cardinality

In [0]:
for column in categorical_columns:
    category_count = feature_df.select(column).distinct().count()

    print(f"\n{column}: {category_count} unique values")

    feature_df.groupBy(column) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(20, truncate=False)


currency: 1 unique values
+--------+------+
|currency|count |
+--------+------+
|ZAR     |300000|
+--------+------+


payment_terms: 3 unique values
+-------------+------+
|payment_terms|count |
+-------------+------+
|NET30        |180337|
|NET60        |89883 |
|NET90        |29780 |
+-------------+------+


invoice_type: 2 unique values
+------------+------+
|invoice_type|count |
+------------+------+
|GOODS       |164863|
|SERVICES    |135137|
+------------+------+


supplier_country: 126 unique values
+----------------+-----+
|supplier_country|count|
+----------------+-----+
|KZ              |6207 |
|IN              |6088 |
|DZ              |5421 |
|DE              |5365 |
|CU              |5309 |
|CG              |5301 |
|NO              |5150 |
|DJ              |5114 |
|IQ              |5014 |
|MU              |4985 |
|TG              |4934 |
|WS              |4857 |
|SO              |4657 |
|DK              |4575 |
|IT              |4527 |
|BD              |4496 |
|BW         

## Categorical Variable Encoding

Categorical variables must be converted into suitable numerical representations before they can be used by machine learning algorithms.

The categorical variables show different levels of cardinality:

- `currency` contains only one value (ZAR) and therefore provides no discriminatory information for fraud prediction. It is excluded from the modelling predictors.
- `payment_terms` contains 3 categories and is converted into binary indicator variables.
- `invoice_type` contains 2 categories and is converted into binary indicator variables.
- `region` contains 24 categories and is excluded from the final numerical feature set at this stage.
- `supplier_country` contains 126 categories and is excluded from the final numerical feature set at this stage to avoid unnecessary high-dimensional encoding.

The low-cardinality variables are encoded directly using binary indicator columns. Higher-cardinality categorical variables can be reconsidered using an appropriate encoding strategy during the modelling stage.

### Removing Constant Currency Predictor & Defining Encoding Groups

In [0]:
predictor_columns = [
    column for column in predictor_columns
    if column != "currency"
]

low_cardinality_categoricals = [
    "payment_terms",
    "invoice_type",
    "region"
]

high_cardinality_categoricals = [
    "supplier_country"
]

print(f"Predictor columns after removing currency: {len(predictor_columns)}")
print("One-hot encoded variables:", low_cardinality_categoricals)
print("Indexed variable:", high_cardinality_categoricals)

Predictor columns after removing currency: 34
One-hot encoded variables: ['payment_terms', 'invoice_type', 'region']
Indexed variable: ['supplier_country']


### Fit Categorical Transformations on Training Data

In [0]:
encoded_df = (
    feature_df

    
    .withColumn("payment_terms_NET30", F.when(F.col("payment_terms") == "NET30", 1).otherwise(0))
    .withColumn("payment_terms_NET60", F.when(F.col("payment_terms") == "NET60", 1).otherwise(0))
    .withColumn("payment_terms_NET90", F.when(F.col("payment_terms") == "NET90", 1).otherwise(0))

    
    .withColumn("invoice_type_GOODS", F.when(F.col("invoice_type") == "GOODS", 1).otherwise(0))
    .withColumn("invoice_type_SERVICES", F.when(F.col("invoice_type") == "SERVICES", 1).otherwise(0))
)

print("Low-cardinality categorical encoding completed.")
print(f"Rows: {encoded_df.count():,}")
print(f"Columns: {len(encoded_df.columns)}")

Low-cardinality categorical encoding completed.
Rows: 300,000
Columns: 47


### Inspecting Encoded Categorical Ceatures

In [0]:
encoded_df.select(
    "payment_terms",
    "payment_terms_NET30",
    "payment_terms_NET60",
    "payment_terms_NET90",
    "invoice_type",
    "invoice_type_GOODS",
    "invoice_type_SERVICES"
).show(10, truncate=False)

+-------------+-------------------+-------------------+-------------------+------------+------------------+---------------------+
|payment_terms|payment_terms_NET30|payment_terms_NET60|payment_terms_NET90|invoice_type|invoice_type_GOODS|invoice_type_SERVICES|
+-------------+-------------------+-------------------+-------------------+------------+------------------+---------------------+
|NET60        |0                  |1                  |0                  |GOODS       |1                 |0                    |
|NET30        |1                  |0                  |0                  |SERVICES    |0                 |1                    |
|NET60        |0                  |1                  |0                  |SERVICES    |0                 |1                    |
|NET30        |1                  |0                  |0                  |GOODS       |1                 |0                    |
|NET30        |1                  |0                  |0                  |GOODS       |1 

## Missing-Value Treatment & Numerical Preparation

Before the final modelling datasets are assembled, remaining numerical missing values must be handled consistently.

Image-derived variables require special treatment because image metadata is available for only a subset of invoices. The `image_metadata_available_flag` preserves whether image information was originally present.

For modelling purposes, missing image-derived numerical values are replaced with zero while retaining the availability indicator. This allows the final feature dataset to remain numerically complete while distinguishing invoices with unavailable image information from invoices with observed image metadata.

The remaining numerical predictors are then checked for missing values before the final train, validation, and test datasets are created.

### Handling Image-related Missing Values

In [0]:
image_numeric_columns = [
    "ocr_total_extracted",
    "image_tamper_flag",
    "invoice_ocr_amount_diff",
    "invoice_ocr_relative_diff"
]

prepared_df = encoded_df.fillna(
    0,
    subset=image_numeric_columns
)

print("Image-related missing values handled.")
print(f"Rows: {prepared_df.count():,}")
print(f"Columns: {len(prepared_df.columns)}")

Image-related missing values handled.
Rows: 300,000
Columns: 47


### Validate Remaining Numerical Nulls

In [0]:
columns_to_check = [
    "invoice_amount",
    "supplier_invoice_count_30d",
    "supplier_avg_amount_90d",
    "invoice_amount_zscore",
    "supplier_age_days",
    "supplier_risk_score",
    "avg_invoice_amount",
    "annual_budget",
    "amount_to_supplier_90d_avg_ratio",
    "amount_to_supplier_avg_ratio",
    "amount_diff_supplier_90d_avg",
    "supplier_age_years",
    "invoice_to_department_budget_ratio",
    "log_invoice_amount",
    "ocr_total_extracted",
    "image_tamper_flag",
    "invoice_ocr_amount_diff",
    "invoice_ocr_relative_diff"
]

null_summary = prepared_df.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in columns_to_check
])

null_summary.show(truncate=False)

+--------------+--------------------------+-----------------------+---------------------+-----------------+-------------------+------------------+-------------+--------------------------------+----------------------------+----------------------------+------------------+----------------------------------+------------------+-------------------+-----------------+-----------------------+-------------------------+
|invoice_amount|supplier_invoice_count_30d|supplier_avg_amount_90d|invoice_amount_zscore|supplier_age_days|supplier_risk_score|avg_invoice_amount|annual_budget|amount_to_supplier_90d_avg_ratio|amount_to_supplier_avg_ratio|amount_diff_supplier_90d_avg|supplier_age_years|invoice_to_department_budget_ratio|log_invoice_amount|ocr_total_extracted|image_tamper_flag|invoice_ocr_amount_diff|invoice_ocr_relative_diff|
+--------------+--------------------------+-----------------------+---------------------+-----------------+-------------------+------------------+-------------+--------------

## Create Final Modelling Datasets

The prepared analytical dataset is now divided into the predefined training, validation, and test partitions.

Only modelling-ready predictors are retained. Target-leakage variables, identifiers, raw categorical variables, and operational fields are excluded from the predictor set.

The original dataset split is preserved:

- Training set: 210,000 invoices
- Validation set: 30,000 invoices
- Test set: 60,000 invoices

The fraud target (`is_fraud`) is retained in each partition for supervised machine learning.

### Defining Final Modeling Features

In [0]:
final_excluded_columns = [
    
    
    "invoice_id",
    "supplier_id",
    "department_id",
    "image_path",
    "invoice_date",
    "split",

    
    "fraud_type",
    "fraud_tags",
    "explanations",

    
    "currency",
    "payment_terms",
    "invoice_type",
    "supplier_country",
    "region"
]


target_column = "is_fraud"

final_feature_columns = [
    column
    for column in prepared_df.columns
    if column not in final_excluded_columns
    and column != target_column
]

print(f"Final number of modelling features: {len(final_feature_columns)}")
print("\nFinal feature columns:")

for column in final_feature_columns:
    print(f"- {column}")

Final number of modelling features: 32

Final feature columns:
- invoice_amount
- submission_hour
- supplier_invoice_count_30d
- supplier_avg_amount_90d
- invoice_amount_zscore
- duplicate_invoice_flag
- split_invoice_flag
- late_night_submission_flag
- supplier_age_days
- supplier_risk_score
- blacklisted_flag
- avg_invoice_amount
- annual_budget
- ocr_total_extracted
- image_tamper_flag
- amount_to_supplier_90d_avg_ratio
- amount_to_supplier_avg_ratio
- amount_diff_supplier_90d_avg
- supplier_age_years
- invoice_day_of_week
- weekend_invoice_flag
- outside_business_hours_flag
- invoice_to_department_budget_ratio
- log_invoice_amount
- image_metadata_available_flag
- invoice_ocr_amount_diff
- invoice_ocr_relative_diff
- payment_terms_NET30
- payment_terms_NET60
- payment_terms_NET90
- invoice_type_GOODS
- invoice_type_SERVICES


In [0]:
training_base_table = "workspace.default.riskcheck_training_base"

training_base_df = prepared_df.select(
    "invoice_id",
    "split",
    *final_feature_columns,
    "is_fraud"
)

(
    training_base_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(training_base_table)
)

print("RiskCheck training base saved successfully.")
print(f"Rows: {training_base_df.count():,}")
print(f"Columns: {len(training_base_df.columns)}")

RiskCheck training base saved successfully.
Rows: 300,000
Columns: 35


In [0]:
training_base_table = "workspace.default.riskcheck_training_base"

training_base_df = spark.table(training_base_table)

print("Training base loaded successfully.")
print(f"Rows: {training_base_df.count():,}")
print(f"Columns: {len(training_base_df.columns)}")

Training base loaded successfully.
Rows: 300,000
Columns: 35


In [0]:
feedback_table = "workspace.default.riskcheck_investigator_feedback"

verified_feedback_df = spark.sql(f"""
    SELECT
        invoice_id,
        verified_label,
        investigator_note,
        verified_at
    FROM {feedback_table}
    WHERE consumed_by_training = false
""")

print(f"New verified labels available: {verified_feedback_df.count():,}")

New verified labels available: 0


In [0]:
from pyspark.sql import functions as F

verified_training_rows = (
    verified_feedback_df.alias("f")
    .join(
        training_base_df.alias("b"),
        F.col("f.invoice_id") == F.col("b.invoice_id"),
        "inner"
    )
    .select(
        F.col("b.invoice_id"),
        F.col("b.split"),
        *[F.col(f"b.{c}") for c in training_base_df.columns
          if c not in ["invoice_id", "split", "is_fraud"]],
        F.col("f.verified_label").alias("is_fraud")
    )
)

print(f"Verified feedback available: {verified_feedback_df.count():,}")
print(f"Matched training rows: {verified_training_rows.count():,}")

display(
    verified_training_rows.select(
        "invoice_id",
        "split",
        "is_fraud"
    )
)

Verified feedback available: 0
Matched training rows: 0


invoice_id,split,is_fraud


In [0]:
%sql
DESCRIBE workspace.default.riskcheck_scored_invoices;

col_name,data_type,comment
invoice_id,string,null
riskcheck_score,double,null
risk_category,string,null
priority_rank,int,null
priority_level,string,null
recommended_action,string,null
fraud_prediction,int,null
fraud_probability,double,null
investigation_reasons,string,null
scored_at,timestamp,null


In [0]:
%sql
SELECT
    invoice_id,
    invoice_amount,
    invoice_amount_zscore,
    supplier_risk_score,
    fraud_probability,
    riskcheck_score,
    scored_at
FROM workspace.default.riskcheck_scored_invoices
WHERE invoice_id = 'INV-DEMO-005'
ORDER BY scored_at DESC;

### Create the Three Datasets

In [0]:
train_df = (
    prepared_df
    .filter(F.col("split") == "train")
    .select(*(final_feature_columns + [target_column]))
)

val_df = (
    prepared_df
    .filter(F.col("split") == "val")
    .select(*(final_feature_columns + [target_column]))
)

test_df = (
    prepared_df
    .filter(F.col("split") == "test")
    .select(*(final_feature_columns + [target_column]))
)

print("Final modelling datasets created.")
print(f"Train rows:      {train_df.count():,}")
print(f"Validation rows: {val_df.count():,}")
print(f"Test rows:       {test_df.count():,}")

## Final Validation

Before completing Week 3, the final modelling datasets are validated to confirm that:

- The predefined train, validation, and test partitions were preserved.
- No invoice records were lost during feature preparation.
- The fraud target is available in every modelling dataset.
- The engineered numerical features contain no missing values.
- Target-leakage variables are excluded from the modelling feature set.
- Raw categorical variables are excluded after the selected categorical transformations.
- The final datasets are ready for the machine learning stage.

In [0]:
train_count = train_df.count()
val_count = val_df.count()
test_count = test_df.count()

total_model_rows = train_count + val_count + test_count

train_missing_target = train_df.filter(F.col("is_fraud").isNull()).count()
val_missing_target = val_df.filter(F.col("is_fraud").isNull()).count()
test_missing_target = test_df.filter(F.col("is_fraud").isNull()).count()

print("WEEK 3 FINAL VALIDATION")
print("-" * 50)

print(f"Training rows:        {train_count:,}")
print(f"Validation rows:      {val_count:,}")
print(f"Test rows:            {test_count:,}")
print(f"Total modelling rows: {total_model_rows:,}")

print("\nMissing target values")
print(f"Train:      {train_missing_target:,}")
print(f"Validation: {val_missing_target:,}")
print(f"Test:       {test_missing_target:,}")

print(f"\nNumber of modelling features: {len(final_feature_columns)}")

print(
    "\nAll records preserved:",
    total_model_rows == 300000
)

print(
    "Target complete:",
    (
        train_missing_target == 0
        and val_missing_target == 0
        and test_missing_target == 0
    )
)

## Week 3 Conclusion

Week 3 focused on transforming the cleaned RiskCheck AI datasets into an integrated and modelling-ready analytical dataset.

The invoice dataset was successfully combined with behavioural, supplier, department, image metadata, fraud label, and predefined split information. Join-key validation ensured that dataset integration did not introduce duplicate invoice records, and all 300,000 invoices were preserved.

Feature engineering introduced additional fraud-relevant signals covering supplier-relative invoice amounts, supplier age, invoice timing, weekend and outside-business-hours activity, department budget context, logarithmic invoice amounts, image metadata availability, and discrepancies between structured invoice amounts and OCR-extracted values.

Special attention was given to image metadata because it is available for only 45,000 of the 300,000 invoices. An explicit image metadata availability indicator was retained while missing image-derived numerical values were prepared for modelling.

Categorical preparation was performed for low-cardinality variables such as payment terms and invoice type. Constant currency information was excluded because it provided no predictive variation. Higher-cardinality categorical variables were retained outside the final numerical feature set for potential alternative treatment during the modelling stage.

Target leakage was prevented by excluding fraud type, fraud tags, explanations, record identifiers, and operational split information from the predictor set.

The final modelling data preserves the predefined project partitions:

- 210,000 training records
- 30,000 validation records
- 60,000 test records

The resulting feature set contains 32 modelling features together with the fraud target.

Week 3 therefore establishes a validated feature-preparation pipeline and provides the datasets required for the next stage of RiskCheck AI: model development, evaluation, and fraud classification.

# Week 4: Model Development & Baseline Fraud Detection

## Overview

Week 4 focuses on developing and evaluating machine learning models for invoice fraud detection using the modelling-ready datasets prepared during Week 3.

The objective is to establish a reliable baseline, train fraud-classification models, evaluate their performance using appropriate metrics, and select a suitable model for final testing.

Because fraud detection is typically an imbalanced classification problem, model performance will not be assessed using accuracy alone. Greater emphasis will be placed on metrics that describe the model's ability to identify fraudulent invoices while controlling false alerts.

## Objectives

- Examine fraud prevalence within the predefined data partitions.
- Establish a baseline fraud-classification benchmark.
- Prepare the final numerical model inputs.
- Train an initial classification model.
- Evaluate predictions using fraud-relevant metrics.
- Investigate class imbalance and model improvement strategies.
- Compare candidate models using the validation dataset.
- Select the preferred model before accessing the test dataset.
- Perform final evaluation using the held-out test dataset.

### Confirming Week 3 Datasets

In [0]:
print("Training rows:", f"{train_df.count():,}")
print("Validation rows:", f"{val_df.count():,}")
print("Test rows:", f"{test_df.count():,}")

print("\nTraining columns:", len(train_df.columns))
print("Validation columns:", len(val_df.columns))
print("Test columns:", len(test_df.columns))

## Examine the Fraud Target Distribution

Before developing classification models, the distribution of the target variable (`is_fraud`) must be examined across the training, validation, and test datasets.

Fraud detection datasets are often imbalanced, with legitimate invoices substantially outnumbering fraudulent invoices. In such cases, accuracy alone can provide a misleading assessment of model performance.

The class distribution is therefore evaluated before model training to determine the extent of imbalance and guide the selection of appropriate evaluation metrics and modelling strategies.

### Fraud distribution by Split

In [0]:
from pyspark.sql import functions as F

def show_target_distribution(df, dataset_name):
    total = df.count()

    print(f"\n{dataset_name}")
    print("-" * 50)

    (
        df.groupBy("is_fraud")
        .count()
        .withColumn(
            "percentage",
            F.round((F.col("count") / F.lit(total)) * 100, 4)
        )
        .orderBy("is_fraud")
        .show()
    )

show_target_distribution(train_df, "TRAINING SET")
show_target_distribution(val_df, "VALIDATION SET")
show_target_distribution(test_df, "TEST SET")

### Training-set Fraud Ratio

In [0]:
train_class_counts = {
    row["is_fraud"]: row["count"]
    for row in train_df.groupBy("is_fraud").count().collect()
}

legitimate_count = train_class_counts.get(0, 0)
fraud_count = train_class_counts.get(1, 0)

imbalance_ratio = (
    legitimate_count / fraud_count
    if fraud_count > 0
    else None
)

print(f"Legitimate training invoices: {legitimate_count:,}")
print(f"Fraudulent training invoices: {fraud_count:,}")

if imbalance_ratio is not None:
    print(
        f"Class imbalance ratio: "
        f"{imbalance_ratio:.2f} legitimate invoices "
        f"for every fraudulent invoice"
    )

## Establish a Baseline Classifier

A simple baseline is established before training machine learning models.

Because legitimate invoices represent the majority class, the baseline classifier predicts every invoice as legitimate (`is_fraud = 0`).

This baseline provides a minimum benchmark against which trained fraud-detection models can be compared. Although such a classifier may achieve relatively high overall accuracy, it will fail to identify fraudulent invoices.

For this reason, model evaluation will consider multiple classification metrics, including:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix

The validation dataset will be used for model comparison and selection. The test dataset will remain untouched until the final model has been selected.

### Creating Baseline Validation Predictions

In [0]:
baseline_predictions = val_df.withColumn(
    "prediction",
    F.lit(0.0)
)

baseline_predictions.groupBy(
    "is_fraud",
    "prediction"
).count().orderBy(
    "is_fraud",
    "prediction"
).show()

### Calculate Baseline Metrics

In [0]:
TP = baseline_predictions.filter(
    (F.col("is_fraud") == 1) &
    (F.col("prediction") == 1)
).count()

TN = baseline_predictions.filter(
    (F.col("is_fraud") == 0) &
    (F.col("prediction") == 0)
).count()

FP = baseline_predictions.filter(
    (F.col("is_fraud") == 0) &
    (F.col("prediction") == 1)
).count()

FN = baseline_predictions.filter(
    (F.col("is_fraud") == 1) &
    (F.col("prediction") == 0)
).count()

accuracy = (TP + TN) / (TP + TN + FP + FN)

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0 else 0
)

recall = (
    TP / (TP + FN)
    if (TP + FN) > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("BASELINE VALIDATION PERFORMANCE")
print("-" * 45)

print(f"True Positives:  {TP:,}")
print(f"True Negatives:  {TN:,}")
print(f"False Positives: {FP:,}")
print(f"False Negatives: {FN:,}")

print("\nMetrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

## Prepare Features for Machine Learning

The Week 3 modelling dataset contains 32 numerical predictor variables and the binary fraud target.

Spark ML classification algorithms require predictor variables to be represented as a single feature vector. The numerical predictor columns are therefore assembled into a `features` vector while `is_fraud` is retained as the classification label.

No information from the test dataset is used for model selection. The training dataset is used to fit models, while the validation dataset is used to evaluate and compare candidate models.

### Verify Feature Types

In [0]:
for column_name, data_type in train_df.dtypes:
    if column_name != "is_fraud":
        print(f"{column_name:<40} {data_type}")

### Assembling Features

In [0]:
from pyspark.ml.feature import VectorAssembler

model_feature_columns = [
    column
    for column in train_df.columns
    if column != "is_fraud"
]

assembler = VectorAssembler(
    inputCols=model_feature_columns,
    outputCol="features",
    handleInvalid="error"
)

train_ml_df = assembler.transform(train_df).select(
    F.col("is_fraud").cast("double").alias("label"),
    "features"
)

val_ml_df = assembler.transform(val_df).select(
    F.col("is_fraud").cast("double").alias("label"),
    "features"
)

print("ML feature datasets created successfully.")
print(f"Number of predictors: {len(model_feature_columns)}")
print(f"Training rows: {train_ml_df.count():,}")
print(f"Validation rows: {val_ml_df.count():,}")

### Inspecting Assembled Vector

In [0]:
train_ml_df.show(5, truncate=False)

## Train the Initial Fraud Classification Model

Logistic Regression is used as the first machine learning classifier for RiskCheck AI.

It provides a useful initial benchmark because it is computationally efficient, interpretable, and suitable for binary classification problems.

The model is trained using the 32 numerical predictors prepared during Week 3. The training dataset is used exclusively for model fitting, while performance is evaluated on the validation dataset.

This initial model is trained without class weighting. Its performance will provide a benchmark for determining whether additional treatment of class imbalance is beneficial.

### Training Logistic Regression

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_ml_df)

print("Logistic Regression training completed successfully.")

### Generate Validation Predictions

In [0]:
lr_val_predictions = lr_model.transform(val_ml_df)

print(
    "Validation predictions:",
    f"{lr_val_predictions.count():,}"
)

lr_val_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

### Evaluate ROC-AUC

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

lr_roc_auc = roc_evaluator.evaluate(lr_val_predictions)

print("LOGISTIC REGRESSION — VALIDATION")
print("-" * 45)
print(f"ROC-AUC: {lr_roc_auc:.4f}")

### Confusion Matrix & Fraud Metrics

In [0]:
TP = lr_val_predictions.filter(
    (F.col("label") == 1) &
    (F.col("prediction") == 1)
).count()

TN = lr_val_predictions.filter(
    (F.col("label") == 0) &
    (F.col("prediction") == 0)
).count()

FP = lr_val_predictions.filter(
    (F.col("label") == 0) &
    (F.col("prediction") == 1)
).count()

FN = lr_val_predictions.filter(
    (F.col("label") == 1) &
    (F.col("prediction") == 0)
).count()

accuracy = (TP + TN) / (TP + TN + FP + FN)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0

recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("LOGISTIC REGRESSION — VALIDATION PERFORMANCE")
print("-" * 50)

print("Confusion Matrix")
print(f"True Positives:  {TP:,}")
print(f"True Negatives:  {TN:,}")
print(f"False Positives: {FP:,}")
print(f"False Negatives: {FN:,}")

print("\nMetrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {lr_roc_auc:.4f}")

## Optimize the Fraud Classification Threshold

The initial Logistic Regression model demonstrates strong validation performance, with high precision, F1-score, and ROC-AUC.

However, the default classification threshold of 0.50 still results in some fraudulent invoices being classified as legitimate.

In fraud detection, false negatives can be particularly important because they represent fraudulent invoices that pass through the detection system. The classification threshold is therefore evaluated at several values below 0.50.

Lowering the threshold is expected to increase fraud recall, although this may also increase the number of false-positive fraud alerts.

The preferred threshold will therefore be selected by considering the trade-off between precision, recall, and F1-score using the validation dataset only.

### Extract Fraud Probability

In [0]:
from pyspark.ml.functions import vector_to_array

lr_val_scored = lr_val_predictions.withColumn(
    "fraud_probability",
    vector_to_array(F.col("probability"))[1]
)

lr_val_scored.select(
    "label",
    "fraud_probability"
).show(10, truncate=False)

### Comparing Thresholds

In [0]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

threshold_results = []

for threshold in thresholds:

    scored = lr_val_scored.withColumn(
        "threshold_prediction",
        F.when(
            F.col("fraud_probability") >= threshold,
            1.0
        ).otherwise(0.0)
    )

    counts = (
        scored
        .groupBy("label", "threshold_prediction")
        .count()
        .collect()
    )

    count_dict = {
        (row["label"], row["threshold_prediction"]): row["count"]
        for row in counts
    }

    TP = count_dict.get((1.0, 1.0), 0)
    TN = count_dict.get((0.0, 0.0), 0)
    FP = count_dict.get((0.0, 1.0), 0)
    FN = count_dict.get((1.0, 0.0), 0)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    threshold_results.append(
        (
            threshold,
            TP,
            FP,
            FN,
            TN,
            precision,
            recall,
            f1
        )
    )

print(
    f"{'Threshold':<10}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
    f"{'TN':>8}"
    f"{'Precision':>12}"
    f"{'Recall':>10}"
    f"{'F1':>10}"
)

print("-" * 84)

for result in threshold_results:
    threshold, TP, FP, FN, TN, precision, recall, f1 = result

    print(
        f"{threshold:<10.2f}"
        f"{TP:>8,}"
        f"{FP:>8,}"
        f"{FN:>8,}"
        f"{TN:>8,}"
        f"{precision:>12.4f}"
        f"{recall:>10.4f}"
        f"{f1:>10.4f}"
    )

## Train a Decision Tree Classifier

After optimizing the Logistic Regression classification threshold, a Decision Tree classifier is trained as a second candidate model.

Unlike Logistic Regression, a Decision Tree can capture nonlinear relationships and interactions between fraud-related features without requiring those relationships to be specified in advance.

The Decision Tree is trained using the same training dataset and evaluated using the same validation dataset to ensure a fair comparison.

Model selection will be based primarily on fraud-detection performance, including precision, recall, F1-score, and ROC-AUC.

### Training Decision Tree

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxDepth=8,
    minInstancesPerNode=20,
    seed=42
)

dt_model = dt.fit(train_ml_df)

print("Decision Tree training completed successfully.")

### Generate validation predictions & ROC-AUC

In [0]:
dt_val_predictions = dt_model.transform(val_ml_df)

dt_roc_auc = roc_evaluator.evaluate(dt_val_predictions)

print("Decision Tree validation predictions created.")
print(f"Validation rows: {dt_val_predictions.count():,}")
print(f"ROC-AUC: {dt_roc_auc:.4f}")

### Decision Tree Validation Metrics

In [0]:
metrics_row = dt_val_predictions.select(
    F.sum(
        F.when(
            (F.col("label") == 1) & (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("TP"),

    F.sum(
        F.when(
            (F.col("label") == 0) & (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("TN"),

    F.sum(
        F.when(
            (F.col("label") == 0) & (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("FP"),

    F.sum(
        F.when(
            (F.col("label") == 1) & (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("FN")
).collect()[0]

dt_TP = metrics_row["TP"]
dt_TN = metrics_row["TN"]
dt_FP = metrics_row["FP"]
dt_FN = metrics_row["FN"]

dt_accuracy = (
    (dt_TP + dt_TN) /
    (dt_TP + dt_TN + dt_FP + dt_FN)
)

dt_precision = (
    dt_TP / (dt_TP + dt_FP)
    if (dt_TP + dt_FP) > 0 else 0
)

dt_recall = (
    dt_TP / (dt_TP + dt_FN)
    if (dt_TP + dt_FN) > 0 else 0
)

dt_f1 = (
    2 * dt_precision * dt_recall /
    (dt_precision + dt_recall)
    if (dt_precision + dt_recall) > 0
    else 0
)

print("DECISION TREE — VALIDATION PERFORMANCE")
print("-" * 50)

print(f"True Positives:  {dt_TP:,}")
print(f"True Negatives:  {dt_TN:,}")
print(f"False Positives: {dt_FP:,}")
print(f"False Negatives: {dt_FN:,}")

print("\nMetrics")
print(f"Accuracy:  {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall:    {dt_recall:.4f}")
print(f"F1-score:  {dt_f1:.4f}")
print(f"ROC-AUC:   {dt_roc_auc:.4f}")

### Interpretation Note

The Decision Tree achieved a strong F1-score at its default classification threshold, but its ROC-AUC was considerably lower than that of Logistic Regression and Random Forest.

These results are not contradictory. F1-score evaluates classification performance at a specific operating threshold, whereas ROC-AUC evaluates the model's ability to rank fraudulent and legitimate invoices across all possible classification thresholds.

Therefore, although the Decision Tree performs strongly at its selected operating point, Random Forest provides stronger overall discrimination while maintaining comparable F1 performance.

## Candidate Model Comparison

The baseline classifier, Logistic Regression model, and Decision Tree model were compared using the validation dataset.

The majority-class baseline achieved relatively high accuracy because legitimate invoices dominate the dataset, but it failed to detect any fraudulent invoices.

Logistic Regression produced strong overall discrimination, achieving a ROC-AUC of 0.9618. Lowering its classification threshold from 0.50 to 0.30 improved fraud recall and F1-score while maintaining high precision.

The Decision Tree achieved the strongest threshold-based validation performance, with an F1-score of 0.9430 and fraud recall of 0.9130. It also produced fewer false negatives and false positives than Logistic Regression at the selected 0.30 threshold.

However, Logistic Regression achieved substantially higher ROC-AUC, indicating stronger ranking performance across classification thresholds.

The models will therefore be compared carefully before final model selection, with particular emphasis on fraud recall, precision, F1-score, and operational false-positive rates.

### Creating Comparison Table

In [0]:
comparison_data = [
    (
        "Baseline",
        0.7742,
        0.0000,
        0.0000,
        0.0000,
        None
    ),
    (
        "Logistic Regression (0.50)",
        0.9693,
        0.9779,
        0.8840,
        0.9286,
        0.9618
    ),
    (
        "Logistic Regression (0.30)",
        (6133 + 23015) / 30000,
        0.9666,
        0.9055,
        0.9351,
        0.9618
    ),
    (
        "Decision Tree",
        dt_accuracy,
        dt_precision,
        dt_recall,
        dt_f1,
        dt_roc_auc
    )
]

comparison_df = spark.createDataFrame(
    comparison_data,
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc"
    ]
)

comparison_df.show(truncate=False)

## Train a Random Forest Classifier

A Random Forest classifier is trained as the final candidate model.

Random Forest extends the Decision Tree approach by combining predictions from multiple trees. This can improve generalization, reduce sensitivity to individual tree structures, and capture nonlinear relationships and interactions between fraud-related features.

The model is trained using the same training dataset and evaluated on the validation dataset to maintain a consistent comparison with Logistic Regression and the single Decision Tree.

A moderate model configuration is used to balance predictive performance with computational efficiency.

### Training Random Forest

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=30,
    maxDepth=8,
    minInstancesPerNode=20,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_ml_df)

print("Random Forest training completed successfully.")

### Validation predictions & ROC-AUC

In [0]:
rf_val_predictions = rf_model.transform(val_ml_df)

rf_roc_auc = roc_evaluator.evaluate(rf_val_predictions)

print("Random Forest validation predictions created.")
print(f"Validation rows: {rf_val_predictions.count():,}")
print(f"ROC-AUC: {rf_roc_auc:.4f}")

### Random Forest Metrics

In [0]:
rf_metrics_row = rf_val_predictions.select(

    F.sum(
        F.when(
            (F.col("label") == 1) &
            (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("TP"),

    F.sum(
        F.when(
            (F.col("label") == 0) &
            (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("TN"),

    F.sum(
        F.when(
            (F.col("label") == 0) &
            (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("FP"),

    F.sum(
        F.when(
            (F.col("label") == 1) &
            (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("FN")

).collect()[0]

rf_TP = rf_metrics_row["TP"]
rf_TN = rf_metrics_row["TN"]
rf_FP = rf_metrics_row["FP"]
rf_FN = rf_metrics_row["FN"]

rf_accuracy = (
    (rf_TP + rf_TN) /
    (rf_TP + rf_TN + rf_FP + rf_FN)
)

rf_precision = (
    rf_TP / (rf_TP + rf_FP)
    if (rf_TP + rf_FP) > 0
    else 0
)

rf_recall = (
    rf_TP / (rf_TP + rf_FN)
    if (rf_TP + rf_FN) > 0
    else 0
)

rf_f1 = (
    2 * rf_precision * rf_recall /
    (rf_precision + rf_recall)
    if (rf_precision + rf_recall) > 0
    else 0
)

print("RANDOM FOREST — VALIDATION PERFORMANCE")
print("-" * 50)

print(f"True Positives:  {rf_TP:,}")
print(f"True Negatives:  {rf_TN:,}")
print(f"False Positives: {rf_FP:,}")
print(f"False Negatives: {rf_FN:,}")

print("\nMetrics")
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-score:  {rf_f1:.4f}")
print(f"ROC-AUC:   {rf_roc_auc:.4f}")

## Final Model Selection

Three machine learning approaches were evaluated using the validation dataset: Logistic Regression, Decision Tree, and Random Forest.

The Decision Tree achieved the highest validation F1-score (0.9430) and precision (0.9751). However, the Random Forest achieved an almost identical F1-score (0.9429), while providing the highest fraud recall (0.9178) and the highest ROC-AUC (0.9638).

The Random Forest correctly identified 6,216 of the 6,773 fraudulent validation invoices, leaving 557 false negatives. It maintained high precision at 0.9694, indicating that most invoices flagged as fraudulent were genuinely fraudulent.

Given the objective of RiskCheck AI to identify fraudulent invoices while maintaining a low false-alert rate, Random Forest is selected as the preferred final model. Its combination of strong recall, precision, F1-score, and ROC-AUC provides the most balanced validation performance.

The held-out test dataset, which has not been used for model training or model selection, will now be used once for final performance evaluation.

### Final Test Evaluation

In [0]:
test_ml_df = assembler.transform(test_df).select(
    F.col("is_fraud").cast("double").alias("label"),
    "features"
)

print("Test ML dataset created successfully.")
print(f"Test rows: {test_ml_df.count():,}")

### Generate Final Random Forest Test Predictions

In [0]:
rf_test_predictions = rf_model.transform(test_ml_df)

test_roc_auc = roc_evaluator.evaluate(rf_test_predictions)

print("Final Random Forest test predictions created.")
print(f"Test predictions: {rf_test_predictions.count():,}")
print(f"Test ROC-AUC: {test_roc_auc:.4f}")

### Final Test Metrics

In [0]:
test_metrics_row = rf_test_predictions.select(

    F.sum(
        F.when(
            (F.col("label") == 1) &
            (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("TP"),

    F.sum(
        F.when(
            (F.col("label") == 0) &
            (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("TN"),

    F.sum(
        F.when(
            (F.col("label") == 0) &
            (F.col("prediction") == 1),
            1
        ).otherwise(0)
    ).alias("FP"),

    F.sum(
        F.when(
            (F.col("label") == 1) &
            (F.col("prediction") == 0),
            1
        ).otherwise(0)
    ).alias("FN")

).collect()[0]

test_TP = test_metrics_row["TP"]
test_TN = test_metrics_row["TN"]
test_FP = test_metrics_row["FP"]
test_FN = test_metrics_row["FN"]

test_accuracy = (
    (test_TP + test_TN) /
    (test_TP + test_TN + test_FP + test_FN)
)

test_precision = (
    test_TP / (test_TP + test_FP)
    if (test_TP + test_FP) > 0 else 0
)

test_recall = (
    test_TP / (test_TP + test_FN)
    if (test_TP + test_FN) > 0 else 0
)

test_f1 = (
    2 * test_precision * test_recall /
    (test_precision + test_recall)
    if (test_precision + test_recall) > 0
    else 0
)

print("RISKCHECK AI — FINAL TEST PERFORMANCE")
print("-" * 50)

print("Confusion Matrix")
print(f"True Positives:  {test_TP:,}")
print(f"True Negatives:  {test_TN:,}")
print(f"False Positives: {test_FP:,}")
print(f"False Negatives: {test_FN:,}")

print("\nFinal Metrics")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")

## Week 4 Conclusion

Week 4 focused on developing, evaluating, and selecting a machine learning model for fraud detection using the modelling-ready datasets prepared during Week 3.

The target-distribution analysis showed that approximately 22% of invoices were labelled as fraudulent. A majority-class baseline demonstrated why accuracy alone was insufficient for evaluating fraud detection: although the baseline achieved approximately 77% validation accuracy, it failed to identify any fraudulent invoices.

Three machine learning approaches were subsequently evaluated: Logistic Regression, Decision Tree, and Random Forest.

Logistic Regression demonstrated strong discrimination, achieving a validation ROC-AUC of 0.9618. Threshold optimization improved its fraud recall and F1-score, with a threshold of 0.30 providing the strongest performance among the evaluated thresholds.

The Decision Tree achieved strong threshold-based classification performance, producing a validation F1-score of 0.9430 and recall of 0.9130. However, its ROC-AUC of 0.8699 was substantially lower than the other candidate models.

Random Forest provided the strongest overall balance of performance. On the validation dataset, it achieved:

- Accuracy: 0.9749
- Precision: 0.9694
- Recall: 0.9178
- F1-score: 0.9429
- ROC-AUC: 0.9638

Based on its high fraud recall, strong precision and F1-score, and highest ROC-AUC, Random Forest was selected as the final RiskCheck AI fraud-classification model.

The selected model was then evaluated once on the previously untouched 60,000-record test dataset. It achieved:

- Accuracy: 0.9745
- Precision: 0.9703
- Recall: 0.9148
- F1-score: 0.9417
- ROC-AUC: 0.9631

The final model correctly identified 12,359 fraudulent invoices while producing 378 false-positive alerts and 1,151 false negatives.

The close agreement between validation and test performance indicates stable generalization to unseen data. The final Random Forest model therefore provides a strong foundation for the next stage of RiskCheck AI, where model interpretation, fraud-risk explanation, and deployment-oriented analysis can be developed.

In [0]:
import mlflow
import mlflow.spark

from pyspark.ml import PipelineModel

riskcheck_pipeline_model = PipelineModel(stages=[
    assembler,
    rf_model
])

print("RiskCheck AI pipeline created successfully.")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.riskcheck_ai_mlflow;

In [0]:
import mlflow
import mlflow.spark

mlflow.set_experiment("/Shared/RiskCheck_AI")

with mlflow.start_run(run_name="RiskCheck_AI_Final_RandomForest") as run:

    mlflow.spark.log_model(
    spark_model=riskcheck_pipeline_model,
    artifact_path="riskcheck_model",
    dfs_tmpdir="/Volumes/workspace/default/riskcheck_ai_mlflow"
)

    mlflow.log_metric("test_accuracy", float(test_accuracy))
    mlflow.log_metric("test_precision", float(test_precision))
    mlflow.log_metric("test_recall", float(test_recall))
    mlflow.log_metric("test_f1", float(test_f1))
    mlflow.log_metric("test_roc_auc", float(test_roc_auc))

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", 30)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_instances_per_node", 20)
    mlflow.log_param("feature_subset_strategy", "sqrt")
    mlflow.log_param("seed", 42)

    run_id = run.info.run_id

print("RiskCheck AI model logged successfully.")
print("Run ID:", run_id)

In [0]:
import mlflow
import mlflow.spark

from mlflow.models import infer_signature

# Exact 32 feature columns
feature_cols = assembler.getInputCols()

# Use test_df, NOT test_ml_df
input_example_pdf = (
    test_df
    .select(*feature_cols)
    .limit(5)
    .toPandas()
)

print("Number of model features:", len(feature_cols))
print("Input example created successfully.")
print("Shape:", input_example_pdf.shape)

display(input_example_pdf)

In [0]:
# Create Spark DataFrame from the 5-row input example
sample_spark_df = spark.createDataFrame(input_example_pdf)

# Run the complete RiskCheck pipeline
sample_predictions = (
    riskcheck_pipeline_model
    .transform(sample_spark_df)
    .select("prediction", "probability")
)

# Convert to pandas
sample_output_pdf = sample_predictions.toPandas()

# Extract fraud probability = probability of class 1
sample_output_pdf["fraud_probability"] = (
    sample_output_pdf["probability"]
    .apply(lambda x: float(x[1]))
)

# Keep only clean serving outputs
signature_output = sample_output_pdf[
    ["prediction", "fraud_probability"]
]

# Infer MLflow signature
signature = infer_signature(
    input_example_pdf,
    signature_output
)

print("Signature created successfully.")
print(signature)

display(signature_output)

In [0]:
import mlflow
import mlflow.spark

mlflow.set_experiment("/Shared/RiskCheck_AI")

with mlflow.start_run(run_name="RiskCheck_AI_Final_RandomForest_Signed") as run:

    mlflow.spark.log_model(
        spark_model=riskcheck_pipeline_model,
        artifact_path="riskcheck_model",
        dfs_tmpdir="/Volumes/workspace/default/riskcheck_ai_mlflow",
        signature=signature,
        input_example=input_example_pdf
    )

    mlflow.log_metric("test_accuracy", float(test_accuracy))
    mlflow.log_metric("test_precision", float(test_precision))
    mlflow.log_metric("test_recall", float(test_recall))
    mlflow.log_metric("test_f1", float(test_f1))
    mlflow.log_metric("test_roc_auc", float(test_roc_auc))

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", 30)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_instances_per_node", 20)
    mlflow.log_param("feature_subset_strategy", "sqrt")
    mlflow.log_param("seed", 42)

    signed_run_id = run.info.run_id

print("Signed RiskCheck AI model logged successfully.")
print("Run ID:", signed_run_id)

In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/riskcheck_ai_mlflow"

print("MLFLOW_DFS_TMP set successfully.")

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

model_uri = "runs:/ec270755c7aa4ac2a86f2b251f39d0b5/riskcheck_model"
registered_model_name = "workspace.default.riskcheck_ai"

result = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("RiskCheck AI model registered successfully.")
print("Model name:", result.name)
print("Version:", result.version)

In [0]:
import pyspark
import mlflow

print("PySpark version:", pyspark.__version__)
print("MLflow version:", mlflow.__version__)

print("\nRandom Forest model parameters:")
for p in rf_model.params:
    try:
        print(f"{p.name} = {rf_model.getOrDefault(p)}")
    except:
        print(f"{p.name} = <no default>")

In [0]:
print("Number of trees:", len(rf_model.trees))
print("Total nodes:", rf_model.totalNumNodes)

print("\nFirst tree preview:")
print(rf_model.trees[0].toDebugString[:3000])

In [0]:
# Export all Random Forest trees as debug strings

tree_debug_strings = [
    tree.toDebugString
    for tree in rf_model.trees
]

print("Trees exported:", len(tree_debug_strings))
print("First tree length:", len(tree_debug_strings[0]))
print("Last tree length:", len(tree_debug_strings[-1]))

In [0]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Same 32 features used by the Spark model
feature_cols = assembler.getInputCols()

# Convert the existing Spark train/test datasets to pandas
train_pdf = (
    train_df
    .select(*feature_cols, "is_fraud")
    .toPandas()
)

test_pdf = (
    test_df
    .select(*feature_cols, "is_fraud")
    .toPandas()
)

X_train = train_pdf[feature_cols]
y_train = train_pdf["is_fraud"].astype(int)

X_test = test_pdf[feature_cols]
y_test = test_pdf["is_fraud"].astype(int)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Serving-compatible Random Forest
sk_rf_model = RandomForestClassifier(
    n_estimators=30,
    max_depth=8,
    min_samples_leaf=20,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

sk_rf_model.fit(X_train, y_train)

# Predictions
sk_predictions = sk_rf_model.predict(X_test)
sk_probabilities = sk_rf_model.predict_proba(X_test)[:, 1]

# Evaluation
sk_accuracy = accuracy_score(y_test, sk_predictions)
sk_precision = precision_score(y_test, sk_predictions)
sk_recall = recall_score(y_test, sk_predictions)
sk_f1 = f1_score(y_test, sk_predictions)
sk_roc_auc = roc_auc_score(y_test, sk_probabilities)

print("\nScikit-learn Random Forest Results")
print("----------------------------------")
print(f"Accuracy : {sk_accuracy:.4f}")
print(f"Precision: {sk_precision:.4f}")
print(f"Recall   : {sk_recall:.4f}")
print(f"F1 Score : {sk_f1:.4f}")
print(f"ROC-AUC  : {sk_roc_auc:.4f}")

In [0]:
probability_check = test_pdf[feature_cols].head(10).copy()

probability_check["prediction"] = sk_rf_model.predict(
    probability_check[feature_cols]
)

probability_check["fraud_probability"] = sk_rf_model.predict_proba(
    probability_check[feature_cols]
)[:, 1]

print(
    probability_check[
        ["prediction", "fraud_probability"]
    ]
)

In [0]:
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

# Input example
sk_input_example = X_test.head(5).copy()

# Output example
sk_output_example = sk_rf_model.predict_proba(sk_input_example)[:, 1]

# Signature: 32 input features -> fraud probability
sk_signature = infer_signature(
    sk_input_example,
    sk_output_example
)

mlflow.set_experiment("/Shared/RiskCheck_AI")

with mlflow.start_run(run_name="RiskCheck_AI_Sklearn_Serving") as run:

    mlflow.sklearn.log_model(
        sk_model=sk_rf_model,
        artifact_path="riskcheck_model",
        signature=sk_signature,
        input_example=sk_input_example
    )

    mlflow.log_metric("test_accuracy", float(sk_accuracy))
    mlflow.log_metric("test_precision", float(sk_precision))
    mlflow.log_metric("test_recall", float(sk_recall))
    mlflow.log_metric("test_f1", float(sk_f1))
    mlflow.log_metric("test_roc_auc", float(sk_roc_auc))

    mlflow.log_param("model_type", "sklearn.RandomForestClassifier")
    mlflow.log_param("n_estimators", 30)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_samples_leaf", 20)
    mlflow.log_param("max_features", "sqrt")
    mlflow.log_param("random_state", 42)

    sklearn_run_id = run.info.run_id

print("Scikit-learn serving model logged successfully.")
print("Run ID:", sklearn_run_id)

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

model_uri = "runs:/50fe7bec746a486ca68bf16e6600bdaa/riskcheck_model"
registered_model_name = "workspace.default.riskcheck_ai"

result_v2 = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("RiskCheck AI serving model registered successfully.")
print("Model name:", result_v2.name)
print("Version:", result_v2.version)

In [0]:
import mlflow
import pandas as pd
import numpy as np

class RiskCheckPyFuncModel(mlflow.pyfunc.PythonModel):

    def __init__(self, model, feature_cols):
        self.model = model
        self.feature_cols = feature_cols

    def predict(self, context, model_input):
        # Ensure expected column order
        X = model_input[self.feature_cols].copy()

        predictions = self.model.predict(X)
        fraud_probabilities = self.model.predict_proba(X)[:, 1]

        return pd.DataFrame({
            "prediction": predictions.astype(int),
            "fraud_probability": fraud_probabilities.astype(float)
        })


riskcheck_pyfunc_model = RiskCheckPyFuncModel(
    model=sk_rf_model,
    feature_cols=feature_cols
)

print("RiskCheck probability wrapper created successfully.")

In [0]:
wrapper_test = riskcheck_pyfunc_model.predict(
    None,
    X_test.head(5)
)

print(wrapper_test)
print("\nColumns:", wrapper_test.columns.tolist())

In [0]:
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature

# Sample input/output for signature
pyfunc_input_example = X_test.head(5).copy()
pyfunc_output_example = wrapper_test.copy()

pyfunc_signature = infer_signature(
    pyfunc_input_example,
    pyfunc_output_example
)

mlflow.set_experiment("/Shared/RiskCheck_AI")

with mlflow.start_run(run_name="RiskCheck_AI_PyFunc_Probability") as run:

    mlflow.pyfunc.log_model(
        artifact_path="riskcheck_model",
        python_model=riskcheck_pyfunc_model,
        signature=pyfunc_signature,
        input_example=pyfunc_input_example
    )

    mlflow.log_metric("test_accuracy", float(sk_accuracy))
    mlflow.log_metric("test_precision", float(sk_precision))
    mlflow.log_metric("test_recall", float(sk_recall))
    mlflow.log_metric("test_f1", float(sk_f1))
    mlflow.log_metric("test_roc_auc", float(sk_roc_auc))

    mlflow.log_param("model_type", "RiskCheckPyFunc + sklearn RandomForest")
    mlflow.log_param("n_estimators", 30)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("min_samples_leaf", 20)
    mlflow.log_param("max_features", "sqrt")
    mlflow.log_param("random_state", 42)

    pyfunc_run_id = run.info.run_id

print("RiskCheck probability-serving model logged successfully.")
print("Run ID:", pyfunc_run_id)

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

model_uri = "runs:/5e375481a4434822be0dd8a40313ad2c/riskcheck_model"
registered_model_name = "workspace.default.riskcheck_ai"

result_probability = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("RiskCheck probability model registered successfully.")
print("Model name:", result_probability.name)
print("Version:", result_probability.version)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.riskcheck_feedback (
    invoice_id STRING,
    verified_outcome STRING,
    investigator_note STRING,
    verified_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
SELECT *
FROM workspace.default.riskcheck_feedback;

In [0]:
%sql
SELECT *
FROM workspace.default.riskcheck_investigator_feedback
ORDER BY verified_at DESC;

In [0]:
%sql
DELETE FROM workspace.default.riskcheck_investigator_feedback
WHERE invoice_id = 'INV-DEMO-005';

In [0]:
%sql

ALTER TABLE workspace.default.riskcheck_scored_invoices
ADD COLUMNS (
    batch_id STRING,
    source_file STRING
);

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.riskcheck_scored_invoices;

COUNT(*)
1881


# Week 5: Model Explainability and Fraud Risk Interpretation

## Overview

Week 5 focuses on interpreting the fraud detection model selected during Week 4 and transforming its predictions into meaningful fraud risk information.

The Random Forest classifier demonstrated strong performance on both the validation and held-out test datasets. However, predictive performance alone does not explain which invoice characteristics are most influential in the model's fraud detection decisions.

This week therefore focuses on examining model feature importance, understanding the strongest fraud-related signals, analysing predicted fraud probabilities, and developing interpretable risk categories that can support invoice investigation and decision-making.

## Week 5 Objectives

- Confirm the selected Random Forest model and modelling feature structure.
- Extract and rank global feature importance from the Random Forest model.
- Identify the strongest predictive signals associated with fraud detection.
- Interpret important invoice, supplier, behavioural, timing, and image-related features.
- Analyse fraud probabilities generated by the selected model.
- Develop interpretable fraud risk categories.
- Create investigation-oriented risk outputs for individual invoices.
- Validate the explainability and fraud risk interpretation workflow.

## Confirm the Selected Model & Feature Structure

Week 4 selected the Random Forest classifier as the final RiskCheck AI fraud detection model based on its strong validation performance and stable generalisation to the held-out test dataset.

Before analysing model explainability, the selected model and its feature structure are verified.

The order of the modelling features is particularly important because the Random Forest feature importance values correspond directly to the feature positions used during model training.

In [0]:
print("Selected RiskCheck AI model:")
print(type(rf_model).__name__)

print(f"\nNumber of modelling features: {len(model_feature_columns)}")

print("\nModel feature structure:")
print("-" * 55)

for index, feature in enumerate(model_feature_columns):
    print(f"{index:>2}: {feature}")

## Analyse Global Feature Importance

Random Forest feature importance provides a global view of which predictor variables contribute most strongly to the model's classification decisions.

Each modelling feature receives an importance score representing its relative contribution across the collection of decision trees. Higher importance values indicate that a feature was used more strongly by the model when separating fraudulent and legitimate invoices.

Feature importance should be interpreted as a measure of predictive influence within the model rather than evidence of a causal relationship with fraud.

The feature importance values are mapped back to the original 32 modelling features and ranked from highest to lowest importance.

### Extract Feature Importance

In [0]:
feature_importance_values = rf_model.featureImportances.toArray()

feature_importance_data = [
    (feature, float(importance))
    for feature, importance
    in zip(model_feature_columns, feature_importance_values)
]

feature_importance_df = spark.createDataFrame(
    feature_importance_data,
    ["feature", "importance"]
).orderBy(F.desc("importance"))

print("Random Forest feature importance extracted successfully.")
print(f"Number of features: {feature_importance_df.count()}")

feature_importance_df.show(32, truncate=False)

### Validating Importance Scores

In [0]:
importance_total = sum(feature_importance_values)

print(f"Number of importance values: {len(feature_importance_values)}")
print(f"Total feature importance: {importance_total:.6f}")

##  Identify the Strongest Fraud Predictors

The global feature importance results indicate that the Random Forest does not rely equally on all 32 predictor variables.

Several features contribute substantially more to the model than others. In particular, split-invoice behaviour, invoice amounts relative to supplier historical averages, supplier blacklist status, and invoice amount characteristics appear among the strongest predictive signals.

To make the model interpretation more concise, the ten most important features are extracted and their cumulative contribution to the Random Forest model is calculated.

This analysis provides a high-level explanation of the signals most strongly used by RiskCheck AI when distinguishing potentially fraudulent invoices from legitimate invoices.

### Top 10 Features

In [0]:
top_10_importance = (
    feature_importance_df
    .limit(10)
    .collect()
)

print("TOP 10 RANDOM FOREST FRAUD PREDICTORS")
print("-" * 75)

for rank, row in enumerate(top_10_importance, start=1):
    print(
        f"{rank:>2}. "
        f"{row['feature']:<38} "
        f"{row['importance']:.6f}"
    )

### Measure Cumulative Importance

In [0]:
sorted_importances = sorted(
    feature_importance_data,
    key=lambda x: x[1],
    reverse=True
)

top_4_importance = sum(
    importance
    for _, importance in sorted_importances[:4]
)

top_10_total_importance = sum(
    importance
    for _, importance in sorted_importances[:10]
)

print("FEATURE IMPORTANCE CONCENTRATION")
print("-" * 50)

print(
    f"Top 4 features:  "
    f"{top_4_importance:.4f} "
    f"({top_4_importance * 100:.2f}%)"
)

print(
    f"Top 10 features: "
    f"{top_10_total_importance:.4f} "
    f"({top_10_total_importance * 100:.2f}%)"
)

print(
    f"Remaining 22:    "
    f"{1 - top_10_total_importance:.4f} "
    f"({(1 - top_10_total_importance) * 100:.2f}%)"
)

## Interpret the Strongest Fraud Signals

The feature importance analysis identifies which variables contribute most strongly to the Random Forest model, but importance scores alone do not indicate the direction of their relationship with fraud.

To better understand the model's strongest predictive signals, the leading features are compared between legitimate and fraudulent invoices.

The analysis focuses on the highest-ranked behavioural, supplier-risk, and invoice-amount features. Mean values and binary-flag rates are examined separately for legitimate and fraudulent invoices.

This provides additional context for interpreting the patterns used by RiskCheck AI while avoiding the assumption that feature importance represents a causal relationship with fraud.

### Compare the Major Binary Fraud Indicators

In [0]:
binary_signal_summary = (
    train_df
    .groupBy("is_fraud")
    .agg(
        F.count("*").alias("invoice_count"),

        F.avg("split_invoice_flag").alias(
            "split_invoice_rate"
        ),

        F.avg("blacklisted_flag").alias(
            "blacklisted_supplier_rate"
        )
    )
    .withColumn(
        "split_invoice_percentage",
        F.round(F.col("split_invoice_rate") * 100, 2)
    )
    .withColumn(
        "blacklisted_supplier_percentage",
        F.round(F.col("blacklisted_supplier_rate") * 100, 2)
    )
    .select(
        "is_fraud",
        "invoice_count",
        "split_invoice_percentage",
        "blacklisted_supplier_percentage"
    )
    .orderBy("is_fraud")
)

binary_signal_summary.show(truncate=False)

### Compare Major Amount-based Signals

In [0]:
amount_signal_summary = (
    train_df
    .groupBy("is_fraud")
    .agg(
        F.avg(
            "amount_to_supplier_90d_avg_ratio"
        ).alias("avg_90d_amount_ratio"),

        F.avg(
            "amount_to_supplier_avg_ratio"
        ).alias("avg_supplier_amount_ratio"),

        F.avg(
            "invoice_amount"
        ).alias("avg_invoice_amount_current"),

        F.avg(
            "amount_diff_supplier_90d_avg"
        ).alias("avg_amount_difference"),

        F.avg(
            "invoice_amount_zscore"
        ).alias("avg_invoice_zscore"),

        F.avg(
            "supplier_risk_score"
        ).alias("avg_supplier_risk_score")
    )
    .orderBy("is_fraud")
)

amount_signal_summary.show(
    truncate=False
)

## Analyse Fraud Probability Scores

The Random Forest classifier produces a probability estimate for each invoice in addition to its final binary fraud prediction.

The fraud probability represents the model's estimated confidence that an invoice belongs to the fraudulent class based on the patterns learned during training.

Analysing these probabilities provides more information than a simple fraud or non-fraud classification. An invoice with a very high fraud probability may warrant greater investigative attention than an invoice positioned close to the classification threshold.

The held-out test predictions are used at this stage to examine the distribution of model scores. No further model training or parameter tuning is performed using the test dataset.

### Extract Fraud Probability

In [0]:
from pyspark.ml.functions import vector_to_array

risk_scored_df = (
    rf_test_predictions
    .withColumn(
        "fraud_probability",
        vector_to_array(F.col("probability"))[1]
    )
)

print("Fraud probability scores extracted successfully.")
print(f"Scored test invoices: {risk_scored_df.count():,}")

risk_scored_df.select(
    "label",
    "prediction",
    "fraud_probability"
).show(10, truncate=False)

### Probability Summary by Actual Class

In [0]:
probability_summary = (
    risk_scored_df
    .groupBy("label")
    .agg(
        F.count("*").alias("invoice_count"),
        F.avg("fraud_probability").alias("avg_fraud_probability"),
        F.min("fraud_probability").alias("min_fraud_probability"),
        F.max("fraud_probability").alias("max_fraud_probability")
    )
    .orderBy("label")
)

probability_summary.show(truncate=False)

## Develop Fraud Risk Bands

Binary fraud predictions provide a simple classification outcome, but operational fraud investigation can benefit from a more detailed representation of model risk.

The Random Forest fraud probability is therefore transformed into a set of interpretable risk bands. These categories allow invoices to be prioritised according to the strength of the model-generated fraud signal rather than relying only on a binary prediction.

Before finalising the risk categories, the distribution of invoices across fraud probability ranges is examined. This helps ensure that the selected bands reflect the behaviour of the trained model.

The resulting risk levels are intended to support investigation prioritisation and should be interpreted as model-generated risk categories rather than definitive evidence of fraud.

### Inspect Probability Bands

In [0]:
probability_band_df = (
    risk_scored_df
    .withColumn(
        "probability_band",
        F.when(
            F.col("fraud_probability") < 0.20,
            "0.00 - 0.19"
        )
        .when(
            F.col("fraud_probability") < 0.40,
            "0.20 - 0.39"
        )
        .when(
            F.col("fraud_probability") < 0.60,
            "0.40 - 0.59"
        )
        .when(
            F.col("fraud_probability") < 0.80,
            "0.60 - 0.79"
        )
        .otherwise("0.80 - 1.00")
    )
)

band_summary = (
    probability_band_df
    .groupBy("probability_band")
    .agg(
        F.count("*").alias("invoice_count"),

        F.sum(
            F.when(F.col("label") == 1, 1).otherwise(0)
        ).alias("fraud_count"),

        F.sum(
            F.when(F.col("label") == 0, 1).otherwise(0)
        ).alias("legitimate_count")
    )
    .withColumn(
        "observed_fraud_rate",
        F.round(
            F.col("fraud_count") /
            F.col("invoice_count") * 100,
            2
        )
    )
)

band_summary.orderBy(
    F.when(F.col("probability_band") == "0.00 - 0.19", 1)
     .when(F.col("probability_band") == "0.20 - 0.39", 2)
     .when(F.col("probability_band") == "0.40 - 0.59", 3)
     .when(F.col("probability_band") == "0.60 - 0.79", 4)
     .otherwise(5)
).show(truncate=False)

### Checking Concentration of Fraud Cases

In [0]:
fraud_band_summary = (
    probability_band_df
    .filter(F.col("label") == 1)
    .groupBy("probability_band")
    .count()
    .withColumn(
        "percentage_of_all_fraud",
        F.round(
            F.col("count") / F.lit(13510) * 100,
            2
        )
    )
)

fraud_band_summary.orderBy(
    F.when(F.col("probability_band") == "0.00 - 0.19", 1)
     .when(F.col("probability_band") == "0.20 - 0.39", 2)
     .when(F.col("probability_band") == "0.40 - 0.59", 3)
     .when(F.col("probability_band") == "0.60 - 0.79", 4)
     .otherwise(5)
).show(truncate=False)

## Create Interpretable Fraud Risk Categories

The fraud probability analysis demonstrates a clear relationship between the Random Forest risk score and observed fraud prevalence.

Invoices with scores below 0.20 have a relatively low observed fraud rate, while invoices receiving scores above 0.60 show a very high concentration of fraudulent cases. The intermediate score ranges provide additional separation between lower-risk and increasingly suspicious invoices.

Based on the observed score distribution, five fraud risk categories are defined:

- **Low Risk:** fraud score below 0.20
- **Moderate Risk:** fraud score from 0.20 to below 0.40
- **Elevated Risk:** fraud score from 0.40 to below 0.60
- **High Risk:** fraud score from 0.60 to below 0.80
- **Critical Risk:** fraud score from 0.80 to 1.00

These categories are intended to support investigation prioritisation rather than provide definitive conclusions about whether fraud has occurred. In particular, a low model-generated risk score does not guarantee that an invoice is legitimate.

### Assigning Risk Categories

In [0]:
risk_scored_df = (
    risk_scored_df
    .withColumn(
        "risk_category",
        F.when(
            F.col("fraud_probability") < 0.20,
            "Low Risk"
        )
        .when(
            F.col("fraud_probability") < 0.40,
            "Moderate Risk"
        )
        .when(
            F.col("fraud_probability") < 0.60,
            "Elevated Risk"
        )
        .when(
            F.col("fraud_probability") < 0.80,
            "High Risk"
        )
        .otherwise("Critical Risk")
    )
)

print("Fraud risk categories created successfully.")

risk_scored_df.select(
    "label",
    "prediction",
    "fraud_probability",
    "risk_category"
).show(15, truncate=False)

### Validating Categories

In [0]:
risk_category_summary = (
    risk_scored_df
    .groupBy("risk_category")
    .agg(
        F.count("*").alias("invoice_count"),

        F.sum(
            F.when(F.col("label") == 1, 1).otherwise(0)
        ).alias("fraud_count"),

        F.sum(
            F.when(F.col("label") == 0, 1).otherwise(0)
        ).alias("legitimate_count")
    )
    .withColumn(
        "observed_fraud_rate",
        F.round(
            F.col("fraud_count") /
            F.col("invoice_count") * 100,
            2
        )
    )
)

risk_order = (
    F.when(F.col("risk_category") == "Low Risk", 1)
     .when(F.col("risk_category") == "Moderate Risk", 2)
     .when(F.col("risk_category") == "Elevated Risk", 3)
     .when(F.col("risk_category") == "High Risk", 4)
     .otherwise(5)
)

risk_category_summary.orderBy(
    risk_order
).show(truncate=False)

## Create Investigator-Facing Fraud Risk Outputs

The model-generated fraud probability and risk category provide useful measures of invoice risk, but investigators also require contextual information explaining why an invoice may warrant attention.

An investigator-facing output is therefore created by combining the model risk score with selected fraud indicators identified during the global feature importance analysis.

The output includes the invoice identifier, fraud probability, assigned risk category, model prediction, and several important behavioural, supplier, and amount-related indicators.

These indicators provide supporting context for investigation. They should not be interpreted as causal explanations of an individual model prediction, since global Random Forest feature importance does not provide prediction-level attribution.

### Attaching Scores to Original Test Records

In [0]:
test_explain_source_df = (
    prepared_df
    .filter(F.col("split") == "test")
    .select(
        "invoice_id",
        *(final_feature_columns + ["is_fraud"])
    )
)

test_explain_df = assembler.transform(test_explain_source_df)

test_explain_predictions = (
    rf_model
    .transform(test_explain_df)
    .withColumn(
        "fraud_probability",
        vector_to_array(F.col("probability"))[1]
    )
    .withColumn(
        "risk_category",
        F.when(
            F.col("fraud_probability") < 0.20,
            "Low Risk"
        )
        .when(
            F.col("fraud_probability") < 0.40,
            "Moderate Risk"
        )
        .when(
            F.col("fraud_probability") < 0.60,
            "Elevated Risk"
        )
        .when(
            F.col("fraud_probability") < 0.80,
            "High Risk"
        )
        .otherwise("Critical Risk")
    )
)

print("Investigator-ready predictions created successfully.")
print(f"Rows: {test_explain_predictions.count():,}")
print(
    "Unique invoices:",
    f"{test_explain_predictions.select('invoice_id').distinct().count():,}"
)

### Building Investigation View

In [0]:
investigation_view_df = (
    test_explain_predictions
    .select(
        "invoice_id",

        F.col("is_fraud")
        .alias("actual_label"),

        F.col("prediction")
        .cast("int")
        .alias("model_prediction"),

        F.round(
            F.col("fraud_probability"),
            4
        ).alias("fraud_risk_score"),

        "risk_category",

        "split_invoice_flag",
        "blacklisted_flag",

        F.round(
            F.col("amount_to_supplier_90d_avg_ratio"),
            3
        ).alias("supplier_90d_amount_ratio"),

        F.round(
            F.col("amount_to_supplier_avg_ratio"),
            3
        ).alias("supplier_amount_ratio"),

        F.round(
            F.col("invoice_amount"),
            2
        ).alias("invoice_amount"),

        F.round(
            F.col("amount_diff_supplier_90d_avg"),
            2
        ).alias("amount_difference_90d"),

        F.round(
            F.col("invoice_amount_zscore"),
            3
        ).alias("invoice_amount_zscore"),

        F.round(
            F.col("supplier_risk_score"),
            3
        ).alias("supplier_risk_score")
    )
)

print("Investigation view created successfully.")

investigation_view_df.orderBy(
    F.desc("fraud_risk_score")
).show(10, truncate=False)

## Generate Investigation-Oriented Risk Reasons

The investigator-facing dataset provides model scores and important fraud indicators, but reviewing multiple numerical columns for every invoice may be inefficient.

To improve interpretability, concise investigation reasons are generated from selected high-importance fraud signals.

These reasons are rule-based summaries derived from observed invoice characteristics and globally important features. They are intended to provide contextual indicators for investigation rather than exact local explanations of the Random Forest prediction.

The generated reasons focus on:

- Split-invoice behaviour
- Supplier blacklist status
- Unusual invoice amounts relative to supplier history
- Large deviations from recent supplier invoice patterns
- Elevated supplier risk
- Unusual invoice amount z-scores

Multiple indicators can be associated with the same invoice.

### Generate Risk Reasons

In [0]:
from pyspark.sql import functions as F

investigation_reasons_df = (
    investigation_view_df

    .withColumn(
        "reason_split_invoice",
        F.when(
            F.col("split_invoice_flag") == 1,
            F.lit("Split-invoice behaviour detected")
        )
    )

    .withColumn(
        "reason_blacklisted_supplier",
        F.when(
            F.col("blacklisted_flag") == 1,
            F.lit("Supplier is blacklisted")
        )
    )

    .withColumn(
        "reason_supplier_ratio",
        F.when(
            (F.col("supplier_90d_amount_ratio") < 0.60) |
            (F.col("supplier_90d_amount_ratio") > 1.50),
            F.lit("Invoice amount differs substantially from recent supplier pattern")
        )
    )

    .withColumn(
        "reason_amount_difference",
        F.when(
            F.col("amount_difference_90d") > 5000,
            F.lit("Large deviation from supplier 90-day average")
        )
    )

    .withColumn(
        "reason_supplier_risk",
        F.when(
            F.col("supplier_risk_score") >= 0.60,
            F.lit("Elevated supplier risk score")
        )
    )

    .withColumn(
        "reason_zscore",
        F.when(
            F.abs(F.col("invoice_amount_zscore")) >= 2,
            F.lit("Unusual invoice amount relative to historical distribution")
        )
    )

    .withColumn(
        "investigation_reasons",
        F.array_compact(
            F.array(
                "reason_split_invoice",
                "reason_blacklisted_supplier",
                "reason_supplier_ratio",
                "reason_amount_difference",
                "reason_supplier_risk",
                "reason_zscore"
            )
        )
    )
)

print("Investigation reasons generated successfully.")

### Inspecting High-risk Invoices with Reasons

In [0]:
investigation_reasons_df.select(
    "invoice_id",
    "fraud_risk_score",
    "risk_category",
    "model_prediction",
    "investigation_reasons"
).orderBy(
    F.desc("fraud_risk_score")
).show(15, truncate=False)

### Investigation Rule Design Note

The investigation reasons generated in this section use manually defined thresholds for selected fraud indicators.

Examples include:

- Supplier-relative invoice amount ratios below 0.60 or above 1.50
- Invoice amount deviations greater than 5,000 from the supplier's 90-day average
- Supplier risk scores of 0.60 or higher
- Absolute invoice amount z-scores of 2 or greater

These thresholds are not directly learned from the Random Forest model. They are interpretable rule-based criteria used to convert important fraud indicators into concise investigation signals.

In a production RiskCheck AI system, these thresholds should be configurable and may be refined using business requirements, investigator feedback, historical alert outcomes, and operational capacity.

The generated investigation reasons should therefore be treated as contextual review indicators rather than exact explanations of the model's individual prediction.

## Validate the Explainability & Risk Interpretation Workflow

The Week 5 explainability workflow is validated before completion.

The final checks confirm that:

- All 32 modelling features are correctly mapped to Random Forest feature importance values.
- Global feature importance values sum to approximately 1.0.
- The strongest predictive signals are identified and ranked.
- Fraud probability scores are available for all 60,000 test invoices.
- Every scored invoice receives a fraud risk category.
- Investigator-facing outputs preserve the invoice identifier.
- Investigation reasons are generated from selected high-importance fraud indicators.
- The explainability outputs do not alter the original model predictions.
- Investigation reasons are treated as contextual risk signals rather than causal or prediction-level explanations.

### Final Validation

In [0]:
total_scored = investigation_reasons_df.count()

unique_invoices = (
    investigation_reasons_df
    .select("invoice_id")
    .distinct()
    .count()
)

missing_risk_scores = (
    investigation_reasons_df
    .filter(F.col("fraud_risk_score").isNull())
    .count()
)

missing_risk_categories = (
    investigation_reasons_df
    .filter(F.col("risk_category").isNull())
    .count()
)

missing_predictions = (
    investigation_reasons_df
    .filter(F.col("model_prediction").isNull())
    .count()
)

print("WEEK 5 FINAL VALIDATION")
print("-" * 50)

print(f"Total scored invoices:        {total_scored:,}")
print(f"Unique invoice identifiers:   {unique_invoices:,}")
print(f"Missing fraud risk scores:    {missing_risk_scores:,}")
print(f"Missing risk categories:      {missing_risk_categories:,}")
print(f"Missing model predictions:    {missing_predictions:,}")

print(
    "\nFeature importance total:",
    f"{sum(feature_importance_values):.6f}"
)

print(
    "All invoices preserved:",
    total_scored == 60000 and unique_invoices == 60000
)

print(
    "Risk outputs complete:",
    missing_risk_scores == 0
    and missing_risk_categories == 0
    and missing_predictions == 0
)

## Week 5 Conclusion

Week 5 focused on improving the interpretability of the RiskCheck AI Random Forest fraud detection model and transforming its predictions into more useful fraud risk information.

Global Random Forest feature importance was extracted for all 32 modelling features. The analysis showed that model importance is highly concentrated, with the four strongest predictors accounting for approximately 81% of total feature importance and the ten strongest predictors accounting for approximately 97%.

The most influential predictive signals included split-invoice behaviour, invoice amounts relative to supplier historical averages, supplier blacklist status, invoice amount characteristics, and supplier risk information.

Further analysis showed strong differences between legitimate and fraudulent invoices. Split-invoice behaviour was observed in approximately 0.37% of legitimate invoices compared with 62.61% of fraudulent invoices, while blacklisted supplier activity increased from approximately 0.12% among legitimate invoices to 22.77% among fraudulent invoices.

Fraud probability scores were extracted from the selected Random Forest model and analysed across the held-out test dataset. Fraudulent invoices received substantially higher average model risk scores than legitimate invoices.

Five interpretable fraud risk categories were then created:

- Low Risk
- Moderate Risk
- Elevated Risk
- High Risk
- Critical Risk

Observed fraud prevalence increased substantially across the risk categories, reaching approximately 97% to 98% within the High and Critical Risk groups.

An investigator-facing output was also developed, combining invoice identifiers, fraud risk scores, risk categories, model predictions, and selected high-importance fraud indicators. Rule-based investigation reasons were generated to provide concise contextual signals for fraud review.

These investigation reasons should be interpreted as supporting risk indicators rather than exact local explanations of individual model predictions.

Week 5 therefore establishes an explainability and fraud risk interpretation layer for RiskCheck AI, creating a foundation for operational risk scoring, alert prioritisation, and downstream application integration.

# Week 6: Operational Risk Scoring and Investigation Prioritisation

## Overview

Week 6 focuses on transforming the RiskCheck AI model outputs into an operational fraud-risk scoring and investigation prioritisation framework.

Week 5 established an explainability layer around the selected Random Forest model by analysing global feature importance, fraud probability scores, risk categories, and investigation-oriented risk signals.

This week extends those outputs into an operational structure that can support fraud investigation workflows. Model-generated fraud probabilities are converted into user-friendly risk scores, investigation priorities are defined, recommended review actions are assigned, and a consolidated scoring output is created for downstream use.

The objective is to bridge the gap between machine learning predictions and practical fraud investigation by providing clear, prioritised, and actionable invoice-level risk information.

## Week 6 Objectives

- Confirm the Week 5 investigator-facing risk outputs.
- Convert model-generated fraud probabilities into operational risk scores.
- Define investigation priority levels.
- Assign recommended review actions based on invoice risk.
- Build a consolidated RiskCheck AI operational scoring output.
- Analyse the resulting investigation queue and operational workload.
- Validate the consistency and completeness of operational risk outputs.
- Prepare the RiskCheck AI scoring layer for downstream application and deployment integration.

## Confirming Week 5 Risk-Scoring Outputs

Week 5 produced an investigator-facing dataset containing model-generated fraud risk scores, risk categories, predictions, and contextual investigation signals for the held-out test invoices.

Before developing the operational scoring framework, the Week 5 output is validated to confirm that the required invoice-level information remains available and complete.

This provides the foundation for converting model predictions into operational investigation priorities and recommended actions.

In [0]:

print("WEEK 6 — INPUT VALIDATION")
print("-" * 50)

total_rows = investigation_reasons_df.count()

unique_invoices = (
    investigation_reasons_df
    .select("invoice_id")
    .distinct()
    .count()
)

print(f"Available invoices:       {total_rows:,}")
print(f"Unique invoice IDs:       {unique_invoices:,}")

print("\nRequired Week 5 fields:")
print("-" * 50)

required_columns = [
    "invoice_id",
    "actual_label",
    "model_prediction",
    "fraud_risk_score",
    "risk_category",
    "investigation_reasons"
]

for column_name in required_columns:
    status = (
        "Available"
        if column_name in investigation_reasons_df.columns
        else "MISSING"
    )

    print(f"{column_name:<25} {status}")

print(
    "\nWeek 6 input ready:",
    total_rows == 60000
    and unique_invoices == 60000
    and all(
        column_name in investigation_reasons_df.columns
        for column_name in required_columns
    )
)

## Create the Operational RiskCheck Score

The Random Forest model produces a fraud risk score between 0 and 1. While this representation is appropriate for machine learning analysis, an operational fraud investigation system benefits from a more intuitive scoring scale.

The model-generated fraud risk score is therefore converted into a RiskCheck Score ranging from 0 to 100.

A higher RiskCheck Score represents a stronger model-generated fraud signal, while a lower score represents a weaker fraud signal.

The transformation does not modify the underlying Random Forest prediction. It provides a more interpretable representation of the existing model score for operational use.

The RiskCheck Score should be interpreted as an investigation-prioritisation measure rather than a definitive probability that fraud has occurred.

### Creating 0–100 score

In [0]:
operational_risk_df = (
    investigation_reasons_df
    .withColumn(
        "riskcheck_score",
        F.round(
            F.col("fraud_risk_score") * 100,
            2
        )
    )
)

print("Operational RiskCheck Score created successfully.")
print(f"Scored invoices: {operational_risk_df.count():,}")

operational_risk_df.select(
    "invoice_id",
    "fraud_risk_score",
    "riskcheck_score",
    "risk_category"
).orderBy(
    F.desc("riskcheck_score")
).show(10, truncate=False)

### Validating Score

In [0]:
riskcheck_score_validation = (
    operational_risk_df
    .agg(
        F.count("*").alias("total_invoices"),
        F.min("riskcheck_score").alias("minimum_score"),
        F.max("riskcheck_score").alias("maximum_score"),
        F.avg("riskcheck_score").alias("average_score"),
        F.sum(
            F.when(
                F.col("riskcheck_score").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_scores")
    )
)

riskcheck_score_validation.show(truncate=False)

## Define Investigation Priority Levels

The RiskCheck Score provides an intuitive numerical representation of model-generated fraud risk. However, fraud investigation teams also require a clear mechanism for determining which invoices should receive attention first.

Investigation priority levels are therefore assigned using the fraud risk categories established during Week 5.

The priority framework translates model risk into an operational queue:

- **Critical Risk → Priority 1 — Immediate**
- **High Risk → Priority 2 — High**
- **Elevated Risk → Priority 3 — Medium**
- **Moderate Risk → Priority 4 — Low**
- **Low Risk → Priority 5 — Routine**

Lower priority numbers represent greater investigative urgency.

These priority levels are designed to support workflow organisation and investigation prioritisation. They do not represent definitive determinations of fraudulent activity.

### Assigning Priorities

In [0]:
priority_df = (
    operational_risk_df
    .withColumn(
        "priority_level",
        F.when(
            F.col("risk_category") == "Critical Risk",
            "Priority 1 - Immediate"
        )
        .when(
            F.col("risk_category") == "High Risk",
            "Priority 2 - High"
        )
        .when(
            F.col("risk_category") == "Elevated Risk",
            "Priority 3 - Medium"
        )
        .when(
            F.col("risk_category") == "Moderate Risk",
            "Priority 4 - Low"
        )
        .otherwise("Priority 5 - Routine")
    )
    .withColumn(
        "priority_rank",
        F.when(F.col("risk_category") == "Critical Risk", 1)
         .when(F.col("risk_category") == "High Risk", 2)
         .when(F.col("risk_category") == "Elevated Risk", 3)
         .when(F.col("risk_category") == "Moderate Risk", 4)
         .otherwise(5)
    )
)

print("Investigation priority levels assigned successfully.")

priority_df.select(
    "invoice_id",
    "riskcheck_score",
    "risk_category",
    "priority_level"
).orderBy(
    "priority_rank",
    F.desc("riskcheck_score")
).show(15, truncate=False)

### Validating Priority Distribution

In [0]:
priority_summary = (
    priority_df
    .groupBy(
        "priority_rank",
        "priority_level"
    )
    .agg(
        F.count("*").alias("invoice_count"),

        F.sum(
            F.when(
                F.col("actual_label") == 1,
                1
            ).otherwise(0)
        ).alias("fraud_count")
    )
    .withColumn(
        "percentage_of_queue",
        F.round(
            F.col("invoice_count") /
            F.lit(60000) * 100,
            2
        )
    )
    .withColumn(
        "observed_fraud_rate",
        F.round(
            F.col("fraud_count") /
            F.col("invoice_count") * 100,
            2
        )
    )
    .orderBy("priority_rank")
)

priority_summary.show(truncate=False)

## Assign Recommended Investigation Actions

Investigation priority levels indicate the relative urgency of invoice review, but an operational fraud detection system should also communicate what action should be taken for each priority level.

Recommended actions are therefore assigned to each invoice based on its investigation priority.

The operational action framework is defined as follows:

- **Priority 1 — Immediate:** Immediate fraud investigation
- **Priority 2 — High:** High-priority manual review
- **Priority 3 — Medium:** Standard manual review
- **Priority 4 — Low:** Monitor and review if additional risk signals emerge
- **Priority 5 — Routine:** Routine processing with continued monitoring

These recommendations provide workflow guidance rather than automated final decisions. RiskCheck AI is designed to support fraud investigators and should not independently determine that an invoice is fraudulent or automatically reject a transaction.

### Assign recommended actions

In [0]:
action_df = (
    priority_df
    .withColumn(
        "recommended_action",
        F.when(
            F.col("priority_rank") == 1,
            "Immediate fraud investigation"
        )
        .when(
            F.col("priority_rank") == 2,
            "High-priority manual review"
        )
        .when(
            F.col("priority_rank") == 3,
            "Standard manual review"
        )
        .when(
            F.col("priority_rank") == 4,
            "Monitor and review if additional risk signals emerge"
        )
        .otherwise(
            "Routine processing with continued monitoring"
        )
    )
)

print("Recommended investigation actions assigned successfully.")

action_df.select(
    "invoice_id",
    "riskcheck_score",
    "risk_category",
    "priority_level",
    "recommended_action"
).orderBy(
    "priority_rank",
    F.desc("riskcheck_score")
).show(15, truncate=False)

### Validating Action Mapping

In [0]:
action_summary = (
    action_df
    .groupBy(
        "priority_rank",
        "priority_level",
        "recommended_action"
    )
    .agg(
        F.count("*").alias("invoice_count")
    )
    .withColumn(
        "percentage_of_invoices",
        F.round(
            F.col("invoice_count") /
            F.lit(60000) * 100,
            2
        )
    )
    .orderBy("priority_rank")
)

action_summary.show(truncate=False)

## Build the Consolidated RiskCheck AI Operational Output

The previous steps transformed the Random Forest fraud prediction into a structured operational decision-support framework.

A consolidated RiskCheck AI output is now created to bring together the most important information required by a fraud investigation workflow.

For each invoice, the operational output includes:

- Invoice identifier
- RiskCheck Score
- Fraud risk category
- Investigation priority
- Recommended investigation action
- Model prediction
- Investigation-oriented risk signals

During model development and evaluation, the known fraud label is also retained for validation purposes. In a production environment, this label would not be available when a new invoice is scored.

The consolidated dataset represents the primary operational output of the RiskCheck AI scoring pipeline and provides a structured foundation for downstream application and deployment integration.

### Creating Consolidated Output

In [0]:
riskcheck_operational_df = (
    action_df
    .select(
        "invoice_id",

        "riskcheck_score",
        "risk_category",

        "priority_rank",
        "priority_level",

        "recommended_action",

        F.col("model_prediction")
        .cast("int")
        .alias("fraud_prediction"),

        "investigation_reasons",

        # Retained for development/evaluation only
        "actual_label"
    )
)

print("RiskCheck AI operational output created successfully.")
print(f"Operational records: {riskcheck_operational_df.count():,}")

riskcheck_operational_df.orderBy(
    "priority_rank",
    F.desc("riskcheck_score")
).show(15, truncate=False)

### Validating Operational Dataset

In [0]:
operational_validation = (
    riskcheck_operational_df
    .agg(
        F.count("*").alias("total_records"),

        F.countDistinct(
            "invoice_id"
        ).alias("unique_invoices"),

        F.sum(
            F.when(
                F.col("riskcheck_score").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_scores"),

        F.sum(
            F.when(
                F.col("priority_level").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_priorities"),

        F.sum(
            F.when(
                F.col("recommended_action").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_actions")
    )
)

operational_validation.show(truncate=False)

## Analyse the Investigation Queue and Operational Workload

The operational scoring framework assigns every invoice to an investigation priority, but the practical usefulness of the system also depends on the volume of invoices requiring investigator attention.

The investigation queue is therefore analysed to determine how much of the total invoice population falls into each operational priority level.

Particular attention is given to the higher-priority groups because these invoices would consume the greatest investigative resources.

The analysis also measures how many known fraudulent invoices are captured within progressively larger portions of the priority queue. This provides an indication of how effectively RiskCheck AI concentrates fraud cases into the invoices receiving the greatest investigative attention.

The known fraud labels are used only for retrospective model evaluation. In a production environment, these labels would not be available when the investigation queue is created.

### Analyse Review Workload

In [0]:
workload_summary = (
    riskcheck_operational_df
    .groupBy(
        "priority_rank",
        "priority_level"
    )
    .agg(
        F.count("*").alias("invoice_count"),

        F.sum(
            F.when(
                F.col("actual_label") == 1,
                1
            ).otherwise(0)
        ).alias("fraud_count")
    )
    .withColumn(
        "queue_percentage",
        F.round(
            F.col("invoice_count") /
            F.lit(60000) * 100,
            2
        )
    )
    .withColumn(
        "fraud_capture_percentage",
        F.round(
            F.col("fraud_count") /
            F.lit(13510) * 100,
            2
        )
    )
    .orderBy("priority_rank")
)

workload_summary.show(truncate=False)

### Cumulative Queue Effectiveness

In [0]:
workload_rows = (
    workload_summary
    .orderBy("priority_rank")
    .collect()
)

cumulative_results = []

cumulative_invoices = 0
cumulative_fraud = 0

for row in workload_rows:

    cumulative_invoices += row["invoice_count"]
    cumulative_fraud += row["fraud_count"]

    cumulative_results.append(
        (
            row["priority_rank"],
            row["priority_level"],
            cumulative_invoices,
            round(cumulative_invoices / 60000 * 100, 2),
            cumulative_fraud,
            round(cumulative_fraud / 13510 * 100, 2)
        )
    )

cumulative_workload = spark.createDataFrame(
    cumulative_results,
    [
        "priority_rank",
        "priority_level",
        "cumulative_invoices",
        "cumulative_queue_percentage",
        "cumulative_fraud",
        "cumulative_fraud_capture"
    ]
)

cumulative_workload.show(truncate=False)

## Validate Operational Consistency and Exceptions

Before the RiskCheck AI operational scoring layer is considered complete, the consistency of the generated scores, risk categories, priorities, and recommended actions must be validated.

The validation checks confirm that:

- Every invoice has a RiskCheck Score between 0 and 100.
- Risk categories remain consistent with the defined score boundaries.
- Each risk category maps to the correct investigation priority.
- Every priority level maps to the expected recommended action.
- No duplicate invoice identifiers exist.
- No required operational fields are missing.

These checks help ensure that the operational decision-support layer behaves consistently before it is prepared for downstream application integration.

### Checking Score/Category Consistency

In [0]:
score_category_errors = (
    riskcheck_operational_df
    .filter(
        (F.col("riskcheck_score") < 0) |
        (F.col("riskcheck_score") > 100) |

        (
            (F.col("riskcheck_score") < 20) &
            (F.col("risk_category") != "Low Risk")
        ) |

        (
            (F.col("riskcheck_score") >= 20) &
            (F.col("riskcheck_score") < 40) &
            (F.col("risk_category") != "Moderate Risk")
        ) |

        (
            (F.col("riskcheck_score") >= 40) &
            (F.col("riskcheck_score") < 60) &
            (F.col("risk_category") != "Elevated Risk")
        ) |

        (
            (F.col("riskcheck_score") >= 60) &
            (F.col("riskcheck_score") < 80) &
            (F.col("risk_category") != "High Risk")
        ) |

        (
            (F.col("riskcheck_score") >= 80) &
            (F.col("risk_category") != "Critical Risk")
        )
    )
    .count()
)

print(
    "Score/category inconsistencies:",
    score_category_errors
)

### Check Category/Priority Consistency

In [0]:
priority_mapping_errors = (
    riskcheck_operational_df
    .filter(
        (
            (F.col("risk_category") == "Critical Risk") &
            (F.col("priority_rank") != 1)
        ) |
        (
            (F.col("risk_category") == "High Risk") &
            (F.col("priority_rank") != 2)
        ) |
        (
            (F.col("risk_category") == "Elevated Risk") &
            (F.col("priority_rank") != 3)
        ) |
        (
            (F.col("risk_category") == "Moderate Risk") &
            (F.col("priority_rank") != 4)
        ) |
        (
            (F.col("risk_category") == "Low Risk") &
            (F.col("priority_rank") != 5)
        )
    )
    .count()
)

print(
    "Category/priority inconsistencies:",
    priority_mapping_errors
)

### Checking Action Mapping & Completeness

In [0]:
action_mapping_errors = (
    riskcheck_operational_df
    .filter(
        (
            (F.col("priority_rank") == 1) &
            (F.col("recommended_action") !=
             "Immediate fraud investigation")
        ) |
        (
            (F.col("priority_rank") == 2) &
            (F.col("recommended_action") !=
             "High-priority manual review")
        ) |
        (
            (F.col("priority_rank") == 3) &
            (F.col("recommended_action") !=
             "Standard manual review")
        ) |
        (
            (F.col("priority_rank") == 4) &
            (F.col("recommended_action") !=
             "Monitor and review if additional risk signals emerge")
        ) |
        (
            (F.col("priority_rank") == 5) &
            (F.col("recommended_action") !=
             "Routine processing with continued monitoring")
        )
    )
    .count()
)

duplicate_invoices = (
    riskcheck_operational_df.count()
    -
    riskcheck_operational_df
    .select("invoice_id")
    .distinct()
    .count()
)

missing_required_fields = (
    riskcheck_operational_df
    .filter(
        F.col("invoice_id").isNull() |
        F.col("riskcheck_score").isNull() |
        F.col("risk_category").isNull() |
        F.col("priority_level").isNull() |
        F.col("recommended_action").isNull() |
        F.col("fraud_prediction").isNull()
    )
    .count()
)

print("OPERATIONAL CONSISTENCY CHECK")
print("-" * 50)

print(f"Score/category errors:     {score_category_errors}")
print(f"Category/priority errors:  {priority_mapping_errors}")
print(f"Priority/action errors:    {action_mapping_errors}")
print(f"Duplicate invoices:        {duplicate_invoices}")
print(f"Missing required fields:   {missing_required_fields}")

print(
    "\nOperational rules consistent:",
    score_category_errors == 0
    and priority_mapping_errors == 0
    and action_mapping_errors == 0
    and duplicate_invoices == 0
    and missing_required_fields == 0
)

## Final Validation of the Operational Risk Scoring Framework

The final Week 6 validation confirms that the RiskCheck AI operational scoring framework is complete, internally consistent, and ready for downstream integration.

The validation verifies that:

- All 60,000 test invoices are preserved.
- Every invoice has a unique identifier.
- Every invoice has a valid RiskCheck Score.
- Every invoice has an assigned fraud risk category.
- Every invoice has an investigation priority.
- Every invoice has a recommended operational action.
- Model predictions are retained within the operational output.
- Investigation-oriented risk signals remain available.
- No inconsistencies exist between scores, risk categories, priorities, and recommended actions.
- The operational framework successfully concentrates a large proportion of known fraud cases within the highest-priority investigation queues.

The known fraud label remains available only for retrospective evaluation and would not form part of the production-facing scoring output.

### Final Validation

In [0]:
total_records = riskcheck_operational_df.count()

unique_invoices = (
    riskcheck_operational_df
    .select("invoice_id")
    .distinct()
    .count()
)

missing_scores = (
    riskcheck_operational_df
    .filter(F.col("riskcheck_score").isNull())
    .count()
)

missing_categories = (
    riskcheck_operational_df
    .filter(F.col("risk_category").isNull())
    .count()
)

missing_priorities = (
    riskcheck_operational_df
    .filter(F.col("priority_level").isNull())
    .count()
)

missing_actions = (
    riskcheck_operational_df
    .filter(F.col("recommended_action").isNull())
    .count()
)

missing_predictions = (
    riskcheck_operational_df
    .filter(F.col("fraud_prediction").isNull())
    .count()
)

print("WEEK 6 FINAL VALIDATION")
print("-" * 50)

print(f"Total operational records:  {total_records:,}")
print(f"Unique invoice identifiers: {unique_invoices:,}")
print(f"Missing RiskCheck Scores:   {missing_scores:,}")
print(f"Missing risk categories:    {missing_categories:,}")
print(f"Missing priorities:         {missing_priorities:,}")
print(f"Missing actions:            {missing_actions:,}")
print(f"Missing predictions:        {missing_predictions:,}")

print(
    "\nAll records preserved:",
    total_records == 60000
    and unique_invoices == 60000
)

print(
    "Operational outputs complete:",
    missing_scores == 0
    and missing_categories == 0
    and missing_priorities == 0
    and missing_actions == 0
    and missing_predictions == 0
)

print(
    "Operational rules consistent:",
    score_category_errors == 0
    and priority_mapping_errors == 0
    and action_mapping_errors == 0
)

## Week 6 Conclusion

Week 6 transformed the RiskCheck AI model outputs into an operational fraud-risk scoring and investigation prioritisation framework.

The Random Forest fraud risk score was converted into an intuitive RiskCheck Score ranging from 0 to 100 while preserving the underlying model-generated risk information.

The existing fraud risk categories were translated into five operational investigation priorities:

- Priority 1 — Immediate
- Priority 2 — High
- Priority 3 — Medium
- Priority 4 — Low
- Priority 5 — Routine

Each priority level was also mapped to a recommended investigation action, creating a structured workflow from model prediction to operational response.

A consolidated RiskCheck AI operational dataset was created containing invoice identifiers, RiskCheck Scores, risk categories, investigation priorities, recommended actions, fraud predictions, and investigation-oriented risk signals.

Operational workload analysis demonstrated that the highest-priority queues substantially concentrate known fraudulent invoices. Priority 1 and Priority 2 together contain 12,392 invoices, representing 20.65% of the test population, while capturing 12,125 of the 13,510 known fraud cases, equivalent to 89.75% of observed fraud.

Consistency checks confirmed that there are no conflicts between RiskCheck Scores, risk categories, investigation priorities, or recommended actions. No duplicate invoices or missing required operational fields were identified.

The operational framework is designed as a decision-support mechanism rather than an automated fraud determination system. Investigation priorities and recommended actions support human review and should not independently determine whether an invoice is fraudulent.

Week 6 therefore establishes the operational scoring and investigation-prioritisation layer of RiskCheck AI, providing a structured foundation for downstream application, API, and deployment integration.

# Week 7: Deployment Preparation and Application Integration

## Overview

Week 7 focuses on preparing RiskCheck AI for integration with downstream applications and deployment workflows.

The previous weeks established the complete analytical and operational fraud detection pipeline, including feature engineering, model development, model evaluation, explainability, fraud risk scoring, investigation prioritisation, and recommended investigation actions.

Week 7 moves beyond analytical model development and defines how RiskCheck AI can operate as a reusable scoring component.

The operational scoring output is separated from evaluation-only information, a production-facing response structure is created, and the inputs and outputs required by an application integration layer are defined.

The week also introduces an API-style scoring workflow to demonstrate how invoice information could be submitted to RiskCheck AI and returned as structured fraud risk information.

## Week 7 Objectives

- Confirm the finalized Week 6 operational scoring output.
- Separate production-facing information from model evaluation fields.
- Create a clean production scoring response structure.
- Define the RiskCheck AI scoring request and response contract.
- Prepare the model and scoring workflow for deployment.
- Build an API-style scoring function.
- Validate integration and error-handling scenarios.
- Confirm deployment readiness for the final project stage.

## Confirming Week 6 Operational Scoring Output

Week 6 produced the consolidated operational RiskCheck AI dataset containing invoice-level fraud scores, risk categories, investigation priorities, recommended actions, model predictions, and investigation-oriented risk signals.

Before preparing the deployment and application integration layer, the finalized Week 6 output is validated.

This ensures that the deployment workflow begins from a complete and internally consistent operational scoring dataset.

In [0]:
print("INPUT VALIDATION")
print("-" * 50)

total_records = riskcheck_operational_df.count()

unique_invoices = (
    riskcheck_operational_df
    .select("invoice_id")
    .distinct()
    .count()
)

required_columns = [
    "invoice_id",
    "riskcheck_score",
    "risk_category",
    "priority_rank",
    "priority_level",
    "recommended_action",
    "fraud_prediction",
    "investigation_reasons",
    "actual_label"
]

available_columns = riskcheck_operational_df.columns

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in available_columns
]

print(f"Available records:       {total_records:,}")
print(f"Unique invoice IDs:      {unique_invoices:,}")

print("\nRequired Week 6 fields:")
print("-" * 50)

for column_name in required_columns:

    status = (
        "Available"
        if column_name in available_columns
        else "MISSING"
    )

    print(f"{column_name:<25} {status}")

print(f"\nMissing required fields: {len(missing_columns)}")

print(
    "Week 7 input ready:",
    total_records == 60000
    and unique_invoices == 60000
    and len(missing_columns) == 0
)

## Defining Production-Facing Scoring Schema

The Week 6 operational dataset contains both production-relevant information and development-only evaluation fields.

For deployment and downstream application integration, the production-facing output should contain only the information required by an operational fraud review workflow.

The known fraud label (`actual_label`) is retained during model development for retrospective evaluation, but it would not be available when a new invoice is scored in production.

The production-facing RiskCheck AI response therefore includes:

- Invoice identifier
- RiskCheck Score
- Fraud risk category
- Investigation priority
- Recommended investigation action
- Model fraud prediction
- Investigation-oriented risk signals

Evaluation-only fields are excluded from the production schema.

### Creating Production-facing Output

In [0]:
riskcheck_production_df = (
    riskcheck_operational_df
    .select(
        "invoice_id",
        "riskcheck_score",
        "risk_category",
        "priority_rank",
        "priority_level",
        "recommended_action",
        "fraud_prediction",
        "investigation_reasons"
    )
)

print("Production-facing RiskCheck AI output created successfully.")
print(f"Production records: {riskcheck_production_df.count():,}")
print(f"Production columns: {len(riskcheck_production_df.columns)}")

print("\nProduction schema:")
riskcheck_production_df.printSchema()

### Validate that Evaluation Fields are Removed

In [0]:
production_columns = riskcheck_production_df.columns

evaluation_only_fields = [
    "actual_label"
]

unexpected_evaluation_fields = [
    column_name
    for column_name in evaluation_only_fields
    if column_name in production_columns
]

print("PRODUCTION SCHEMA VALIDATION")
print("-" * 50)

print(f"Production field count: {len(production_columns)}")
print(f"Evaluation fields present: {len(unexpected_evaluation_fields)}")

print(
    "Production schema ready:",
    len(unexpected_evaluation_fields) == 0
)

## Defining RiskCheck AI Request and Response Contract

For application integration, RiskCheck AI requires a clear contract describing the information submitted for scoring and the information returned by the scoring service.

The request contract represents the invoice and fraud-related feature information required by the trained model.

The response contract represents the operational fraud-risk information returned after scoring.

A structured request and response contract helps ensure that downstream applications interact with RiskCheck AI consistently and provides a foundation for future API implementation.

The production response includes the RiskCheck Score, risk category, investigation priority, recommended action, fraud prediction, and investigation-oriented risk signals.

### Defining Response Contract

In [0]:
riskcheck_response_fields = [
    ("invoice_id", "string", "Unique invoice identifier"),
    ("riskcheck_score", "double", "Operational fraud risk score from 0 to 100"),
    ("risk_category", "string", "Low, Moderate, Elevated, High, or Critical Risk"),
    ("priority_rank", "integer", "Investigation priority from 1 to 5"),
    ("priority_level", "string", "Operational investigation priority"),
    ("recommended_action", "string", "Recommended review action"),
    ("fraud_prediction", "integer", "Binary model prediction: 0 = legitimate, 1 = fraud"),
    ("investigation_reasons", "array<string>", "Contextual fraud investigation signals")
]

print("RISKCHECK AI — RESPONSE CONTRACT")
print("-" * 95)

for field_name, field_type, description in riskcheck_response_fields:
    print(
        f"{field_name:<25}"
        f"{field_type:<18}"
        f"{description}"
    )

### Validating Response Contract Against the DataFrame

In [0]:
expected_response_columns = [
    field_name
    for field_name, _, _
    in riskcheck_response_fields
]

actual_response_columns = riskcheck_production_df.columns

missing_response_fields = [
    column_name
    for column_name in expected_response_columns
    if column_name not in actual_response_columns
]

unexpected_response_fields = [
    column_name
    for column_name in actual_response_columns
    if column_name not in expected_response_columns
]

print("RESPONSE CONTRACT VALIDATION")
print("-" * 50)

print(f"Expected fields:   {len(expected_response_columns)}")
print(f"Actual fields:     {len(actual_response_columns)}")
print(f"Missing fields:    {len(missing_response_fields)}")
print(f"Unexpected fields: {len(unexpected_response_fields)}")

print(
    "\nResponse contract valid:",
    len(missing_response_fields) == 0
    and len(unexpected_response_fields) == 0
)

## Defining RiskCheck AI Input Request Schema

The response contract defines what RiskCheck AI returns after scoring. The next requirement is to define the information that must be supplied when a new invoice is submitted for fraud-risk assessment.

The input request should contain the predictor variables required by the trained Random Forest model.

The request contract therefore reflects the same 32 modelling features used during model training.

Invoice identifiers may also be supplied for traceability, but the identifier itself is not used as a predictive feature.

Defining the request schema helps ensure that downstream applications submit complete and correctly structured data to the RiskCheck AI scoring service.

### Defining Request Fields

In [0]:
riskcheck_request_fields = [
    ("invoice_id", "string", "Unique invoice identifier used for traceability")
]

for feature_name in model_feature_columns:
    riskcheck_request_fields.append(
        (
            feature_name,
            "numeric",
            "Model input feature"
        )
    )

print("RISKCHECK AI — REQUEST CONTRACT")
print("-" * 95)

for field_name, field_type, description in riskcheck_request_fields:
    print(
        f"{field_name:<40}"
        f"{field_type:<15}"
        f"{description}"
    )

print(
    f"\nTotal request fields: {len(riskcheck_request_fields)}"
)

### Validating Request Feature Coverage

In [0]:
request_field_names = [
    field_name
    for field_name, _, _
    in riskcheck_request_fields
]

missing_model_features = [
    feature_name
    for feature_name in model_feature_columns
    if feature_name not in request_field_names
]

extra_model_features = [
    field_name
    for field_name in request_field_names
    if field_name != "invoice_id"
    and field_name not in model_feature_columns
]

print("REQUEST CONTRACT VALIDATION")
print("-" * 50)

print(f"Model predictors:        {len(model_feature_columns)}")
print(f"Request fields:          {len(request_field_names)}")
print(f"Missing model features:  {len(missing_model_features)}")
print(f"Unexpected model fields: {len(extra_model_features)}")

print(
    "\nRequest contract valid:",
    len(missing_model_features) == 0
    and len(extra_model_features) == 0
)

## Build an API-Style RiskCheck AI Scoring Function

With the production request and response contracts defined, the next step is to simulate how an external application could submit a new invoice to RiskCheck AI for scoring.

An API-style scoring function is created to:

- Accept a single invoice record.
- Validate that all required model features are present.
- Convert the input into the feature structure expected by the trained Random Forest model.
- Generate the model fraud prediction and fraud risk score.
- Assign the corresponding risk category.
- Assign the operational investigation priority.
- Assign the recommended review action.
- Return a structured response matching the production response contract.

This function demonstrates the scoring workflow that could later be exposed through an API or application service.

### Building Scoring Function

In [0]:
from pyspark.sql import Row
from pyspark.ml.functions import vector_to_array

def score_riskcheck_invoice(invoice_record):

    required_fields = [
        "invoice_id"
    ] + model_feature_columns

    missing_fields = [
        field
        for field in required_fields
        if field not in invoice_record
    ]

    if missing_fields:
        return {
            "status": "error",
            "message": "Missing required fields",
            "missing_fields": missing_fields
        }

    invoice_df = spark.createDataFrame(
        [Row(**invoice_record)]
    )

    assembled_invoice_df = assembler.transform(
        invoice_df
    )

    prediction_df = (
        rf_model
        .transform(assembled_invoice_df)
        .withColumn(
            "fraud_probability",
            vector_to_array(
                F.col("probability")
            )[1]
        )
    )

    prediction_row = prediction_df.select(
        "invoice_id",
        "prediction",
        "fraud_probability"
    ).first()

    fraud_probability = float(
        prediction_row["fraud_probability"]
    )

    riskcheck_score = round(
        fraud_probability * 100,
        2
    )

    fraud_prediction = int(
        prediction_row["prediction"]
    )

    if riskcheck_score < 20:
        risk_category = "Low Risk"
        priority_rank = 5
        priority_level = "Priority 5 - Routine"
        recommended_action = (
            "Routine processing with continued monitoring"
        )

    elif riskcheck_score < 40:
        risk_category = "Moderate Risk"
        priority_rank = 4
        priority_level = "Priority 4 - Low"
        recommended_action = (
            "Monitor and review if additional risk signals emerge"
        )

    elif riskcheck_score < 60:
        risk_category = "Elevated Risk"
        priority_rank = 3
        priority_level = "Priority 3 - Medium"
        recommended_action = (
            "Standard manual review"
        )

    elif riskcheck_score < 80:
        risk_category = "High Risk"
        priority_rank = 2
        priority_level = "Priority 2 - High"
        recommended_action = (
            "High-priority manual review"
        )

    else:
        risk_category = "Critical Risk"
        priority_rank = 1
        priority_level = "Priority 1 - Immediate"
        recommended_action = (
            "Immediate fraud investigation"
        )

    reasons = []

    if invoice_record["split_invoice_flag"] == 1:
        reasons.append(
            "Split-invoice behaviour detected"
        )

    if invoice_record["blacklisted_flag"] == 1:
        reasons.append(
            "Supplier is blacklisted"
        )

    ratio_90d = invoice_record[
        "amount_to_supplier_90d_avg_ratio"
    ]

    if ratio_90d < 0.60 or ratio_90d > 1.50:
        reasons.append(
            "Invoice amount differs substantially from recent supplier pattern"
        )

    if invoice_record[
        "amount_diff_supplier_90d_avg"
    ] > 5000:
        reasons.append(
            "Large deviation from supplier 90-day average"
        )

    if invoice_record[
        "supplier_risk_score"
    ] >= 0.60:
        reasons.append(
            "Elevated supplier risk score"
        )

    if abs(
        invoice_record[
            "invoice_amount_zscore"
        ]
    ) >= 2:
        reasons.append(
            "Unusual invoice amount relative to historical distribution"
        )

    return {
        "status": "success",
        "invoice_id": prediction_row["invoice_id"],
        "riskcheck_score": riskcheck_score,
        "risk_category": risk_category,
        "priority_rank": priority_rank,
        "priority_level": priority_level,
        "recommended_action": recommended_action,
        "fraud_prediction": fraud_prediction,
        "investigation_reasons": reasons
    }


print("RiskCheck AI scoring function created successfully.")

### Preparing a Real Test Request

In [0]:


sample_invoice_row = (
    test_explain_source_df
    .select(
        "invoice_id",
        *model_feature_columns
    )
    .limit(1)
    .collect()[0]
)

sample_invoice_request = sample_invoice_row.asDict()

print("Sample request prepared successfully.")
print(f"Invoice ID: {sample_invoice_request['invoice_id']}")
print(f"Request fields: {len(sample_invoice_request)}")
print(f"Expected request fields: {len(model_feature_columns) + 1}")

### Scoring Sample Invoice

In [0]:
sample_response = score_riskcheck_invoice(
    sample_invoice_request
)

print("RISKCHECK AI — SAMPLE API RESPONSE")
print("-" * 60)

for key, value in sample_response.items():
    print(f"{key:<25}: {value}")

## Validating API-Style Scoring Workflow

The API-style scoring function should reproduce the same operational fraud-risk outputs generated by the existing RiskCheck AI scoring pipeline.

A known invoice from the test dataset is therefore scored through the new function and compared with its existing production-facing record.

The validation compares:

- Invoice identifier
- RiskCheck Score
- Risk category
- Investigation priority
- Recommended action
- Fraud prediction
- Investigation-oriented risk signals

This confirms that the application-facing scoring workflow remains consistent with the previously validated operational scoring pipeline.

### Retrieving Existing Production Records

In [0]:
existing_production_row = (
    riskcheck_production_df
    .filter(
        F.col("invoice_id")
        == sample_invoice_request["invoice_id"]
    )
    .first()
)

existing_production_response = (
    existing_production_row.asDict()
)

print("Existing production record retrieved.")
print(
    f"Invoice ID: "
    f"{existing_production_response['invoice_id']}"
)

### Comparing API vs Production Output

In [0]:
score_difference = abs(
    sample_response["riskcheck_score"]
    - existing_production_response["riskcheck_score"]
)

validation_checks = {
    "invoice_id": (
        sample_response["invoice_id"]
        == existing_production_response["invoice_id"]
    ),

    "riskcheck_score": (
        score_difference <= 0.01
    ),

    "risk_category": (
        sample_response["risk_category"]
        == existing_production_response["risk_category"]
    ),

    "priority_rank": (
        sample_response["priority_rank"]
        == existing_production_response["priority_rank"]
    ),

    "priority_level": (
        sample_response["priority_level"]
        == existing_production_response["priority_level"]
    ),

    "recommended_action": (
        sample_response["recommended_action"]
        == existing_production_response["recommended_action"]
    ),

    "fraud_prediction": (
        sample_response["fraud_prediction"]
        == existing_production_response["fraud_prediction"]
    ),

    "investigation_reasons": (
        sample_response["investigation_reasons"]
        == existing_production_response["investigation_reasons"]
    )
}

print("API SCORING CONSISTENCY VALIDATION")
print("-" * 60)

for field_name, passed in validation_checks.items():
    print(
        f"{field_name:<25}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(
    "\nScore difference:",
    round(score_difference, 4)
)

print(
    "\nAPI scoring consistent:",
    all(validation_checks.values())
)

## Validate Integration & Error-Handling Scenarios

A production scoring service must handle invalid or incomplete requests in a predictable manner.

The API-style RiskCheck AI function is therefore tested using several integration scenarios, including:

- A valid scoring request
- A request with a missing required model feature
- A request with a missing invoice identifier

The objective is to confirm that valid requests return a structured scoring response while incomplete requests return clear validation errors rather than attempting to generate a prediction.

These checks provide an initial demonstration of application-level input validation before a formal API or production service is implemented.

### Testing Missing Model Feature

In [0]:
missing_feature_request = sample_invoice_request.copy()

missing_feature_request.pop(
    "supplier_risk_score"
)

missing_feature_response = score_riskcheck_invoice(
    missing_feature_request
)

print("MISSING FEATURE TEST")
print("-" * 60)

for key, value in missing_feature_response.items():
    print(f"{key:<20}: {value}")

### Testing Missing Invoice ID

In [0]:
missing_id_request = sample_invoice_request.copy()

missing_id_request.pop(
    "invoice_id"
)

missing_id_response = score_riskcheck_invoice(
    missing_id_request
)

print("MISSING INVOICE ID TEST")
print("-" * 60)

for key, value in missing_id_response.items():
    print(f"{key:<20}: {value}")

### Validating Three Integration Scenarios

In [0]:
integration_validation = {
    "valid_request": (
        sample_response.get("status")
        == "success"
    ),

    "missing_feature_request": (
        missing_feature_response.get("status")
        == "error"
        and "supplier_risk_score"
        in missing_feature_response.get(
            "missing_fields",
            []
        )
    ),

    "missing_invoice_id": (
        missing_id_response.get("status")
        == "error"
        and "invoice_id"
        in missing_id_response.get(
            "missing_fields",
            []
        )
    )
}

print("INTEGRATION SCENARIO VALIDATION")
print("-" * 60)

for test_name, passed in integration_validation.items():

    print(
        f"{test_name:<30}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(
    "\nIntegration scenarios valid:",
    all(integration_validation.values())
)

## Final Validation of the Deployment and Integration Layer

The final Week 7 validation confirms that the RiskCheck AI deployment and application integration layer is complete and internally consistent.

The validation verifies that:

- The Week 6 operational scoring output is available and complete.
- The production-facing schema excludes development-only evaluation fields.
- The production response contract matches the operational scoring output.
- The request contract contains all 32 model predictors together with the invoice identifier.
- The API-style scoring function produces results consistent with the previously validated operational pipeline.
- Valid scoring requests return structured production responses.
- Requests with missing required fields return clear validation errors.
- The application-facing workflow preserves the same RiskCheck Score, risk category, priority, recommended action, prediction, and investigation signals as the existing production dataset.

These checks demonstrate that the RiskCheck AI scoring workflow is ready for downstream application or API implementation.

In [0]:
production_record_count = riskcheck_production_df.count()

production_unique_invoices = (
    riskcheck_production_df
    .select("invoice_id")
    .distinct()
    .count()
)

response_contract_valid = (
    len(missing_response_fields) == 0
    and len(unexpected_response_fields) == 0
)

request_contract_valid = (
    len(missing_model_features) == 0
    and len(extra_model_features) == 0
)

api_consistency_valid = all(
    validation_checks.values()
)

integration_scenarios_valid = all(
    integration_validation.values()
)

print("FINAL VALIDATION")
print("-" * 60)

print(
    f"Production records:           "
    f"{production_record_count:,}"
)

print(
    f"Unique invoice identifiers:   "
    f"{production_unique_invoices:,}"
)

print(
    f"Production response fields:   "
    f"{len(riskcheck_production_df.columns)}"
)

print(
    f"Model request predictors:     "
    f"{len(model_feature_columns)}"
)

print(
    "\nResponse contract valid:",
    response_contract_valid
)

print(
    "Request contract valid:",
    request_contract_valid
)

print(
    "API scoring consistent:",
    api_consistency_valid
)

print(
    "Integration scenarios valid:",
    integration_scenarios_valid
)

print(
    "\nDeployment layer ready:",
    production_record_count == 60000
    and production_unique_invoices == 60000
    and response_contract_valid
    and request_contract_valid
    and api_consistency_valid
    and integration_scenarios_valid
)

## Week 7 Conclusion

Week 7 focused on preparing RiskCheck AI for downstream deployment and application integration.

The Week 6 operational scoring output was first validated and then separated into a production-facing schema that excludes development-only evaluation fields such as the known fraud label.

A production response contract was defined containing eight operational fields: invoice identifier, RiskCheck Score, risk category, investigation priority, recommended action, fraud prediction, and investigation-oriented risk signals.

The request contract was also defined using the invoice identifier together with the 32 predictor variables required by the trained Random Forest model.

An API-style scoring function was developed to simulate how an external application could submit invoice information to RiskCheck AI and receive a structured fraud-risk response.

The scoring function successfully reproduced the previously validated operational output for a known test invoice, including the RiskCheck Score, risk category, priority, recommended action, fraud prediction, and investigation reasons.

Basic application-level validation was also implemented. Valid requests returned structured scoring responses, while requests with missing required model features or missing invoice identifiers returned clear validation errors.

Week 7 therefore establishes the production-facing schema, request and response contracts, and application-level scoring workflow required for future API or application deployment.

The RiskCheck AI project is now ready for the final stage, which will focus on end-to-end validation, documentation, and final project presentation.

# Week 8: Final Validation and Project Readiness

## Overview

Week 8 represents the final stage of the RiskCheck AI development project.

The previous weeks established the complete fraud detection workflow, including data preparation, feature engineering, machine learning model development, model evaluation, explainability, operational risk scoring, investigation prioritisation, and application integration.

Week 8 brings these components together to perform final end-to-end validation and document the completed RiskCheck AI solution.

The final stage focuses on confirming that the analytical, machine learning, operational, and application-facing components remain consistent and ready for demonstration.

## Week 8 Objectives

- Confirm the finalized Week 7 production-facing outputs.
- Validate the completed RiskCheck AI workflow end-to-end.
- Summarize model and operational performance.
- Document the RiskCheck AI system architecture and workflow.
- Document project limitations, assumptions, and operational considerations.
- Define potential future improvements and a production roadmap.
- Produce the final project conclusion and presentation-ready summary.

## Confirm the Week 7 Production & Integration Outputs

Week 7 prepared RiskCheck AI for downstream application integration by creating the production-facing scoring schema, defining request and response contracts, implementing an API-style scoring workflow, and validating integration scenarios.

Before performing final project validation, the Week 7 outputs are confirmed to ensure that the final stage begins from a complete and deployment-ready scoring layer.

The validation confirms the availability and completeness of the production-facing dataset and the core integration components required by RiskCheck AI.

In [0]:
print("INPUT VALIDATION")
print("-" * 60)

production_records = riskcheck_production_df.count()

unique_invoices = (
    riskcheck_production_df
    .select("invoice_id")
    .distinct()
    .count()
)

required_production_fields = [
    "invoice_id",
    "riskcheck_score",
    "risk_category",
    "priority_rank",
    "priority_level",
    "recommended_action",
    "fraud_prediction",
    "investigation_reasons"
]

missing_production_fields = [
    field
    for field in required_production_fields
    if field not in riskcheck_production_df.columns
]

integration_components = {
    "scoring_function": callable(
        globals().get("score_riskcheck_invoice")
    ),

    "request_contract": (
        "riskcheck_request_fields" in globals()
    ),

    "response_contract": (
        "riskcheck_response_fields" in globals()
    ),

    "model_features": (
        "model_feature_columns" in globals()
        and len(model_feature_columns) == 32
    )
}

print(f"Production records:         {production_records:,}")
print(f"Unique invoice IDs:        {unique_invoices:,}")
print(
    f"Production fields:         "
    f"{len(riskcheck_production_df.columns)}"
)

print("\nRequired production fields:")
print("-" * 60)

for field in required_production_fields:

    status = (
        "Available"
        if field in riskcheck_production_df.columns
        else "MISSING"
    )

    print(f"{field:<25} {status}")

print("\nIntegration components:")
print("-" * 60)

for component, available in integration_components.items():

    print(
        f"{component:<25} "
        f"{'Available' if available else 'MISSING'}"
    )

week8_input_ready = (
    production_records == 60000
    and unique_invoices == 60000
    and len(missing_production_fields) == 0
    and all(integration_components.values())
)

print(
    "\nWeek 8 input ready:",
    week8_input_ready
)

## Validate the Complete RiskCheck AI Workflow

The final RiskCheck AI solution now includes data preparation, feature engineering, model training, model evaluation, explainability, operational risk scoring, investigation prioritisation, and application-facing integration.

An end-to-end validation is performed to confirm that the major components of the completed workflow remain available and consistent.

The validation checks that:

- The selected Random Forest model is available.
- The 32 modelling features remain defined.
- The production-facing dataset contains 60,000 scored invoices.
- The RiskCheck AI scoring function is available.
- The production request and response contracts remain defined.
- The operational risk categories, priorities, and recommended actions are available.
- A sample API-style request can be scored successfully.
- The API-style result remains consistent with the validated production output.

This validation provides a final check that the major analytical, modelling, operational, and integration components work together as a complete RiskCheck AI solution.

### End-to-end Validation

In [0]:
end_to_end_checks = {
    "random_forest_model": (
        "rf_model" in globals()
    ),

    "model_features": (
        "model_feature_columns" in globals()
        and len(model_feature_columns) == 32
    ),

    "production_dataset": (
        "riskcheck_production_df" in globals()
        and riskcheck_production_df.count() == 60000
    ),

    "scoring_function": callable(
        globals().get("score_riskcheck_invoice")
    ),

    "request_contract": (
        "riskcheck_request_fields" in globals()
    ),

    "response_contract": (
        "riskcheck_response_fields" in globals()
    ),

    "sample_request": (
        "sample_invoice_request" in globals()
    ),

    "sample_response": (
        "sample_response" in globals()
        and sample_response.get("status") == "success"
    ),

    "api_consistency": (
        "validation_checks" in globals()
        and all(validation_checks.values())
    ),

    "integration_validation": (
        "integration_validation" in globals()
        and all(integration_validation.values())
    )
}

print("RISKCHECK AI — END-TO-END VALIDATION")
print("-" * 60)

for component, passed in end_to_end_checks.items():

    print(
        f"{component:<30}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(
    "\nEnd-to-end workflow valid:",
    all(end_to_end_checks.values())
)

### Confirming Sample Scoring Output

In [0]:
print("FINAL RISKCHECK AI SAMPLE RESPONSE")
print("-" * 60)

for key, value in sample_response.items():
    print(f"{key:<25}: {value}")

## Summarizing Final Model & Operational Performance

The completed RiskCheck AI solution is evaluated using both machine learning performance metrics and operational fraud investigation outcomes.

The final Random Forest model was selected during Week 4 based on its strong validation performance and stable generalisation to the held-out test dataset.

Operational analysis performed during Weeks 5 and 6 demonstrated that the model-generated fraud scores can also support effective investigation prioritisation.

This section consolidates the key model and operational performance measures into a final project summary.

In [0]:
final_performance_summary = {
    "Test Accuracy": 0.9745,
    "Test Precision": 0.9703,
    "Test Recall": 0.9148,
    "Test F1 Score": 0.9417,
    "Test ROC-AUC": 0.9631,
    "Test Fraud Cases": 13510,
    "Fraud Correctly Detected": 12359,
    "False Positives": 378,
    "False Negatives": 1151,
    "Priority 1 + 2 Queue Share (%)": 20.65,
    "Priority 1 + 2 Fraud Capture (%)": 89.75
}

print("RISKCHECK AI — FINAL PERFORMANCE SUMMARY")
print("-" * 65)

for metric, value in final_performance_summary.items():
    print(f"{metric:<40}: {value}")

### Validate Summary Figures

In [0]:
performance_validation = {
    "accuracy": round(test_accuracy, 4) == 0.9745,
    "precision": round(test_precision, 4) == 0.9703,
    "recall": round(test_recall, 4) == 0.9148,
    "f1_score": round(test_f1, 4) == 0.9417,
    "roc_auc": round(test_roc_auc, 4) == 0.9631,
    "true_positives": test_TP == 12359,
    "false_positives": test_FP == 378,
    "false_negatives": test_FN == 1151
}

print("FINAL PERFORMANCE VALIDATION")
print("-" * 60)

for metric, passed in performance_validation.items():
    print(
        f"{metric:<25}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(
    "\nPerformance summary valid:",
    all(performance_validation.values())
)

## Documentation of RiskCheck AI System Architecture & Workflow

RiskCheck AI combines data preparation, machine learning, explainability, operational risk scoring, and application integration into a single fraud detection workflow.

The completed solution follows the architecture below:

### 1. Invoice Data Input

Invoice, supplier, behavioural, financial, OCR, and image-related information enters the RiskCheck AI pipeline.

### 2. Data Preparation and Feature Engineering

Raw data is cleaned, validated, encoded, and transformed into the modelling features required by the fraud detection model.

The final model uses 32 predictor variables covering areas such as:

- Invoice amount and historical amount behaviour
- Supplier transaction history
- Supplier risk characteristics
- Duplicate and split-invoice indicators
- Submission timing behaviour
- Department budget relationships
- OCR-based invoice comparisons
- Image-related indicators
- Payment terms
- Invoice type

### 3. Fraud Prediction Model

The engineered feature vector is submitted to the selected Random Forest classifier.

The model generates:

- A binary fraud prediction
- A fraud probability score

### 4. RiskCheck Score

The model fraud probability is converted into an operational RiskCheck Score ranging from 0 to 100.

Higher scores represent greater estimated fraud risk.

### 5. Risk Classification

The RiskCheck Score is translated into five operational risk categories:

- Low Risk
- Moderate Risk
- Elevated Risk
- High Risk
- Critical Risk

### 6. Investigation Prioritisation

Each risk category is mapped to an investigation priority ranging from Priority 1 — Immediate to Priority 5 — Routine.

This allows investigation resources to be directed toward invoices with the highest estimated fraud risk.

### 7. Investigation Reasons

Important fraud indicators are converted into investigator-oriented reasons, such as:

- Split-invoice behaviour
- Blacklisted supplier status
- Unusual supplier amount patterns
- Large deviations from recent supplier averages
- Elevated supplier risk
- Unusual invoice amount behaviour

These signals provide operational context for the model-generated risk assessment.

### 8. Recommended Action

RiskCheck AI assigns an operational recommendation based on the invoice risk level, ranging from routine processing and monitoring to immediate fraud investigation.

### 9. Application Integration

The production-facing scoring layer accepts the invoice identifier and required model features and returns a structured RiskCheck AI response containing:

- RiskCheck Score
- Risk category
- Investigation priority
- Recommended action
- Fraud prediction
- Investigation reasons

This architecture separates machine learning prediction from operational decision support while maintaining a consistent end-to-end fraud assessment workflow.

### Compact Architecture Summary

In [0]:
riskcheck_architecture = [
    ("1", "Invoice Input", "Invoice, supplier, financial, OCR and behavioural data"),
    ("2", "Data Preparation", "Cleaning, validation, encoding and feature engineering"),
    ("3", "Feature Vector", "32 final modelling predictors"),
    ("4", "Random Forest", "Fraud probability and binary prediction"),
    ("5", "RiskCheck Score", "Fraud probability converted to a 0–100 score"),
    ("6", "Risk Classification", "Low through Critical Risk"),
    ("7", "Investigation Priority", "Priority 1 through Priority 5"),
    ("8", "Decision Support", "Investigation reasons and recommended action"),
    ("9", "Application Output", "Structured production-facing scoring response")
]

print("RISKCHECK AI — SYSTEM ARCHITECTURE")
print("-" * 100)

for step, component, description in riskcheck_architecture:
    print(
        f"{step:<5}"
        f"{component:<27}"
        f"{description}"
    )

print(
    f"\nArchitecture components documented: "
    f"{len(riskcheck_architecture)}"
)

## Documenting Limitations, Assumptions, and Operational Considerations

Although RiskCheck AI demonstrates strong fraud detection and investigation prioritisation performance, several limitations and assumptions must be considered before real-world deployment.

### Dataset Limitations

The current RiskCheck AI solution was developed and evaluated using the project dataset rather than a live production accounts-payable environment.

Model performance therefore represents performance on the available project data and should not automatically be interpreted as expected performance on future real-world invoices.

### Fraud Pattern Dependence

The model learns relationships between the available predictor variables and known fraud labels.

Fraud behaviour can change over time, and new fraud patterns may differ from those represented in the training dataset. Model performance should therefore be monitored for data drift and changes in fraud behaviour.

### False Positives and False Negatives

RiskCheck AI does not provide perfect fraud detection.

The final test results include both false positives and false negatives.

False positives may create unnecessary investigation workload, while false negatives represent fraudulent invoices that were not identified by the model.

The RiskCheck Score should therefore support investigation and decision-making rather than automatically determine whether an invoice is fraudulent.

### Risk Thresholds

The five operational risk categories and associated investigation priorities are based on defined RiskCheck Score ranges.

These thresholds provide a practical demonstration of investigation prioritisation but should be reviewed against organisational risk appetite, investigation capacity, fraud costs, and operational requirements before production deployment.

### Investigation Reasons

The investigation reasons provide contextual risk indicators based on selected invoice and supplier characteristics.

They are designed to assist investigators in understanding relevant risk signals but should not be interpreted as a complete causal explanation of the Random Forest prediction.

### Feature Availability

Production scoring assumes that all 32 required model predictors are available and correctly calculated when an invoice is submitted.

Missing, delayed, incorrectly calculated, or materially changed input features could affect scoring reliability.

### Human Oversight

RiskCheck AI is designed as a decision-support system.

High Risk or Critical Risk classifications should trigger appropriate review processes rather than automatically establish that fraud has occurred.

Final investigation and payment decisions should remain subject to appropriate human and organisational controls.

### Model Monitoring

A deployed version of RiskCheck AI would require ongoing monitoring of:

- Input data quality
- Feature distributions
- Fraud prevalence
- Model performance
- False-positive and false-negative rates
- Risk category distributions
- Investigation workload
- Model and data drift

Periodic model validation and retraining may be required as new labelled invoice data becomes available.

### Security and Governance

A production implementation would also require appropriate controls for authentication, authorisation, data privacy, audit logging, model versioning, access management, and secure handling of invoice and supplier information.

## Defining Future Improvements & the Production Roadmap

RiskCheck AI currently demonstrates a complete fraud-risk workflow from prepared invoice data through machine learning prediction, operational risk scoring, investigation prioritisation, and application-facing output.

Moving from the current project implementation to a production fraud detection system would require additional engineering, monitoring, governance, and model-development activities.

### 1. Production Data Integration

The current workflow should be connected to live invoice, supplier, procurement, finance, OCR, and other relevant enterprise data sources.

Production feature calculations should use controlled and repeatable data pipelines so that the same feature definitions used during model development are applied during operational scoring.

### 2. Automated Scoring Service

The API-style scoring function developed during Week 7 could be converted into a formal scoring service.

A production service could accept invoice information, execute the required feature transformations, apply the trained model, and return the RiskCheck Score and investigation information to downstream applications.

### 3. Model Registry and Versioning

The selected model should be stored and managed through an appropriate model registry.

Production implementation should track:

- Model version
- Training dataset version
- Feature definitions
- Evaluation metrics
- Deployment date
- Approval status

This would support reproducibility, governance, rollback, and controlled model updates.

### 4. Monitoring and Drift Detection

Automated monitoring should be introduced to detect changes in:

- Input data distributions
- Feature distributions
- Fraud prevalence
- RiskCheck Score distributions
- Risk category volumes
- Model performance
- Investigation outcomes

Significant changes could indicate data quality problems, behavioural changes, or model drift.

### 5. Investigator Feedback Loop

Investigation outcomes could be captured and returned to the RiskCheck AI data pipeline.

Confirmed fraud, cleared invoices, and investigator feedback could provide new labelled data for future model evaluation and retraining.

This would allow the system to improve as additional operational evidence becomes available.

### 6. Threshold Optimisation

The current risk bands provide a practical operational framework.

Future threshold selection could incorporate:

- Investigation team capacity
- Cost of false positives
- Cost of missed fraud
- Invoice value
- Fraud severity
- Organisational risk appetite

This would allow investigation resources to be allocated according to business impact rather than model probability alone.

### 7. Explainability Enhancement

The current investigation reasons provide contextual risk indicators.

Future versions could introduce additional model explainability techniques to provide more detailed invoice-level explanations and support investigator understanding of individual predictions.

### 8. Security and Governance

A production deployment should introduce appropriate controls for:

- Authentication and authorisation
- Role-based access
- Data encryption
- Audit logging
- Model governance
- Data privacy
- Model approval and change management

### 9. User Interface and Investigation Dashboard

RiskCheck AI could be integrated with an operational investigation interface or dashboard.

Investigators could use the interface to:

- View invoices ranked by investigation priority
- Review RiskCheck Scores
- Examine investigation reasons
- Filter by risk category
- Record investigation outcomes
- Track investigation status
- Monitor fraud trends and operational workload

### Production Roadmap

A practical production roadmap would therefore progress through:

**Validated Prototype → Controlled Pilot → Production Integration → Monitoring and Feedback → Continuous Model Improvement**

The current RiskCheck AI project represents the validated prototype stage and establishes the analytical and application-facing foundation required for these future stages.

## Final RiskCheck AI Project Validation

The final project validation confirms that the major components developed throughout the RiskCheck AI project remain complete and consistent.

The validation brings together the final modelling, operational, and application-facing checks performed during the project.

A successful final validation confirms that:

- The selected Random Forest model remains available.
- All 32 modelling features are defined.
- The production-facing scoring dataset contains all 60,000 test invoices.
- Production invoice identifiers remain unique.
- The production response schema contains the required operational fields.
- The RiskCheck AI scoring function remains available.
- Request and response contracts remain defined.
- The end-to-end workflow validation has passed.
- Final model performance validation has passed.
- API scoring remains consistent with the validated production output.
- Integration and error-handling scenarios have passed.

Passing these checks confirms completion of the validated RiskCheck AI prototype.

In [0]:
final_project_checks = {
    "random_forest_model": (
        "rf_model" in globals()
    ),

    "model_features": (
        "model_feature_columns" in globals()
        and len(model_feature_columns) == 32
    ),

    "production_records": (
        riskcheck_production_df.count() == 60000
    ),

    "unique_invoices": (
        riskcheck_production_df
        .select("invoice_id")
        .distinct()
        .count() == 60000
    ),

    "production_schema": (
        len(riskcheck_production_df.columns) == 8
    ),

    "scoring_function": callable(
        globals().get("score_riskcheck_invoice")
    ),

    "request_contract": (
        "riskcheck_request_fields" in globals()
    ),

    "response_contract": (
        "riskcheck_response_fields" in globals()
    ),

    "end_to_end_validation": (
        "end_to_end_checks" in globals()
        and all(end_to_end_checks.values())
    ),

    "performance_validation": (
        "performance_validation" in globals()
        and all(performance_validation.values())
    ),

    "api_consistency": (
        "validation_checks" in globals()
        and all(validation_checks.values())
    ),

    "integration_validation": (
        "integration_validation" in globals()
        and all(integration_validation.values())
    )
}

print("RISKCHECK AI — FINAL PROJECT VALIDATION")
print("-" * 65)

for check_name, passed in final_project_checks.items():

    print(
        f"{check_name:<32}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

project_complete = all(
    final_project_checks.values()
)

print("-" * 65)
print(
    "RiskCheck AI project complete:",
    project_complete
)

## Week 8 Conclusion

Week 8 completed the final validation and documentation of the RiskCheck AI prototype.

The complete workflow was successfully validated from the trained Random Forest model through production-facing fraud-risk scoring and application integration.

Final validation confirmed that all 32 modelling features remain available, all 60,000 test invoices are preserved in the production-facing scoring dataset, invoice identifiers remain unique, and the production response schema is complete.

The final Random Forest model achieved:

- Accuracy: 97.45%
- Precision: 97.03%
- Recall: 91.48%
- F1-score: 94.17%
- ROC-AUC: 96.31%

The operational prioritisation workflow demonstrated that Priority 1 and Priority 2 together represent 20.65% of the investigation queue while capturing 89.75% of known fraud cases in the held-out test dataset.

The completed solution provides:

- Fraud probability prediction
- A 0–100 RiskCheck Score
- Five operational risk categories
- Investigation priority ranking
- Recommended investigation actions
- Investigator-oriented risk signals
- A production-facing scoring schema
- Defined request and response contracts
- An API-style scoring workflow
- Input validation and basic error handling

The project also documented important limitations, operational considerations, governance requirements, monitoring needs, and potential future improvements required before real-world production deployment.

RiskCheck AI should therefore be considered a **validated fraud-risk detection and investigation-prioritisation prototype**, rather than a fully deployed production fraud system.

The prototype demonstrates how machine learning predictions can be transformed into interpretable and operationally useful fraud-risk information while retaining human oversight in investigation and decision-making.

                DATABRICKS
                    │
          Your 300,000 invoices
                    │
                    ▼
            32 model features
                    │
                    ▼
          Random Forest Model
                    │
              Model Serving
                    │
                    ▼
┌───────────────────────────────────────┐
│          RISKCHECK AI APP             │
│                                       │
│ Client uploads CSV / Parquet          │
│              ↓                        │
│ Validate invoice data                 │
│              ↓                        │
│ Send features → Databricks            │
│              ↓                        │
│ Receive fraud probability             │
│              ↓                        │
│ RiskCheck Score 0–100                 │
│              ↓                        │
│ P1 / P2 / P3 / P4 / P5               │
│              ↓                        │
│ Dashboard + Investigation Queue       │
│              ↓                        │
│ Automated priority email              │
└───────────────────────────────────────┘
                    │
                    ▼
          Investigator decision
        Fraud / Not Fraud
                    │
                    ▼
           Verified training data
                    │
                    ▼
          Scheduled retraining